In [ ]:
"""
Analysis for "Waste Data: Operational Dairy Records for Pollution Accounting",
by I. Avery Bick, Daly Wettermark, Gina Hervey, Elena Eneva and Daniel E. Ho
(2026).

This notebook makes every figure and table in the paper and computes each number
quoted in its text. It builds results digures and tables from the reviewed
annual-report summary and transport manifests (outputs/review/), the routed
manure hauls from Routing.ipynb (outputs/results/routing/), and the Census,
boundary and EMFAC emission-rate files in data/. Nothing is re-extracted or
re-routed. Run the cells in order:

  0  setup: file paths, the routed manifests and their validation, the haul
     network, truck-miles, fuel and emissions (Figure S8, Table S7)
  1  routed against dropped manifests (Table S5)
  2  trucking exposure by block group, place and income (Tables S8-S11, and the
     inputs to Figure 4)
  3  nitrogen balance against the CARB inventory (Figure 5)
  4  how dropped manifests differ from routed ones (Figure S7)
  5  manifest destinations, nitrogen accounting and the trucking network
     (Figures 2, 3, 4, S1, S4, S5 and S9; Table S6)
  6  manifest-extraction accuracy and templates (Figure S10, Table S1)
  7  the manuscript's numbers as LaTeX macros, and the SI tables in LaTeX

Outputs:
  outputs/results/figures/   figures, each as a PNG and an EPS with the same name
  outputs/results/tables/    tables as CSV, and the SI tables in LaTeX
                             (SI_Tables.md, SI_Tables_S2_S4.tex)
  outputs/results/Manuscript_Parameters.tex
                             one LaTeX macro per number in the text, to include
                             in the manuscript preamble; Manuscript_Parameters.csv
                             lists each value and the cell it comes from
  outputs/results/routing/pair_cache/
                             road-buffer intersections, rebuilt when missing
"""
import os
import re
import calendar
import shutil
import subprocess
import tempfile
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
import matplotlib.cm as cm
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
import matplotlib.patheffects as path_effects
import seaborn as sns
from pyproj import Transformer
from scipy.stats import spearmanr, mannwhitneyu, binomtest

warnings.filterwarnings('ignore')

# Draw gridlines and ticks below bars and other patches.
plt.rcParams['axes.axisbelow'] = True

# =========================================================================
# 0) FILE PATHS
#   Paths are relative to the repository root, found by walking up from the
#   working directory (analysis/ when run in Jupyter).
# =========================================================================
def find_repo_root():
    """Nearest folder at or above the working directory holding data/ and
    analysis/Analysis.ipynb."""
    d = os.path.abspath(os.getcwd())
    while True:
        if (os.path.isdir(os.path.join(d, "data"))
                and os.path.isfile(os.path.join(d, "analysis", "Analysis.ipynb"))):
            return d + os.sep
        if os.path.dirname(d) == d:
            raise FileNotFoundError(
                f"no repository root above {os.getcwd()}; run this notebook "
                "from inside the manure-trucking folder")
        d = os.path.dirname(d)


ROOT_PATH = find_repo_root()
# Case-sensitive check: a folder named "Outputs" does not match, even on a
# case-insensitive disk.
if "outputs" not in os.listdir(ROOT_PATH):
    raise FileNotFoundError(f"no outputs/ folder in {ROOT_PATH}")
FIGURES_DIR = ROOT_PATH + "outputs/results/figures"
TABLES_DIR  = ROOT_PATH + "outputs/results/tables"
# Exposure pair-table caches (cell 2). The folder sorts after
# network_with_hauls_v13.gpkg, so a git checkout writes the caches after the
# network file and the cache age check in cell 2 accepts them.
CACHE_DIR   = ROOT_PATH + "outputs/results/routing/pair_cache"
for _d in (FIGURES_DIR, TABLES_DIR, CACHE_DIR):
    os.makedirs(_d, exist_ok=True)


def out_path(filename):
    """Figures go to outputs/results/figures, tables to outputs/results/tables."""
    if filename.startswith("Figure_"):
        return os.path.join(FIGURES_DIR, filename)
    if filename.endswith(".csv"):
        return os.path.join(TABLES_DIR, filename)
    raise ValueError(f"no output folder for {filename}")


# Every figure is also written as EPS: matplotlib writes a PDF and poppler's
# pdftocairo converts it. EPS has no transparency, so pdftocairo rasterizes the
# semi-transparent regions at EPS_RASTER_DPI and keeps everything else as vector
# graphics.
PDFTOCAIRO = shutil.which("pdftocairo")
if PDFTOCAIRO is None:
    raise FileNotFoundError("pdftocairo (poppler) not found; install the environment in environment/environment.yml")
EPS_RASTER_DPI = 300


def save_figure(fig, filename, rasterize=(), **kwargs):
    """Save a figure as PNG under outputs/results/figures, and as EPS beside it.

    The plotted layers (collections) of the axes in `rasterize` go into the EPS
    as images at the savefig dpi; the PNG is the same either way.
    """
    fig.savefig(out_path(filename), **kwargs)
    for ax in rasterize:
        for coll in ax.collections:
            coll.set_rasterized(True)
    eps_path = os.path.splitext(out_path(filename))[0] + ".eps"
    with tempfile.TemporaryDirectory() as tmp:
        pdf_path = os.path.join(tmp, "figure.pdf")
        fig.savefig(pdf_path, format="pdf", **kwargs)
        subprocess.run([PDFTOCAIRO, "-eps", "-r", str(EPS_RASTER_DPI), pdf_path, eps_path],
                       check=True)
    # Drop the conversion timestamp so reruns give identical files.
    with open(eps_path, "rb") as fh:
        eps = fh.read()
    with open(eps_path, "wb") as fh:
        fh.write(re.sub(rb"^%%CreationDate:[^\n]*\n", b"", eps, flags=re.M))


# --- Manure Files ---
# Written by analysis/Routing.ipynb: processed manure manifests with
# destination geometry, recalculated hauls, manual geocode verdicts, and the
# 'valid route' / 'validation reason' columns.
MANURE_VALIDATION_FILE = ROOT_PATH + "outputs/results/routing/processed_manure_manifests_analysis.geojson"
DAIRY_REPORT_FILE    = ROOT_PATH + "outputs/review/annual-reports/Dairy_Annual_Report_Summary_Table_Region_5_2024.csv"

# Operations that filed the dairy annual report form but whose report shows
# they are not dairy-cow herds. Excluded from every report-based total
# (manure, nitrogen, wastewater, herd, animal units).
# Keyed on Source PDF: these facilities cannot be identified from the herd
# table, and "Valley Cross Dairy" is also the name of a separate cow dairy.
# "15373 S Flanagan Rd Dos Palos 2024 Poultry AR.PDF" is not listed: its herd
# table reports dairy cattle.
NON_DAIRY_SOURCE_PDFS = {
    "9500 W. Linwood Ave, Turlock 2024 GoatDairy AR.pdf",   # Valley Cross Dairy, goats
    "8207 Crows Landing Road 2024 Bovine AR.PDF",           # S&D Feedlot, beef
}
# Optional report column; when present it overrides NON_DAIRY_SOURCE_PDFS.
# Truthy values mark a non-dairy row.
NON_DAIRY_FLAG_COL = "Non-Dairy Operation"


def dairy_cow_mask(frame):
    """True where a report row is a dairy-cow herd, for use in every total."""
    if NON_DAIRY_FLAG_COL in frame.columns:
        flag = (frame[NON_DAIRY_FLAG_COL].astype(str).str.strip().str.lower()
                .isin({"1", "true", "yes", "y", "t"}))
        return ~flag
    return ~frame["Source PDF"].isin(NON_DAIRY_SOURCE_PDFS)
NETWORK_SHAPEFILE    = ROOT_PATH + "outputs/results/routing/network_with_hauls_v13.gpkg"
OD_LINES_FILE        = ROOT_PATH + "outputs/results/routing/od_lines_with_status.geojson"
# Per-route, per-segment haul contributions written by analysis/Routing.ipynb.
# The network file's h10/h20 sum over every routed pair, including those
# rejected in review; re-aggregating from this table restricts the network
# maps, emissions and exposure analysis (which read h10/h20) to valid routes.
ROUTE_EDGES_FILE     = ROOT_PATH + "outputs/results/routing/route_edge_usage_v13.csv.gz"
MANIFEST_FORMATS_FILE = ROOT_PATH + "outputs/review/manifests/2024_manifest_template_counts.csv"
REPORT_FORMATS_FILE = (ROOT_PATH
                       + "outputs/review/annual-reports/Annual_Report_Formats.csv")
EMFAC_RATES_FILE      = ROOT_PATH + "data/tabular-data/EMFAC_Rates_2024.csv"
PLACE_SHP_FILE        = (ROOT_PATH + "data/geospatial-data/CA_Census_Place/"
                         "cb_2024_06_place_500k.shp")
PLACE_POP_FILE        = (ROOT_PATH + "data/geospatial-data/CA_Census_Place/"
                         "place acs pop data 2020-2024/nhgis0018_ds272_20245_place.csv")
# --- Wastewater Files ---
# When False, the wastewater inputs, figures and summary lines are skipped and
# WW_CSV_FILE is not read.
INCLUDE_WASTEWATER = True
WW_CSV_FILE       = ROOT_PATH + "outputs/review/manifests/processed_wastewater_manifests.csv"
# --- Geospatial / Census Files ---
BOUNDARY_FILE = ROOT_PATH + "data/geospatial-data/CA_Water_Board_Boundary/CentralValleyRegionalWaterBoard_EPSG4326.geojson"
CA_STATE_FILE = ROOT_PATH + "data/geospatial-data/CA_State/CA_State.geojson"
# California county FIPS codes are the odd numbers 001 to 115 in alphabetical
# order of county name. Names the counties dissolved from the block-group
# shapefile for the Figure 4a overlay.
CA_COUNTY_NAMES = [
    "Alameda", "Alpine", "Amador", "Butte", "Calaveras", "Colusa",
    "Contra Costa", "Del Norte", "El Dorado", "Fresno", "Glenn", "Humboldt",
    "Imperial", "Inyo", "Kern", "Kings", "Lake", "Lassen", "Los Angeles",
    "Madera", "Marin", "Mariposa", "Mendocino", "Merced", "Modoc", "Mono",
    "Monterey", "Napa", "Nevada", "Orange", "Placer", "Plumas", "Riverside",
    "Sacramento", "San Benito", "San Bernardino", "San Diego",
    "San Francisco", "San Joaquin", "San Luis Obispo", "San Mateo",
    "Santa Barbara", "Santa Clara", "Santa Cruz", "Shasta", "Sierra",
    "Siskiyou", "Solano", "Sonoma", "Stanislaus", "Sutter", "Tehama",
    "Trinity", "Tulare", "Tuolumne", "Ventura", "Yolo", "Yuba",
]
COUNTY_FIPS = {f"{2 * i + 1:03d}": n for i, n in enumerate(CA_COUNTY_NAMES)}
# Dairy geocodes cached by review/annual-reports/geocode_dairy_addresses.py, so
# the figures need no online geocoding.
DAIRY_GEOCODE_FILE = ROOT_PATH + "outputs/review/annual-reports/Dairy_Report_Geocodes.csv"
COUNTIES_FILE = ROOT_PATH + "data/geospatial-data/CA_Counties/CA_Counties.shp"
BG_SHP_FILE   = ROOT_PATH + "data/geospatial-data/CA_BGs/nhgis0015_shape-nhgis0015_shapefile_tl2024_060_blck_grp_2024/CA_blck_grp_2024.shp"
BG_CSV_FILE   = ROOT_PATH + "data/geospatial-data/CA_BGs/nhgis0017_ds272_20245_blck_grp.csv"
BG_INCOME_COL     = "AURUE001"   # B19013 median household income, 2020-24 ACS
BG_INCOME_MOE_COL = "AURUM001"   # its margin of error (optional)
# B19301 per capita income, from the same ds272 extract; reported alongside
# median household income. Set to None to skip it.
BG_INCOME_PC_COL     = "AUSYE001"
BG_INCOME_PC_MOE_COL = "AUSYM001"

# --- Buffer-exposure analysis (Figure 4) ---
# Block-group population, for the per-capita normalization.
#   Download from IPUMS NHGIS (https://data2.nhgis.org):
#     Geographic levels : Block Group
#     Years             : 2020-2024
#     Dataset           : 2020_2024_ACS5a  (ds272, as the income extract)
#     Source table      : B01003  (Total Population)
#     Geographic extent : California
#   Joins to the block-group shapefile on GISJOIN. The estimate column is
#   <CODE>E001 for the NHGIS table code of B01003 (as AURUE001 is for B19013).
#   BG_POP_COL = None auto-detects it and the loader prints the column used.
BG_POP_CSV_FILE = ROOT_PATH + "data/geospatial-data/CA_BGs/nhgis0016_ds272_20245_blck_grp.csv"
BG_POP_COL      = None

# Fail early if any input is missing: several readers below skip a panel or
# table when their file is absent.
_REQUIRED_INPUTS = [
    MANURE_VALIDATION_FILE, DAIRY_REPORT_FILE, NETWORK_SHAPEFILE, OD_LINES_FILE,
    ROUTE_EDGES_FILE, MANIFEST_FORMATS_FILE, REPORT_FORMATS_FILE, EMFAC_RATES_FILE,
    PLACE_SHP_FILE, PLACE_POP_FILE, WW_CSV_FILE, BOUNDARY_FILE, CA_STATE_FILE,
    DAIRY_GEOCODE_FILE, COUNTIES_FILE, BG_SHP_FILE, BG_CSV_FILE, BG_POP_CSV_FILE,
]
_missing_inputs = [p for p in _REQUIRED_INPUTS if not os.path.exists(p)]
if _missing_inputs:
    raise FileNotFoundError("missing inputs:\n  " + "\n  ".join(_missing_inputs))

# Flat near-road buffer distance (m) for the exposure analysis.
BUFFER_DISTANCE_M = 500
# Bootstrap draws for the population-weighted correlation CIs.
CORR_BOOTSTRAP_DRAWS = 2000
# CA Albers (metric, equal-area) for buffer distances and polygon areas. The
# study area extends into UTM zone 11, so EPSG:32610 is not used for areas.
ANALYSIS_CRS = "EPSG:3310"

DAIRY_THRESHOLD_SIZE = 1500

# Distances are reported in miles and areas in square miles. Lengths and areas
# are converted where measured, and stored kilometre columns where read.
# DIST_UNIT and AREA_UNIT are used in labels.
KM_PER_MILE = 1.609344
M_PER_MILE = 1000.0 * KM_PER_MILE
M2_PER_MI2 = M_PER_MILE ** 2
MI_PER_KM = 1.0 / KM_PER_MILE
DIST_UNIT = "mi"
AREA_UNIT = "mi$^2$"

# Trucks return empty along the outbound path, so each haul passes every
# segment twice. Vehicle-miles, emissions and fuel scale with this factor
# (EMFAC rates are per vehicle-mile); manure tonnage does not.
ROUND_TRIP_FACTOR = 2.0

# =========================================================================
# CONSISTENT COLOR SCHEMES
# =========================================================================
DEST_COLOR_MAP = {
    'Farmer':              '#F5DEB3',  # wheat
    'Blank':               '#A9A9A9',  # grey
    'Broker':              '#85BB65',  # cash green
    'Composting Facility': '#8B4513',  # dirt / saddle brown
    'Other':               '#C8A2C8',  # light purple (lilac)
}

# Grey for "no information" bars in Figures 2 and 3: blank destination type,
# unrecorded transport mode, unaccounted-for nitrogen.
FIG1_GREY = '0.45'

TRANSPORT_COLOR_MAP = {
    'Pipeline': '#0D98BA',  # blue-green
    'Trucked':  '#FFA500',  # yellow-orange
    'Unsure':   FIG1_GREY,
}

# =========================================================================
# HELPER FUNCTIONS
# =========================================================================
def make_autopct_count(values):
    """Pie label: pct% (raw count with thousands separator)"""
    def my_autopct(pct):
        total = sum(values)
        val = int(round(pct * total / 100.0))
        return f'{pct:.1f}%\n({val:,})'
    return my_autopct

def make_autopct_short(values):
    """Pie label: pct% with abbreviated units (K, M)"""
    def my_autopct(pct):
        total = sum(values)
        val = int(round(pct * total / 100.0))
        if val >= 1_000_000:
            val_str = f'{val/1_000_000:.1f}M'
        elif val >= 1_000:
            val_str = f'{val/1_000:.1f}K'
        else:
            val_str = f'{val:,}'
        return f'{pct:.1f}%\n({val_str})'
    return my_autopct

def get_dest_colors(categories):
    return [DEST_COLOR_MAP.get(c, '#CCCCCC') for c in categories]

# Pie-slice display labels; "Composting Facility" is wrapped so it does not
# overlap neighboring panels.
DEST_DISPLAY_LABELS = {
    'Composting Facility': 'Composting\nFacility',
}

def get_display_labels(categories):
    return [DEST_DISPLAY_LABELS.get(c, c) for c in categories]

def add_panel_label(ax, label, fontsize=24, x=-0.02, y=1.0):
    """Place a bold panel label (e.g. 'a)') left of the top-left corner of
    `ax`, in axes coordinates."""
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=fontsize, fontweight='bold',
            ha='right', va='top')

# Semi-transparent white halo that keeps city labels legible over map features.
CITY_LABEL_HALO = [
    path_effects.Stroke(linewidth=3, foreground='white', alpha=0.5),
    path_effects.Normal(),
]

def add_latlon_ticks(ax, map_crs, fontsize=16, fmt='{:.1f}',
                     sides=('top', 'bottom', 'left', 'right'),
                     show_labels=True, color='black',
                     lon_step=None, lat_step=None):
    """Label a map axis drawn in the projected `map_crs` with longitude /
    latitude ticks.

    sides: which of 'top', 'bottom', 'left', 'right' get ticks.
    show_labels: if False, draw tick marks only.
    color: color of ticks, tick labels and spines.
    lon_step / lat_step: if given (degrees), place ticks at multiples of the
        step; otherwise matplotlib's default locator is used."""
    transformer = Transformer.from_crs(map_crs, "EPSG:4326", always_xy=True)

    def fmt_lon(x, _pos):
        ymid = sum(ax.get_ylim()) / 2.0
        lon, _ = transformer.transform(x, ymid)
        return fmt.format(lon)

    def fmt_lat(y, _pos):
        xmid = sum(ax.get_xlim()) / 2.0
        _, lat = transformer.transform(xmid, y)
        return fmt.format(lat)

    ax.xaxis.set_major_formatter(ticker.FuncFormatter(fmt_lon))
    ax.yaxis.set_major_formatter(ticker.FuncFormatter(fmt_lat))

    # Ticks at round degree values, projected into the map CRS.
    if lon_step or lat_step:
        fwd_t = Transformer.from_crs("EPSG:4326", map_crs, always_xy=True)
        xmin, xmax = ax.get_xlim()
        ymin, ymax = ax.get_ylim()
        lon_min, _ = transformer.transform(xmin, (ymin + ymax) / 2)
        lon_max, _ = transformer.transform(xmax, (ymin + ymax) / 2)
        _, lat_min = transformer.transform((xmin + xmax) / 2, ymin)
        _, lat_max = transformer.transform((xmin + xmax) / 2, ymax)
        mid_lat = (lat_min + lat_max) / 2.0
        mid_lon = (lon_min + lon_max) / 2.0
        if lon_step:
            first = np.ceil(lon_min / lon_step) * lon_step
            ticks_deg = np.arange(first, lon_max + 1e-9, lon_step)
            ax.set_xticks([fwd_t.transform(lon, mid_lat)[0] for lon in ticks_deg])
        if lat_step:
            first = np.ceil(lat_min / lat_step) * lat_step
            ticks_deg = np.arange(first, lat_max + 1e-9, lat_step)
            ax.set_yticks([fwd_t.transform(mid_lon, lat)[1] for lat in ticks_deg])

    sides_set = set(sides)
    ax.tick_params(
        top=    'top'    in sides_set,
        bottom= 'bottom' in sides_set,
        left=   'left'   in sides_set,
        right=  'right'  in sides_set,
        labeltop=    show_labels and 'top'    in sides_set,
        labelbottom= show_labels and 'bottom' in sides_set,
        labelleft=   show_labels and 'left'   in sides_set,
        labelright=  show_labels and 'right'  in sides_set,
        labelsize=fontsize,
        colors=color,
    )
    for spine in ax.spines.values():
        spine.set_color(color)

def standardize_dest_type(series):
    valid_dest = ["Blank", "Broker", "Farmer", "Composting Facility"]
    s = series.fillna('Blank')
    return s.apply(lambda x: x if x in valid_dest else 'Other')

def pick_col(df, base):
    """Return `base`, or the '<base>_manifest' / '<base>_report' column that
    the manifest-report merge creates, preferring the manifest side."""
    for cand in (base, f"{base}_manifest", f"{base}_report"):
        if cand in df.columns:
            return cand
    raise KeyError(f"none of {base!r}/_manifest/_report present")

def normalize_source_pdf(s):
    """Canonical Source PDF key for joining: '.pdf' extension removed in any
    case, whitespace collapsed, lowercased. The manifests store the name
    without the extension; the report keeps it."""
    if pd.isna(s):
        return None
    t = re.sub(r"\.pdf\s*$", "", str(s).strip(), flags=re.I)
    return re.sub(r"\s+", " ", t).strip().lower() or None


def canon_dairy_name(s):
    """Dairy name reduced to its distinguishing words."""
    if pd.isna(s):
        return None
    t = str(s).lower()
    t = re.sub(r"\b(dairy|dairies|farms?|llc|inc|ranch(es)?|and|the)\b", " ", t)
    t = " ".join(re.sub(r"[^a-z0-9]+", " ", t).split())
    return t or None


# Street words as spelled after street_key's normalisation (avenue -> av,
# road -> rd, street -> st). street_key(anywhere=True) does not take a number
# that follows one of these as a house number.
_STREET_WORD = re.compile(
    r"^(?:av|rd|st|hwy|highway|blvd|boulevard|ln|lane|dr|drive|way|ct|court"
    r"|pl|place|cir|circle|trl|trail|pkwy|parkway|y|and|of)$")


def street_key(s, anywhere=False):
    """House number plus the next two words of a street address.

    With anywhere=True, an address not starting with a house number is scanned
    for the first one, so a facility name followed by an address still gets a
    key. Skipped as not house numbers: five-digit ZIPs (90000-96199, or any
    five-digit number after CA), a number after a street word (numbered roads
    such as "Avenue 120"), and a number after another number (fractions such
    as "Avenue 24 1/2").
    """
    if pd.isna(s):
        return None
    t = str(s).lower()
    t = re.sub(r"\b(ave|avenue)\b", "av", t)
    t = re.sub(r"\b(rd|road)\b", "rd", t)
    t = re.sub(r"\b(st|street)\b", "st", t)
    t = " ".join(re.sub(r"[^a-z0-9]+", " ", t).split())
    pattern = r"^(\d+)\s+(\S+)(?:\s+(\S+))?"
    m = re.match(pattern, t)
    if m is None and anywhere:
        tokens = t.split()
        for j in range(1, len(tokens) - 1):
            tok = tokens[j]
            if not tok.isdigit() or (len(tok) == 5 and (90000 <= int(tok) <= 96199
                                                        or tokens[j - 1] in ("ca", "california"))):
                continue
            if not tokens[j + 1][:1].isalpha():
                continue
            prev = tokens[j - 1]
            if prev.isdigit() or _STREET_WORD.match(prev):
                continue
            m = re.match(pattern, " ".join(tokens[j:]))
            break
    return " ".join(x for x in m.groups() if x) if m else None


def unique_lookup(df, keycol):
    """Map key -> row label, keeping only keys that identify exactly one row."""
    counts = df[keycol].value_counts()
    ok = set(counts[counts == 1].index)
    sub = df[df[keycol].isin(ok)]
    return dict(zip(sub[keycol], sub.index))


def resolve_report_rows(man, rep):
    """Match each manifest to at most one report row.

    Keys are tried in order: Source PDF, then dairy name, then street address
    (some manifests carry '2024AR_<name>' where the report filename is
    '2024AR_<name>_<address>_<county>'). A key shared by several report rows
    is not used. Returns the matched row labels and the key that matched.
    """
    by_pdf = unique_lookup(rep, "Source PDF_Cleaned")
    by_name = unique_lookup(rep.assign(_n=rep["Dairy Name"].map(canon_dairy_name))
                            .dropna(subset=["_n"]), "_n")
    by_street = unique_lookup(rep.assign(_s=rep["Dairy Address"].map(street_key))
                              .dropna(subset=["_s"]), "_s")
    idx, how = [], []
    for k, nm, ad in zip(man["Source PDF_Cleaned"],
                         man["Origin Dairy Name"].map(canon_dairy_name),
                         man["Origin Dairy Address"].map(street_key)):
        if k in by_pdf:
            idx.append(by_pdf[k]); how.append("Source PDF")
        elif nm in by_name:
            idx.append(by_name[nm]); how.append("dairy name")
        elif ad in by_street:
            idx.append(by_street[ad]); how.append("street address")
        else:
            idx.append(pd.NA); how.append("unmatched")
    return pd.Series(idx, index=man.index, dtype="Int64"), pd.Series(how, index=man.index)


def dedupe_dairy_report(df_report, key='Source PDF_Cleaned', verbose=True):
    """Collapse the dairy report to at most one row per Source PDF key, so the
    manifest-report join is one-to-one.

      1. Exact duplicate rows are dropped.
      2. Keys shared by different dairies (e.g. 'Dairy Name' parsed as
         "CENTRAL VALLEY REGION", differing herd sizes) are dropped, leaving
         their manifests with missing report fields.
      3. Rows with no key are dropped.
    """
    n0 = len(df_report)
    df_report = df_report.drop_duplicates()
    n_exact = n0 - len(df_report)
    counts = df_report[key].value_counts()
    ambiguous = counts[counts > 1].index.tolist()
    if verbose:
        if n_exact:
            print(f"  dairy report: dropped {n_exact} exact duplicate row(s)")
        if ambiguous:
            print(f"  dairy report: {len(ambiguous)} Source PDF key(s) map to "
                  "multiple different dairies; treating as unmatched "
                  "(report fields left missing):")
            for k in ambiguous[:5]:
                print(f"      {k!r} ({int(counts[k])} rows)")
    df_report = df_report[~df_report[key].isin(ambiguous)].copy()
    n_nullkey = int(df_report[key].isna().sum())
    if n_nullkey and verbose:
        print(f"  dairy report: dropped {n_nullkey} row(s) with no Source PDF key")
    df_report = df_report[df_report[key].notna()].copy()
    assert not df_report[key].duplicated().any(), "report key still not unique"
    return df_report

def get_unique_dairy_count(df):
    for col in ['Origin Dairy Name', 'Origin Dairy Name_manifest', 'Origin Dairy Name_report']:
        if col in df.columns:
            return df[col].nunique()
    return len(df)

def distribute_monthly(df, amount_col, d1_col='First Haul Date', d2_col='Last Haul Date'):
    """Spread each row's amount evenly over the days from d1 to d2 and sum by
    month; a row with only one date goes to that month."""
    monthly = {i: 0.0 for i in range(1, 13)}
    for _, row in df.iterrows():
        amt = row[amount_col]
        if amt <= 0:
            continue
        d1 = row[d1_col]
        d2 = row[d2_col]
        if pd.isna(d1) and pd.notna(d2):
            monthly[d2.month] += amt
        elif pd.notna(d1) and pd.notna(d2):
            if d1 > d2:
                d1, d2 = d2, d1
            days = pd.date_range(start=d1, end=d2, freq='D')
            if len(days) == 0:
                days = [d1]
            daily_amt = amt / len(days)
            for d in days:
                monthly[d.month] += daily_amt
        elif pd.notna(d1) and pd.isna(d2):
            monthly[d1.month] += amt
    return monthly

# =========================================================================
# 1) LOAD AND PREP MANURE DATA
# =========================================================================
print("Loading Manure Data (validated)...")
if not os.path.exists(MANURE_VALIDATION_FILE):
    raise FileNotFoundError(
        f"{MANURE_VALIDATION_FILE} not found.\n"
        "  Run analysis/Routing.ipynb first to generate it."
    )
manure_df = pd.DataFrame(gpd.read_file(MANURE_VALIDATION_FILE))
manure_df['Total Manure Amount (tons)'] = pd.to_numeric(
    manure_df['Total Manure Amount (tons)'], errors='coerce').fillna(0)

if "RouteID" not in manure_df.columns:
    manure_df["RouteID"] = manure_df.index + 1

print("Joining Dairy Report Data...")
if os.path.exists(DAIRY_REPORT_FILE):
    df_report = pd.read_csv(DAIRY_REPORT_FILE)
    df_report['Source PDF_Cleaned'] = df_report['Source PDF'].map(normalize_source_pdf)
    df_report['_is_dairy_cow'] = dairy_cow_mask(df_report)
    _nd = df_report.loc[~df_report['_is_dairy_cow'], 'Dairy Name']
    print(f"  non-dairy operations excluded from report totals: {len(_nd)}"
          + (f"  ({', '.join(map(str, _nd))})" if len(_nd) else ""))
    # Enforce a one-to-one join.
    df_report = dedupe_dairy_report(df_report)
    manure_df['Source PDF_Cleaned'] = manure_df['Source PDF'].map(normalize_source_pdf)
    _rep_idx, _rep_how = resolve_report_rows(manure_df, df_report)
    manure_df['_rep_idx'] = _rep_idx
    manure_df['Report Match Method'] = _rep_how
    print("  manifest-to-report match: " + ", ".join(
        f"{k} {v}" for k, v in _rep_how.value_counts().items()))
    _n_manifests_before_join = len(manure_df)
    manure_df = pd.merge(
        manure_df, df_report.reset_index(names='_rep_idx'),
        on='_rep_idx', how='left', suffixes=('_manifest', '_report')
    )
    assert len(manure_df) == _n_manifests_before_join, (
        f"dairy-report join changed the manifest count: "
        f"{_n_manifests_before_join} -> {len(manure_df)}")
    if 'Total Herd Size' in manure_df.columns:
        manure_df['Total Herd Size'] = pd.to_numeric(manure_df['Total Herd Size'], errors='coerce')

    # Animal-unit factors (animal weight / 1,000 lb) from Table 1 of Central
    # Valley Regional Water Board Order No. R5-2008-0066. The Order's five
    # classes map to the report form's six: bred heifers -> large heifer,
    # heifers -> small heifer, both calf classes -> calves.
    AU_FACTORS = {
        'Milk Cows':        1.40,   # Milk Cow
        'Dry Cows':         1.12,   # Dry Cow
        'Bred Heifers':     1.02,   # Large Heifer
        'Heifers':          0.49,   # Small Heifer
        'Calves (4-6 mo.)': 0.29,   # Calves
        'Calves (0-3 mo.)': 0.29,   # Calves
    }
    # Check: the Order's Table 1 example herd (15,407 head) totals 17,756 AU.
    assert abs(9400 * 1.40 + 1057 * 1.12 + 2125 * 1.02 + 2125 * 0.49
               + 700 * 0.29 - 17756) < 1.0, \
        'AU factors do not reproduce Table 1 of R5-2008-0066'
    if all(c in manure_df.columns for c in AU_FACTORS):
        _h = manure_df[list(AU_FACTORS)].apply(pd.to_numeric, errors='coerce')
        # Only manifests reporting all six classes get animal units; a blank
        # class would understate the weighted sum.
        manure_df['Animal Units'] = ((_h * pd.Series(AU_FACTORS)).sum(axis=1)
                                     .where(_h.notna().all(axis=1)))
        print(f"  animal units computed for "
              f"{int(manure_df['Animal Units'].notna().sum()):,} of "
              f"{len(manure_df):,} manifests "
              f"({int(_h.notna().any(axis=1).sum()) - int(_h.notna().all(axis=1).sum()):,}"
              f" dropped for an incomplete herd table)")
else:
    print(f"  Warning: {DAIRY_REPORT_FILE} not found.")

manure_df['Dest_Type'] = standardize_dest_type(manure_df['Destination Type (Standardized)'])
manure_df['First Haul Date'] = pd.to_datetime(manure_df['First Haul Date'], errors='coerce')
manure_df['Last Haul Date']  = pd.to_datetime(manure_df['Last Haul Date'], errors='coerce')

# =========================================================================
# 2) LOAD AND PREP WASTEWATER DATA
# =========================================================================
if INCLUDE_WASTEWATER:
    print("Loading Wastewater Data...")
    WW_VOL_COL = "Total Process Wastewater Exports (Gallons)"
    ww_df = pd.read_csv(WW_CSV_FILE)
    if "RouteID" not in ww_df.columns:
        ww_df["RouteID"] = ww_df.index + 1
    ww_df[WW_VOL_COL] = pd.to_numeric(ww_df[WW_VOL_COL], errors='coerce').fillna(0)
    ww_df['Dest_Type'] = standardize_dest_type(ww_df['Destination Type (Standardized)'])
    ww_df['First Haul Date'] = pd.to_datetime(ww_df['First Haul Date'], errors='coerce')
    ww_df['Last Haul Date']  = pd.to_datetime(ww_df['Last Haul Date'], errors='coerce')

    def determine_transport_mode(row):
        t = str(row.get('Is Trucked', '')).strip().upper()
        p = str(row.get('Is Pipeline', '')).strip().upper()
        if t == 'TRUE':
            return 'Trucked'
        elif p == 'TRUE':
            return 'Pipeline'
        else:
            return 'Unsure'

    ww_df['Transport_Mode'] = ww_df.apply(determine_transport_mode, axis=1)
else:
    print("Skipping wastewater data (INCLUDE_WASTEWATER = False)")

# =========================================================================
# 3) MANURE OD VALIDATION RESULTS
#    Reads the validation columns written by analysis/Routing.ipynb, prints
#    the drop counts, and keeps the valid routes as successfully_routed_df.
# =========================================================================
print("\nReading Manure OD Validation Results...")

# Water Board boundary, for the study-area checks and map overlays.
water_board = gpd.read_file(BOUNDARY_FILE)
if water_board.crs != "EPSG:4326":
    water_board = water_board.to_crs("EPSG:4326")

VALID_COL  = "valid route"
REASON_COL = "validation reason"

missing_val_cols = [c for c in (VALID_COL, REASON_COL) if c not in manure_df.columns]
if missing_val_cols:
    raise KeyError(
        f"{MANURE_VALIDATION_FILE} is missing {missing_val_cols}. "
        "Re-run analysis/Routing.ipynb."
    )

def as_bool_series(s):
    """'valid route' reads back as bool, numeric, or string depending on driver."""
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype(bool)
    if pd.api.types.is_numeric_dtype(s):
        return s.fillna(0).astype(float) != 0
    return s.astype(str).str.strip().str.lower().isin(['true', 't', 'yes', '1', '1.0'])

m_valid_mask = as_bool_series(manure_df[VALID_COL])
m_reason = manure_df[REASON_COL].fillna('').astype(str).str.strip()

# Drop reasons in the order Routing.ipynb applies its checks. Each manifest
# records only its first failing check, so reasons are mutually exclusive and
# the count after each stage follows by subtraction.
M_DROP_REASONS = [
    "No destination address",
    "Outside water board",
    "Failed geocode",
    "Cross-county APN",
    "Routing Failed",
    "Bad Origin Geocode",
    "Bad Destination Geocode",
    "Routing Incorrect",
]
m_reason_counts = m_reason[~m_valid_mask].value_counts()
m_drops = {r: int(m_reason_counts.get(r, 0)) for r in M_DROP_REASONS}

unexpected_reasons = [r for r in m_reason_counts.index if r and r not in M_DROP_REASONS]
if unexpected_reasons:
    print(f"  Warning: validation reason(s) not in M_DROP_REASONS: {unexpected_reasons}")

m_drop_no_dest         = m_drops["No destination address"]
m_drop_outside_wb      = m_drops["Outside water board"]
m_drop_failed_geo      = m_drops["Failed geocode"]
m_drop_diff_county_apn = m_drops["Cross-county APN"]
m_drop_routing_failed  = m_drops["Routing Failed"]
m_drop_bad_origin      = m_drops["Bad Origin Geocode"]
m_drop_bad_dest        = m_drops["Bad Destination Geocode"]
m_drop_routing_wrong   = m_drops["Routing Incorrect"]

m_total_pairs = len(manure_df)
m_val_after_1 = m_total_pairs - m_drop_no_dest
m_val_after_2 = m_val_after_1 - m_drop_outside_wb
m_val_after_3 = m_val_after_2 - m_drop_failed_geo
m_final_valid = m_val_after_3 - m_drop_diff_county_apn
m_routed        = m_final_valid - m_drop_routing_failed
m_orig_verified = m_routed - m_drop_bad_origin
m_dest_verified = m_orig_verified - m_drop_bad_dest
m_final_successfully_routed = int(m_valid_mask.sum())

successfully_routed_df = manure_df[m_valid_mask].drop(columns=['geometry'], errors='ignore').copy()

m_implied_valid = m_dest_verified - m_drop_routing_wrong
if m_implied_valid != m_final_successfully_routed:
    print(f"  Warning: drop counts imply {m_implied_valid:,} valid routes but "
          f"'{VALID_COL}' marks {m_final_successfully_routed:,}; the stage counts "
          "will not balance.")

# Each drop reason with the manifests remaining, then the count at each stage.
print(f"  manure OD pairs:                {m_total_pairs:>8,}")
_left = m_total_pairs
for r in M_DROP_REASONS:
    _left -= m_drops[r]
    print(f"  dropped: {r:<24}{m_drops[r]:>8,}   remaining {_left:>6,}")
print(f"  with a destination:             {m_val_after_1:>8,}")
print(f"  in the study area, geocoded:    {m_final_valid:>8,}")
print(f"  routed:                         {m_routed:>8,}")
print(f"  origin geocode verified:        {m_orig_verified:>8,}")
print(f"  destination geocode verified:   {m_dest_verified:>8,}")
print(f"  Manure pipeline complete. Successfully routed: {m_final_successfully_routed}")

# =========================================================================
# 4) WASTEWATER OD VALIDATION PIPELINE
# =========================================================================
if INCLUDE_WASTEWATER:
    print("\nValidating Wastewater OD Pairs...")
    ww_total_manifests = len(ww_df)

    # STEP 1: Missing coordinates
    ww_has_coords = (ww_df["Origin Dairy Longitude (Geocoded)"].notna() &
                     ww_df["Destination Longitude (Geocoded)"].notna() &
                     ww_df["Origin Dairy Latitude (Geocoded)"].notna() &
                     ww_df["Destination Latitude (Geocoded)"].notna())
    ww_step1 = ww_df[ww_has_coords].copy()
    ww_val_after_1 = len(ww_step1)
    ww_drop_missing_coords = ww_total_manifests - ww_val_after_1

    # STEP 2: Outside water board
    orig_gdf_ww = gpd.GeoDataFrame(
        ww_step1,
        geometry=gpd.points_from_xy(ww_step1["Origin Dairy Longitude (Geocoded)"],
                                    ww_step1["Origin Dairy Latitude (Geocoded)"]),
        crs="EPSG:4326")
    dest_gdf_ww = gpd.GeoDataFrame(
        ww_step1,
        geometry=gpd.points_from_xy(ww_step1["Destination Longitude (Geocoded)"],
                                    ww_step1["Destination Latitude (Geocoded)"]),
        crs="EPSG:4326")
    orig_in_wb_ww = gpd.sjoin(orig_gdf_ww, water_board, how="inner", predicate="intersects").index
    dest_in_wb_ww = gpd.sjoin(dest_gdf_ww, water_board, how="inner", predicate="intersects").index
    both_in_wb_ww = orig_in_wb_ww.intersection(dest_in_wb_ww)
    ww_step2 = ww_step1.loc[both_in_wb_ww].copy()
    ww_val_after_2 = len(ww_step2)
    ww_drop_outside_wb = ww_val_after_1 - ww_val_after_2

    # STEPS 3-4: wastewater has no geocode verification or routing step, so
    # every record inside the Water Board is kept.
    ww_step3 = ww_step2.copy()
    ww_val_after_3 = len(ww_step3)
    ww_drop_incorrect_geo = ww_val_after_2 - ww_val_after_3
    ww_successfully_routed_df = ww_step3.copy()

    ww_drop_routing_failed = ww_val_after_3 - len(ww_successfully_routed_df)

    def determine_final_mode(row):
        t = str(row.get('Is Trucked', '')).strip().upper() == 'TRUE'
        p = str(row.get('Is Pipeline', '')).strip().upper() == 'TRUE'
        if t: return 'Trucked'
        if p: return 'Pipeline'
        return 'Unsure'

    ww_successfully_routed_df['Final_Transport_Mode'] = ww_successfully_routed_df.apply(determine_final_mode, axis=1)
    ww_trucked_count  = int((ww_successfully_routed_df['Final_Transport_Mode'] == 'Trucked').sum())
    ww_pipeline_count = int((ww_successfully_routed_df['Final_Transport_Mode'] == 'Pipeline').sum())
    ww_unsure_count   = int((ww_successfully_routed_df['Final_Transport_Mode'] == 'Unsure').sum())
    print(f"  WW pipeline complete. Trucked={ww_trucked_count}, Pipeline={ww_pipeline_count}, Unsure={ww_unsure_count}")
else:
    print("Skipping wastewater OD validation")

# =========================================================================
# 5) NETWORK SPATIAL DATA + EMISSIONS
# =========================================================================
print("\nLoading Spatial Network Data...")
network = gpd.read_file(NETWORK_SHAPEFILE)
if network.crs is None:
    network = network.set_crs("EPSG:4326")

network['length_mi'] = network.geometry.length / M_PER_MILE

def network_hauls_from_routes(net, route_edges_file, od_lines_file,
                              valid_only=True, verbose=True):
    """Recompute the network's haul columns (h10, h20, km10, km20, kmtot) from
    the per-route table, keeping routes whose 'valid route' flag in the OD
    table is True (all routes if valid_only=False).

    Asserts that summing over all routes reproduces the stored totals, so a
    stale or truncated table fails here. Returns `net` unchanged if the table
    or the u/v/key edge columns are missing.
    """
    keys = ["u", "v", "key"]
    if not os.path.exists(route_edges_file):
        print(f"  WARNING: {os.path.basename(route_edges_file)} not found. "
              "Falling back to the haul counts stored in the network file, "
              "which cover ALL routed pairs including any that manual review "
              "rejected. Re-run the routing notebook to produce the table.")
        return net
    if not all(c in net.columns for c in keys):
        print(f"  WARNING: network file lacks {keys}; cannot re-aggregate by "
              "route. Using the stored haul counts.")
        return net

    re_df = pd.read_csv(route_edges_file)
    stored = {c: pd.to_numeric(net[c], errors="coerce").fillna(0).sum()
              for c in ("h10", "h20", "km10", "km20")}

    COLMAP = {"hauls_10ton": "h10", "hauls_20ton": "h20",
              "km_10": "km10", "km_20": "km20", "km_tot": "kmtot"}

    def summed(frame):
        g = frame.groupby(keys)[list(COLMAP)].sum().rename(columns=COLMAP)
        return g.reset_index()

    all_agg = summed(re_df)
    for col in ("h10", "h20", "km10", "km20"):
        got = float(all_agg[col].sum())
        assert abs(got - stored[col]) <= max(1.0, 1e-6 * stored[col]), (
            f"the per-route table sums to {got:,.0f} {col} but the network file "
            f"stores {stored[col]:,.0f}; the table does not match this network")

    if not valid_only:
        use = all_agg
        n_routes = re_df["RouteID"].nunique()
    else:
        od = gpd.read_file(od_lines_file)
        ok = as_bool_series(od[VALID_COL]) if VALID_COL in od.columns else None
        if ok is None:
            print(f"  WARNING: '{VALID_COL}' absent from "
                  f"{os.path.basename(od_lines_file)}; keeping all routes.")
            use, n_routes = all_agg, re_df["RouteID"].nunique()
        else:
            keep = set(pd.to_numeric(od.loc[ok, "RouteID"], errors="coerce")
                       .dropna().astype(int))
            sel = re_df[re_df["RouteID"].isin(keep)]
            use = summed(sel)
            n_routes = sel["RouteID"].nunique()
            if verbose:
                print(f"  network hauls re-aggregated from {n_routes:,} of "
                      f"{re_df['RouteID'].nunique():,} routes "
                      f"('{VALID_COL}' is True)")
                print(f"    loaded hauls: 10t {use['h10'].sum():,.0f} of "
                      f"{stored['h10']:,.0f}   20t {use['h20'].sum():,.0f} of "
                      f"{stored['h20']:,.0f}")

    repl = [c for c in COLMAP.values() if c in net.columns]
    out = net.drop(columns=repl).merge(use, on=keys, how="left")
    out[list(COLMAP.values())] = out[list(COLMAP.values())].fillna(0.0)
    assert len(out) == len(net), "re-aggregation changed the edge count"
    return out


network = network_hauls_from_routes(network, ROUTE_EDGES_FILE, OD_LINES_FILE)

# h10 and h20 count loaded hauls; tonnage is computed from them before the
# round-trip factor, since manure crosses a segment once.
network['h10'] = pd.to_numeric(network['h10'], errors='coerce').fillna(0)
network['h20'] = pd.to_numeric(network['h20'], errors='coerce').fillna(0)
network['total_tons'] = (network['h10'] * 10) + (network['h20'] * 20)

# Truck passes per segment: loaded haul plus empty return.
network['pass10'] = network['h10'] * ROUND_TRIP_FACTOR
network['pass20'] = network['h20'] * ROUND_TRIP_FACTOR
network['htot'] = network['pass10'] + network['pass20']
network['edge_mi_10t'] = network['pass10'] * network['length_mi']
network['edge_mi_20t'] = network['pass20'] * network['length_mi']

# km10/km20 are loaded-basis kilometres built from the length_m column. Check in
# aggregate that the mile columns above are on the round-trip basis: per-edge
# lengths differ slightly from the measured geometry, but a basis error would
# be a factor of two.
_mi_stored = MI_PER_KM * float(
    pd.to_numeric(network['km10'], errors='coerce').fillna(0).sum()
    + pd.to_numeric(network['km20'], errors='coerce').fillna(0).sum())
_mi_here = float(network['edge_mi_10t'].sum() + network['edge_mi_20t'].sum())
assert abs(_mi_here - _mi_stored * ROUND_TRIP_FACTOR) / _mi_here < 0.01, \
    (f"stored truck-mi ({_mi_stored:,.0f} loaded) and recomputed "
     f"({_mi_here:,.0f} at x{ROUND_TRIP_FACTOR:g}) are on different bases")

# ---- emission and fuel rates ---------------------------------------------
# EMFAC2025 v2.1.1 emission rates: San Joaquin Valley APCD, calendar year 2024,
# annual, diesel, aggregate model year and speed. Categories are matched on
# vocation, not GVWR class: a "10-wheeler" is a three-axle single-unit dump
# truck of about 54,000 lb GVWR, so both truck types are Class 8 and differ by
# body type.
#   10-ton -> T7 Single Dump Class 8      20-ton -> T7 Tractor Class 8
# Only per-mile rates are used (RUNEX, PMTW, PMBW in g/mile). IDLEX
# (g/vehicle/day) and STREX (g/trip) cannot be applied to distance, so idle and
# start emissions are excluded. PM2.5 = running exhaust + tire wear + brake wear.
# Diesel is derived from CO2 at 10,190 g CO2 per gallon (EIA, Carbon Dioxide
# Emissions Coefficients by Fuel, "Diesel and Home Heating Fuel (Distillate Fuel
# Oil)", 10.19 kg/gal); EMFAC computes CO2 from fuel by carbon balance.
emfac_df = pd.read_csv(EMFAC_RATES_FILE).set_index('class')
EMFAC_META = {c: emfac_df.loc['10t', c] for c in
              ('emfac_version', 'region', 'calendar_year', 'season', 'fuel',
               'model_year', 'speed', 'pm25_components')}
CO2_G_PER_GAL = float(emfac_df.loc['10t', 'co2_g_per_gal'])

def _rates_per_mi(cls):
    """Per-mile rates for one truck class, as EMFAC publishes them."""
    r = emfac_df.loc[cls]
    out = {p: float(r[f'{k}_g_per_mi'])
           for p, k in (('PM2.5', 'PM25'), ('NOx', 'NOx'), ('ROG', 'ROG'),
                        ('CO2', 'CO2'))}
    out['gal'] = float(r['gal_per_mi'])
    return out

ef_c7_g_per_mi = _rates_per_mi('10t')
ef_c8_g_per_mi = _rates_per_mi('20t')
EMFAC_CATEGORY = {'10t': emfac_df.loc['10t', 'emfac_category'],
                  '20t': emfac_df.loc['20t', 'emfac_category']}

network['PM2.5_emissions'] = (network['edge_mi_10t'] * ef_c7_g_per_mi['PM2.5'] +
                              network['edge_mi_20t'] * ef_c8_g_per_mi['PM2.5'])
network['NOx_emissions']   = (network['edge_mi_10t'] * ef_c7_g_per_mi['NOx'] +
                              network['edge_mi_20t'] * ef_c8_g_per_mi['NOx'])
network['ROG_emissions']   = (network['edge_mi_10t'] * ef_c7_g_per_mi['ROG'] +
                              network['edge_mi_20t'] * ef_c8_g_per_mi['ROG'])
network['diesel_gal'] = (network['edge_mi_10t'] * ef_c7_g_per_mi['gal'] +
                         network['edge_mi_20t'] * ef_c8_g_per_mi['gal'])
network['CO2_kg'] = (network['edge_mi_10t'] * ef_c7_g_per_mi['CO2'] +
                     network['edge_mi_20t'] * ef_c8_g_per_mi['CO2']) / 1000.0

# Cities for map overlays
cities_data = {
    'city': ['Fresno', 'Sacramento', 'Bakersfield', 'Stockton', 'Modesto',
             'Roseville', 'Visalia', 'Redding', 'Chico',
             'Merced', 'Tulare', 'Porterville', 'Delano', 'Madera'],
    'pop': [542107, 524943, 403455, 320804, 218464,
            147773, 141384, 93611, 101475,
            86333, 68268, 62623, 51428, 66224],
    'lat': [36.7378, 38.5816, 35.3733, 37.9577, 37.6393,
            38.7521, 36.3302, 40.5865, 39.7285,
            37.3022, 36.2077, 36.0652, 35.7688, 36.9613],
    'lon': [-119.7871, -121.4944, -119.0187, -121.2908, -120.9969,
            -121.2880, -119.2921, -122.3917, -121.8375,
            -120.4830, -119.3473, -119.0168, -119.2471, -120.0607]
}
cities_df = pd.DataFrame(cities_data)
cities = gpd.GeoDataFrame(
    cities_df,
    geometry=gpd.points_from_xy(cities_df.lon, cities_df.lat),
    crs="EPSG:4326")

water_board_proj = water_board.to_crs(network.crs)
cities = cities.to_crs(network.crs)
cities = gpd.clip(cities, water_board_proj)
cities = cities.sort_values(by='pop', ascending=True).head(10)
# Drop city labels that crowd the southern map.
cities = cities[~cities['city'].isin(['Porterville', 'Tulare'])]

# Origin (dairy) and destination points of the valid routes in network.crs;
# the origins are drawn on the Figure S8 zooms.
manure_origins = gpd.GeoDataFrame(
    successfully_routed_df,
    geometry=gpd.points_from_xy(
        successfully_routed_df["Origin Dairy Longitude (Geocoded)"],
        successfully_routed_df["Origin Dairy Latitude (Geocoded)"]),
    crs="EPSG:4326",
).to_crs(network.crs)

manure_destinations = gpd.GeoDataFrame(
    successfully_routed_df,
    geometry=gpd.points_from_xy(
        successfully_routed_df["Destination Longitude (Geocoded)"],
        successfully_routed_df["Destination Latitude (Geocoded)"]),
    crs="EPSG:4326",
).to_crs(network.crs)

def add_base_overlays(ax):
    water_board_proj.plot(ax=ax, facecolor='none', edgecolor='dodgerblue',
                          linewidth=4, linestyle='--', zorder=4)
    cities.plot(ax=ax, color='black', markersize=90, zorder=8)
    for _, row in cities.iterrows():
        ax.text(row.geometry.x, row.geometry.y, row['city'],
                fontsize=19, ha='left', va='bottom', weight='bold',
                zorder=9, path_effects=CITY_LABEL_HALO)


# =========================================================================
# NETWORK LAYERS AND FIGURE S8
#   Truck-pass layer and colour scale shared by Figure 4b (last cell) and
#   both corridor zooms below.
#
#   Figure S8  a) Visalia / Pixley / Delano zoom  b) Modesto / Merced zoom
#
# Outputs: Figure_S8_Routing_Zooms.png
# =========================================================================
print("\nGenerating the network SI figures...")

positive_trips = network[network['htot'] > 0].sort_values(by='htot', ascending=True)

# One truck-pass scale for Figure 4b and both zooms, so the maps are comparable.
vmin_t = max(1, positive_trips['htot'].min()) if not positive_trips.empty else 1
vmax_t = positive_trips['htot'].max() if not positive_trips.empty else 1

# ---- Figure S8: the two corridor zooms --------------------------------
figS8 = plt.figure(figsize=(23, 11))
gsS7 = GridSpec(1, 2, figure=figS8, wspace=0.22)
axS8a = figS8.add_subplot(gsS7[0, 0])
axS8b = figS8.add_subplot(gsS7[0, 1])

# --- S8a: truck passes, Visalia / Pixley / Delano corridor ---
# Same layer and LogNorm as Figure 4b, cropped to the southern Tulare/Kern
# corridor, with its own city labels.
ZOOM_CITIES = pd.DataFrame({
    'city': ['Visalia', 'Pixley', 'Delano'],
    'lat':  [36.3302,    35.9686,  35.7688],
    'lon':  [-119.2921, -119.2856, -119.2471],
})
zoom_cities_gdf = gpd.GeoDataFrame(
    ZOOM_CITIES,
    geometry=gpd.points_from_xy(ZOOM_CITIES.lon, ZOOM_CITIES.lat),
    crs="EPSG:4326",
).to_crs(network.crs)

if not positive_trips.empty:
    positive_trips.plot(ax=axS8a, column='htot', cmap='YlOrRd',
                        norm=LogNorm(vmin=vmin_t, vmax=vmax_t),
                        legend=False, linewidth=4, zorder=3)

# Dairy origins (open blue circles, no fill).
manure_origins.plot(ax=axS8a, marker='o',
                    facecolor='none', edgecolor='blue', linewidth=1.5,
                    markersize=80, zorder=6)

zoom_cities_gdf.plot(ax=axS8a, color='black', markersize=120, zorder=8)
for _, row in zoom_cities_gdf.iterrows():
    axS8a.text(row.geometry.x, row.geometry.y, row['city'],
              fontsize=22, ha='left', va='bottom', weight='bold',
              zorder=9, path_effects=CITY_LABEL_HALO)

# Map extent, given in WGS84 and projected to network.crs for
# set_xlim/set_ylim; leaves room for the Visalia label.
zoom_corners_4326 = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy([-119.75, -118.85], [35.55, 36.55]),
    crs="EPSG:4326",
).to_crs(network.crs)
axS8a.set_xlim(zoom_corners_4326.geometry.x.min(), zoom_corners_4326.geometry.x.max())
axS8a.set_ylim(zoom_corners_4326.geometry.y.min(), zoom_corners_4326.geometry.y.max())
axS8a.set_aspect('equal')

# Lat/lon ticks every 0.25° on the top and left edges; black border.
add_latlon_ticks(axS8a, network.crs, fontsize=18,
                 sides=('top', 'left'),
                 lon_step=0.25, lat_step=0.25)
# Drop the topmost latitude tick, which crowds the top-left corner.
_yt = axS8a.get_yticks()
if len(_yt) > 1:
    axS8a.set_yticks(_yt[:-1])
for s in axS8a.spines.values():
    s.set_linewidth(1.5)

# Colorbar (truck passes, shared scale).
sm_d = cm.ScalarMappable(norm=LogNorm(vmin=vmin_t, vmax=vmax_t), cmap='YlOrRd')
cbar_d = figS8.colorbar(sm_d, ax=axS8a, shrink=0.85, pad=0.03)
cbar_d.set_label("Truck Passes (10t + 20t, loaded + return)",
                 fontsize=22, labelpad=14)
cbar_d.ax.tick_params(labelsize=19)
cbar_d.ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

# Dairy marker legend just below the panel.
dairy_handle = Line2D([0], [0], marker='o', color='blue',
                      markerfacecolor='none', markeredgewidth=1.5,
                      linestyle='None', markersize=12, label='Dairies')
axS8a.legend(handles=[dairy_handle],
            loc='upper center', bbox_to_anchor=(0.5, -0.02),
            fontsize=18, frameon=True, framealpha=0.85)

add_panel_label(axS8a, 'a)', fontsize=32)

# --- S8b: truck passes, Stockton / Modesto / Merced ---
# Same layer and scale as S8a. The eastern bound leaves room for the Merced
# label.
MODESTO_LAT, MODESTO_LON = 37.6393, -120.9969

ZOOM_E_LON_MIN, ZOOM_E_LON_MAX = -121.75, -120.05  # ~1.70° wide
ZOOM_E_LAT_MIN, ZOOM_E_LAT_MAX =  36.89,   38.39   # ~1.50° tall

ZOOM_CITIES_M = pd.DataFrame({
    'city': ['Stockton', 'Modesto', 'Merced'],
    'lat':  [37.9577,    MODESTO_LAT, 37.3022],
    'lon':  [-121.2908,  MODESTO_LON, -120.4830],
})
zoom_cities_m_gdf = gpd.GeoDataFrame(
    ZOOM_CITIES_M,
    geometry=gpd.points_from_xy(ZOOM_CITIES_M.lon, ZOOM_CITIES_M.lat),
    crs="EPSG:4326",
).to_crs(network.crs)

if not positive_trips.empty:
    positive_trips.plot(ax=axS8b, column='htot', cmap='YlOrRd',
                        norm=LogNorm(vmin=vmin_t, vmax=vmax_t),
                        legend=False, linewidth=4, zorder=3)

# Dairy origins (open blue circles, no fill).
manure_origins.plot(ax=axS8b, marker='o',
                    facecolor='none', edgecolor='blue', linewidth=1.5,
                    markersize=80, zorder=6)

zoom_cities_m_gdf.plot(ax=axS8b, color='black', markersize=120, zorder=8)
for _, row in zoom_cities_m_gdf.iterrows():
    axS8b.text(row.geometry.x, row.geometry.y, row['city'],
              fontsize=22, ha='left', va='bottom', weight='bold',
              zorder=9, path_effects=CITY_LABEL_HALO)

# Map extent in WGS84, projected to network.crs as for S8a.
zoom_corners_m_4326 = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(
        [ZOOM_E_LON_MIN, ZOOM_E_LON_MAX],
        [ZOOM_E_LAT_MIN, ZOOM_E_LAT_MAX],
    ),
    crs="EPSG:4326",
).to_crs(network.crs)
axS8b.set_xlim(zoom_corners_m_4326.geometry.x.min(),
              zoom_corners_m_4326.geometry.x.max())
axS8b.set_ylim(zoom_corners_m_4326.geometry.y.min(),
              zoom_corners_m_4326.geometry.y.max())
axS8b.set_aspect('equal')
# Anchor the map west, leaving a gutter on the right for the Merced label.
axS8b.set_anchor('W')

# Lat/lon ticks every 0.5° on the top and left edges; black border.
add_latlon_ticks(axS8b, network.crs, fontsize=18,
                 sides=('top', 'left'),
                 lon_step=0.5, lat_step=0.5)
for s in axS8b.spines.values():
    s.set_linewidth(1.5)

sm_e = cm.ScalarMappable(norm=LogNorm(vmin=vmin_t, vmax=vmax_t), cmap='YlOrRd')
cbar_e = figS8.colorbar(sm_e, ax=axS8b, shrink=0.85, pad=0.03)
cbar_e.set_label("Truck Passes (10t + 20t, loaded + return)",
                 fontsize=22, labelpad=14)
cbar_e.ax.tick_params(labelsize=19)
cbar_e.ax.yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

# Same Dairies legend as S8a.
axS8b.legend(handles=[dairy_handle],
            loc='upper center', bbox_to_anchor=(0.5, -0.02),
            fontsize=18, frameon=True, framealpha=0.85)

add_panel_label(axS8b, 'b)', fontsize=32)

save_figure(figS8, "Figure_S8_Routing_Zooms.png",
            dpi=300, bbox_inches='tight', pad_inches=0.6)
plt.close(figS8)
print("Saved Figure S8.")


# =========================================================================
# TOP LONGEST VALIDATED ROUTES
# Outputs: Table_S7_Longest_Routes.csv
# =========================================================================
TOP_N_ROUTES = 10

if not os.path.exists(OD_LINES_FILE):
    print(f"\n{OD_LINES_FILE} not found; skipping the longest-route table")
else:
    top_lines = gpd.read_file(OD_LINES_FILE)
    for _c in ('route_km', 'total_truck_km'):
        top_lines[_c.replace('_km', '_mi')] = (
            pd.to_numeric(top_lines[_c], errors='coerce') * MI_PER_KM)
    if VALID_COL in top_lines.columns:
        keep = as_bool_series(top_lines[VALID_COL])
        basis = VALID_COL
    else:
        keep = top_lines['routing_status'].eq('success')
        basis = "routing_status == 'success'"
        print(f"\n  '{VALID_COL}' absent from {os.path.basename(OD_LINES_FILE)}; "
              "run review/manifests/geocode-review/Manure_Manifest_Review.ipynb to include the manual review")

    top = top_lines[keep].copy()
    for col in ('route_mi', 'total_truck_mi', 'Hauls_10ton', 'Hauls_20ton'):
        top[col] = pd.to_numeric(top[col], errors='coerce').fillna(0)
    top['Trips'] = top['Hauls_10ton'] + top['Hauls_20ton']
    # 'Tons' is the capacity of the assigned trucks; 'Total Manure Amount
    # (tons)' is the reported shipment, from which the haul counts are rounded
    # up to whole trucks.
    top['Tons'] = top['Hauls_10ton'] * 10 + top['Hauls_20ton'] * 20
    top['RouteID'] = pd.to_numeric(top['RouteID'], errors='coerce')
    _rep_tons = manure_df[['RouteID', 'Total Manure Amount (tons)']].copy()
    _rep_tons['RouteID'] = pd.to_numeric(_rep_tons['RouteID'], errors='coerce')
    top = top.merge(_rep_tons, on='RouteID', how='left')
    assert top['Total Manure Amount (tons)'].notna().all(), \
        'a validated route has no tonnage on its manifest'
    top = top.nlargest(TOP_N_ROUTES, 'route_mi')

    def _txt(row, col, default='(none)'):
        v = row.get(col)
        return default if v is None or pd.isna(v) or str(v).strip() == '' else str(v).strip()

    print("\n" + "=" * 78)
    print(f"{TOP_N_ROUTES} LONGEST VALIDATED ROUTES  (of {int(keep.sum()):,} on {basis})")
    print("=" * 78)
    for rank, (_, row) in enumerate(top.iterrows(), start=1):
        print(f"\n{rank:>2}. RouteID {int(row['RouteID'])}   "
              f"{row['route_mi']:,.1f} {DIST_UNIT} one way")
        print(f"      origin      : {_txt(row, 'origin_name')} - "
              f"{_txt(row, 'origin_address')}  [{_txt(row, 'Origin County')}]")
        print(f"      destination : {_txt(row, 'dest_address')}  "
              f"[{_txt(row, 'dest_type')}, {_txt(row, 'Destination County')}]")
        print(f"      trips       : {row['Hauls_10ton']:,.0f} x 10-ton + "
              f"{row['Hauls_20ton']:,.0f} x 20-ton = {row['Trips']:,.0f} trips")
        print(f"      tonnage     : {row['Tons']:,.0f} tons")
        print(f"      distance    : {row['total_truck_mi']:,.0f} loaded truck-mi "
              f"over all hauls (x{ROUND_TRIP_FACTOR:g} with the return)")

    print("\n" + "-" * 78)
    print(f"{'these ' + str(len(top)) + ' routes':<34}"
          f"{top['Trips'].sum():>10,.0f} trips  "
          f"{top['Tons'].sum():>12,.0f} tons  {top['total_truck_mi'].sum():>14,.0f} truck-mi")
    allv = top_lines[keep].copy()
    for col in ('total_truck_mi', 'Hauls_10ton', 'Hauls_20ton'):
        allv[col] = pd.to_numeric(allv[col], errors='coerce').fillna(0)
    a_trips = allv['Hauls_10ton'].sum() + allv['Hauls_20ton'].sum()
    a_tons = allv['Hauls_10ton'].sum() * 10 + allv['Hauls_20ton'].sum() * 20
    print(f"{'all validated routes':<34}{a_trips:>10,.0f} trips  "
          f"{a_tons:>12,.0f} tons  {allv['total_truck_mi'].sum():>14,.0f} truck-mi")
    TOP_ROUTES_TRIP_SHARE = top['Trips'].sum() / a_trips
    TOP_ROUTES_TON_SHARE = top['Tons'].sum() / a_tons
    TOP_ROUTES_TRUCK_MI_SHARE = top['total_truck_mi'].sum() / allv['total_truck_mi'].sum()
    print(f"{'share from the longest ' + str(len(top)):<34}"
          f"{TOP_ROUTES_TRIP_SHARE:>9.1%} trips  "
          f"{TOP_ROUTES_TON_SHARE:>11.1%} tons  "
          f"{TOP_ROUTES_TRUCK_MI_SHARE:>13.1%} truck-mi")

    # =====================================================================
    # TABLE S7: the same routes as a CSV
    # =====================================================================
    table_s7 = (top.assign(Rank=range(1, len(top) + 1))
                .rename(columns={
                    'origin_name': 'Origin dairy',
                    'origin_address': 'Origin address',
                    'Origin County': 'Origin county',
                    'dest_address': 'Destination address',
                    'dest_type': 'Destination type',
                    'Destination County': 'Destination county',
                    'route_mi': 'Route distance one way (mi)',
                    'Hauls_10ton': '10-ton truck trips',
                    'Hauls_20ton': '20-ton truck trips',
                    'Trips': 'Total truck trips',
                    'Total Manure Amount (tons)': 'Manure shipped, reported (tons)',
                    'Tons': 'Truck capacity implied (tons)',
                    'total_truck_mi': 'Loaded truck-mi over all hauls'})
                [['Rank', 'RouteID', 'Origin dairy', 'Origin address', 'Origin county',
                  'Destination address', 'Destination type', 'Destination county',
                  'Route distance one way (mi)', '10-ton truck trips',
                  '20-ton truck trips', 'Total truck trips',
                  'Manure shipped, reported (tons)', 'Truck capacity implied (tons)',
                  'Loaded truck-mi over all hauls']]
                .round({'Route distance one way (mi)': 1,
                        'Manure shipped, reported (tons)': 1,
                        'Truck capacity implied (tons)': 0,
                        'Loaded truck-mi over all hauls': 0}))
    table_s7.to_csv(out_path("Table_S7_Longest_Routes.csv"), index=False)

    # Report routed pairs longer than the Table S7 cutoff that failed validation.
    _all_rt = top_lines.copy()
    _all_rt['route_mi'] = pd.to_numeric(_all_rt['route_mi'], errors='coerce')
    _dropped = _all_rt[(~keep) & _all_rt['route_mi'].gt(top['route_mi'].min())]
    print(f"\n{len(_dropped)} longer routed pairs carry a geocode or route flag and are "
          f"excluded from Table S7")
    for _, _r in _dropped.nlargest(4, 'route_mi').iterrows():
        print(f"    {_r['route_mi']:>7,.1f} {DIST_UNIT}  "
              f"{str(_r['origin_name'])[:24]:<24} "
              f"({_r['validation reason']})")
    print(f"  wrote {out_path('Table_S7_Longest_Routes.csv')}")

# =========================================================================
# WHY ROUTES FAIL, AND WHICH FACILITIES ARE AFFECTED
#   Manifests by validation reason (automatic checks and the manual review
#   verdicts from review/manifests/geocode-review/Manure_Manifest_Review.ipynb),
#   with the dairies affected.
# =========================================================================
TOP_FACILITIES = 5
HERD_BINS = [0, 500, 1500, 3000, float("inf")]
HERD_LABELS = ["<500", "500-1,500", "1,500-3,000", "3,000+"]

fail = manure_df.copy()
fail["_reason"] = fail[REASON_COL].fillna("(validated)").astype(str).str.strip()
fail["_tons"] = pd.to_numeric(fail["Total Manure Amount (tons)"], errors="coerce").fillna(0)
fail["_trips"] = (pd.to_numeric(fail["Number of 10-ton Hauls for Analysis"], errors="coerce").fillna(0)
                  + pd.to_numeric(fail["Number of 20-ton Hauls for Analysis"], errors="coerce").fillna(0))
fail["_dairy"] = (fail["Origin Dairy Name"].fillna("(unnamed)").astype(str).str.strip()
                  + " | " + fail["Origin Dairy Address"].fillna("").astype(str).str.strip())
fail["_dest"] = fail["Destination Address Final"].fillna("(none)").astype(str).str.strip()
fail["_herd"] = (pd.to_numeric(fail["Total Herd Size"], errors="coerce")
                 if "Total Herd Size" in fail.columns else pd.Series(np.nan, index=fail.index))
fail["_size"] = pd.cut(fail["_herd"], bins=HERD_BINS, labels=HERD_LABELS, right=False)

# Origin county, for locating the affected dairies.
if os.path.exists(COUNTIES_FILE):
    _co = gpd.read_file(COUNTIES_FILE)
    if _co.crs != "EPSG:4326":
        _co = _co.to_crs("EPSG:4326")
    _co_col = next((c for c in ["NAME", "COUNTY", "NAMELSAD", "County"] if c in _co.columns),
                   _co.columns[0])
    _olat = pd.to_numeric(fail["Origin Dairy Latitude (Geocoded)"], errors="coerce")
    _olon = pd.to_numeric(fail["Origin Dairy Longitude (Geocoded)"], errors="coerce")
    _ok = _olat.notna() & _olon.notna()
    _pts = gpd.GeoDataFrame(geometry=gpd.points_from_xy(_olon[_ok], _olat[_ok]),
                            index=fail.index[_ok], crs="EPSG:4326")
    _j = gpd.sjoin(_pts, _co[[_co_col, "geometry"]], how="left", predicate="intersects")
    _j = _j[~_j.index.duplicated(keep="first")]
    fail["_county"] = _j[_co_col].reindex(fail.index)
else:
    print(f"  {COUNTIES_FILE} not found; county breakdown omitted")
    fail["_county"] = np.nan

reason_order = [r for r in M_DROP_REASONS if (fail["_reason"] == r).any()] + ["(validated)"]
n_total, n_valid = len(fail), int((fail["_reason"] == "(validated)").sum())
tons_total = fail["_tons"].sum()

print("\n" + "=" * 96)
print("WHY ROUTES FAIL, AND WHICH FACILITIES ARE AFFECTED")
print("=" * 96)
print(f"{n_total:,} manifests: {n_valid:,} validated ({n_valid / n_total:.1%}), "
      f"{n_total - n_valid:,} excluded")
print(f"{tons_total:,.0f} reported tons in total\n")

hdr = (f"{'reason':<26}{'manifests':>10}{'dairies':>9}{'dests':>7}"
       f"{'tons':>12}{'% tons':>8}{'trips':>10}{'med herd':>10}")
print(hdr); print("-" * len(hdr))
for r in reason_order:
    sl = fail[fail["_reason"] == r]
    med = sl["_herd"].median()
    print(f"{r:<26}{len(sl):>10,}{sl['_dairy'].nunique():>9,}{sl['_dest'].nunique():>7,}"
          f"{sl['_tons'].sum():>12,.0f}{sl['_tons'].sum() / tons_total:>8.1%}"
          f"{sl['_trips'].sum():>10,.0f}"
          f"{('' if pd.isna(med) else f'{med:,.0f}'):>10}")

# Per-reason facility profile.
for r in reason_order:
    if r == "(validated)":
        continue
    sl = fail[fail["_reason"] == r]
    print(f"\n--- {r}  ({len(sl):,} manifests, {sl['_tons'].sum():,.0f} tons) ---")
    dt = sl["Dest_Type"].value_counts()
    print("   destination types : " + ", ".join(f"{k} {v}" for k, v in dt.head(5).items()))
    if sl["_county"].notna().any():
        co = sl["_county"].value_counts()
        print("   origin counties   : " + ", ".join(f"{k} {v}" for k, v in co.head(5).items()))
    if sl["_size"].notna().any():
        sz = sl["_size"].value_counts().reindex(HERD_LABELS).dropna()
        print("   dairy herd size   : " + ", ".join(f"{k} {int(v)}" for k, v in sz.items()))
    src_col = "Destination Address Final Source"
    if src_col in sl.columns and sl[src_col].notna().any():
        s2 = sl[src_col].value_counts()
        print("   address source    : " + ", ".join(f"{k} {v}" for k, v in s2.head(3).items()))
    worst = (sl.groupby("_dairy")
               .agg(manifests=("_dairy", "size"), tons=("_tons", "sum"))
               .nlargest(TOP_FACILITIES, "tons"))
    print(f"   dairies affected  : {sl['_dairy'].nunique():,}")
    for dairy, row in worst.iterrows():
        print(f"       {dairy.split(' | ')[0][:42]:<44}"
              f"{int(row['manifests']):>3} manifests  {row['tons']:>10,.0f} tons")

# Dairies fully or partly excluded from the analysis.
by_dairy = fail.groupby("_dairy").agg(
    manifests=("_dairy", "size"),
    validated=("_reason", lambda s: int((s == "(validated)").sum())),
    tons=("_tons", "sum"))
lost = by_dairy[by_dairy["validated"] == 0]
partial = by_dairy[(by_dairy["validated"] > 0) & (by_dairy["validated"] < by_dairy["manifests"])]
print("\n" + "-" * 96)
print("FACILITY-LEVEL EXPOSURE")
print("-" * 96)
print(f"{'dairies in the manifests':<40}{len(by_dairy):>8,}")
print(f"{'...fully excluded (no validated manifest)':<40}{len(lost):>8,}"
      f"   holding {lost['tons'].sum():,.0f} tons ({lost['tons'].sum() / tons_total:.1%})")
print(f"{'...partially excluded':<40}{len(partial):>8,}"
      f"   holding {partial['tons'].sum():,.0f} tons")
print(f"{'...fully retained':<40}"
      f"{len(by_dairy) - len(lost) - len(partial):>8,}")
if len(lost):
    print(f"\n   largest fully excluded dairies:")
    for dairy, row in lost.nlargest(TOP_FACILITIES, "tons").iterrows():
        why = fail.loc[fail["_dairy"] == dairy, "_reason"].value_counts()
        print(f"       {dairy.split(' | ')[0][:42]:<44}{row['tons']:>10,.0f} tons   "
              + ", ".join(f"{k} x{v}" for k, v in why.items()))

# =========================================================================
# SUMMARY STATISTICS
# =========================================================================
print("\n" + "=" * 60)
print("FINAL SUMMARY STATISTICS")
print("=" * 60)

print("\n--- MANURE ---")
print(f"Total Manifests Processed:            {len(manure_df):,}")
print(f"Total Manure (All Manifests):         {manure_df['Total Manure Amount (tons)'].sum():,.2f} tons")
print(f"Successfully Routed Pairs:            {m_final_successfully_routed:,}")

t10 = network['edge_mi_10t'].sum()
t20 = network['edge_mi_20t'].sum()
loaded_mi = float(((network['h10'] + network['h20']) * network['length_mi']).sum())
print(f"Loaded haul-miles (one way):          {loaded_mi:,.2f} mi")
print(f"Vehicle-miles travelled (loaded x {ROUND_TRIP_FACTOR:g}):")
print(f"  10-Ton Trucks: {t10:,.2f} mi")
print(f"  20-Ton Trucks: {t20:,.2f} mi")
print(f"  Total:         {t10 + t20:,.2f} mi")

tot_pm  = network['PM2.5_emissions'].sum()
tot_nox = network['NOx_emissions'].sum()
tot_rog = network['ROG_emissions'].sum()
print("Total Pollutants:")
print(f"  PM2.5: {tot_pm:,.2f} g ({tot_pm/1000:,.2f} kg)")
print(f"  NOx:   {tot_nox:,.2f} g ({tot_nox/1000:,.2f} kg)")
print(f"  ROG:   {tot_rog:,.2f} g ({tot_rog/1000:,.2f} kg)")

# =========================================================================
# DIESEL FUEL AND CO2
# =========================================================================
print(f"\n--- DIESEL FUEL AND CO2 (loaded + empty return) ---")
print(f"{EMFAC_META['emfac_version']}; {EMFAC_META['region']}; "
      f"CY{EMFAC_META['calendar_year']}; {EMFAC_META['season']}; {EMFAC_META['fuel']}; "
      f"model year {EMFAC_META['model_year']}, speed {EMFAC_META['speed']}")
print(f"  10-ton -> {EMFAC_CATEGORY['10t']}    20-ton -> {EMFAC_CATEGORY['20t']}")
_fuel = {}
for _cls, _mi, _rates in (('10-ton', t10, ef_c7_g_per_mi),
                          ('20-ton', t20, ef_c8_g_per_mi)):
    _fuel[_cls] = dict(mi=_mi, gal=_mi * _rates['gal'],
                       co2_t=_mi * _rates['CO2'] / 1e6,
                       mpg=CO2_G_PER_GAL / _rates['CO2'])
print("Diesel burned:")
for _cls, _v in _fuel.items():
    print(f"  {_cls}:{_v['gal']:>12,.0f} gal   at {_v['mpg']:.2f} mpg "
          f"({_v['mi']:,.0f} mi)")
_gal_tot = sum(v['gal'] for v in _fuel.values())
print(f"  Total: {_gal_tot:>13,.0f} gal")
print("CO2 from fuel:")
for _cls, _v in _fuel.items():
    print(f"  {_cls}:{_v['co2_t']:>12,.0f} t")
_co2_tot = sum(v['co2_t'] for v in _fuel.values())
print(f"  Total: {_co2_tot:>13,.0f} t")
# The network CO2 column is a separate aggregation of the same total and must
# agree with the per-class sum.
_co2_net = float(network['CO2_kg'].sum()) / 1000.0
print(f"  cross-check: network CO2 column = {_co2_net:,.0f} t   "
      f"(delta {abs(_co2_net - _co2_tot) / _co2_tot * 100:.1f}%)")
print(f"  fuel x {CO2_G_PER_GAL / 1000:.2f} kg CO2/gal = "
      f"{_gal_tot * CO2_G_PER_GAL / 1e6:,.0f} t")
print("Excludes idle and start emissions: EMFAC RUNEX rates are running exhaust only.")
assert abs(_gal_tot - float(network['diesel_gal'].sum())) < 1.0, \
    "per-class diesel does not sum to the network total"
assert abs(_co2_tot - _co2_net) < 1.0, "CO2 paths disagree"

if INCLUDE_WASTEWATER:
    print("\n--- WASTEWATER ---")
    print(f"Total Manifests Processed:            {len(ww_df):,}")
    print(f"Total Wastewater (All Manifests):     {ww_df[WW_VOL_COL].sum():,.2f} gallons")
    print(f"  Trucked:  {ww_trucked_count:,}")
    print(f"  Pipeline: {ww_pipeline_count:,}")
    print(f"  Unsure:   {ww_unsure_count:,}")

In [ ]:
# =========================================================================
# TABLE S5: BALANCE OF OBSERVABLES — SUCCESSFULLY ROUTED vs DROPPED
#   Selection diagnostic for the routing attrition (Figure S6). Each dropped
#   manifest carries one first-failing reason, so the drop groups are mutually
#   exclusive. Groups are reported separately as well as pooled, since
#   selection in opposite directions can cancel when pooled.
#
#   Each group is compared with the routed set by a two-sided Mann-Whitney U
#   test, reported with the median difference and the rank-biserial
#   correlation as the effect size.
#
#   Outputs:
#     Table_S5_Dropped_Route_Comparison.csv
# =========================================================================
print("\n" + "=" * 60)
print("TABLE S5: BALANCE OF OBSERVABLES (ROUTED vs DROPPED)")
print("=" * 60)

from scipy.stats import mannwhitneyu

# Continuous observables: manifest tonnage, and origin-dairy animal units
# (computed in the first cell from the annual report herd classes, at the
# factors in Table 1 of Order No. R5-2008-0066).
BALANCE_CONT_VARS = [
    ("Total Manure Amount (tons)", "Total manure (tons)"),
    ("Animal Units",               "Herd size (animal units)"),
]


def build_balance_groups(valid_mask, reason):
    """Ordered {group label: boolean mask}: routed, one per drop reason, pooled.

    Reasons follow M_DROP_REASONS order, then any others sorted; reasons with
    no dropped manifests are omitted.
    """
    groups = {"Routed": valid_mask.to_numpy()}
    present = [r for r in M_DROP_REASONS if (reason[~valid_mask] == r).any()]
    extra = sorted({r for r in reason[~valid_mask].unique()
                    if r and r not in M_DROP_REASONS})
    if extra:
        print(f"  Note: drop reason(s) not in M_DROP_REASONS: {extra}")
    for r in present + extra:
        groups[f"Dropped: {r}"] = ((~valid_mask) & (reason == r)).to_numpy()
    groups["Dropped: ALL"] = (~valid_mask).to_numpy()
    return groups


def continuous_balance(df, groups, variables, ref="Routed"):
    """N, mean (SD), median, and a Mann-Whitney U test of each group vs routed.

    Rank-biserial correlation: 0 means the groups are fully interleaved, +1
    that every value in the group exceeds every routed value, -1 the reverse.
    """
    ref_mask = groups[ref]
    rows = []
    for col, label in variables:
        if col not in df.columns:
            print(f"  Warning: '{col}' not in manure_df; skipping.")
            continue
        vals = pd.to_numeric(df[col], errors="coerce")
        ref_v = vals[ref_mask].dropna()
        for gname, gmask in groups.items():
            g = vals[gmask]
            gv = g.dropna()
            if gname == ref or len(gv) < 2 or len(ref_v) < 2:
                mdiff, rb, pval = np.nan, np.nan, np.nan
            else:
                u, pval = mannwhitneyu(gv, ref_v, alternative="two-sided")
                mdiff = gv.median() - ref_v.median()
                rb = 2.0 * u / (len(gv) * len(ref_v)) - 1.0
            rows.append({
                "variable": label, "group": gname,
                "n": int(len(gv)), "n_missing": int(g.isna().sum()),
                "pct_missing": 100.0 * g.isna().mean() if len(g) else np.nan,
                "mean": gv.mean() if len(gv) else np.nan,
                "sd": gv.std(ddof=1) if len(gv) > 1 else np.nan,
                "median": gv.median() if len(gv) else np.nan,
                "median_diff_vs_routed": mdiff,
                "rank_biserial": rb, "p_value": pval,
            })
    return pd.DataFrame(rows)


# --- build ----------------------------------------------------------------
balance_groups = build_balance_groups(m_valid_mask, m_reason)
_pdf_col = pick_col(manure_df, "Source PDF")

print("\nGroup sizes (n_dairies = unique Source PDFs behind the manifests):")
for _g, _m in balance_groups.items():
    print(f"  {_g:<34} n_manifests={int(_m.sum()):>5}  "
          f"n_dairies={manure_df.loc[_m, _pdf_col].nunique():>5}")

balance_cont = continuous_balance(manure_df, balance_groups, BALANCE_CONT_VARS)

# --- print ----------------------------------------------------------------
_order = list(balance_groups.keys())
_W = 124
print("\n" + "=" * _W)
print("TABLE S5. MANIFEST-LEVEL BALANCE: successfully routed vs dropped")
print("=" * _W)
print(f"{'Variable':<22}{'Group':<32}{'N':>6}{'%miss':>7}"
      f"{'Mean':>12}{'SD':>12}{'Median':>10}{'MedDiff':>11}{'r_rb':>8}{'p':>9}")
print("-" * _W)
for _var in balance_cont["variable"].unique():
    _sub = (balance_cont[balance_cont["variable"] == _var]
            .set_index("group").reindex(_order).reset_index())
    for _i, _r in _sub.iterrows():
        if pd.isna(_r["n"]):
            continue
        _lbl = _var if _i == 0 else ""
        _d = ("" if pd.isna(_r["median_diff_vs_routed"])
              else f"{_r['median_diff_vs_routed']:,.1f}")
        _rb = "" if pd.isna(_r["rank_biserial"]) else f"{_r['rank_biserial']:+.2f}"
        _p = "" if pd.isna(_r["p_value"]) else f"{_r['p_value']:.3f}"
        print(f"{_lbl:<22}{_r['group']:<32}{int(_r['n']):>6}"
              f"{_r['pct_missing']:>6.0f}%{_r['mean']:>12,.1f}{_r['sd']:>12,.1f}"
              f"{_r['median']:>10,.1f}{_d:>11}{_rb:>8}{_p:>9}")
    print("-" * _W)
print("Mann-Whitney U vs the routed group, two-sided. r_rb is the rank-biserial "
      "correlation:\n0 means the groups are interleaved, +1 that every value in "
      "the group exceeds every routed value.")
print(f"{len(BALANCE_CONT_VARS)} variables x {len(_order) - 1} drop groups = "
      f"{len(BALANCE_CONT_VARS) * (len(_order) - 1)} tests with no multiplicity "
      "correction; treat individual\np-values near 0.05 with corresponding caution.")
print("Herd size in animal units is a property of the origin dairy repeated "
      "across that dairy's\nmanifests, so its p-values count one facility many "
      "times and are optimistic. Read the medians\nand the effect size there "
      "rather than the p-value. A manifest whose origin dairy left any herd\n"
      "class blank carries no animal-unit figure, since the total is a weighted "
      "sum.")

# --- save -----------------------------------------------------------------
(balance_cont
 .round({"pct_missing": 1, "mean": 1, "sd": 1, "median": 1,
         "median_diff_vs_routed": 1, "rank_biserial": 3, "p_value": 6})
 .to_csv(out_path("Table_S5_Dropped_Route_Comparison.csv"), index=False))
print("\nSaved Table S5.")


In [ ]:
# =========================================================================
# FIGURE 4 INPUTS: BUFFER-BASED TRUCKING EXPOSURE AND INCOME
#
# Method (near-roadway exposure after Rowangould 2013, applied to the
# manure-hauling network):
#
#   Every road segment carrying hauls is buffered by BUFFER_DISTANCE_M. Each
#   segment's truck-miles are spread uniformly over its own buffer (a VMT
#   density), and each block group receives the share falling inside it:
#
#       VMT_j = sum_e  VMT_e * area(B_e & u_j) / area(B_e)
#
#   Overlapping buffers add. The allocation is mass-conserving: sum_j VMT_j
#   equals the network total, less buffer area outside the retained block
#   groups (checked below).
#
#   Dividing by buffer area makes the result independent of how OSM splits
#   edges; a flat VMT_e over the buffer would weight a 3 mi segment ten times
#   a 0.3 mi segment carrying the same trucks. Unlike a length-share
#   allocation, the buffer lets a road just outside a block group contribute
#   exposure to it.
#
#   Limitation: the flat buffer weights all distances within it equally,
#   while near-road gradients fall by roughly half within 100-150 m, so this
#   is a proximity-weighted allocation, not a concentration estimate, and
#   understates the burden on the nearest households.
#
#   q_tab feeds Figure 4e (exposure by income quintile).
#
# Outputs:
#   BlockGroup_Exposure.csv
#   Exposure_by_Income_Quintile.csv
#   exposure_pairs_<D>m cache (parquet, else pickle)
# =========================================================================
print("\n" + "=" * 60)
print(f"FIGURE 4: BUFFER EXPOSURE ({BUFFER_DISTANCE_M} m) vs INCOME")
print("=" * 60)

from scipy.stats import spearmanr, mannwhitneyu, kruskal


def _union_all(geoseries):
    """Union of a GeoSeries: union_all() in geopandas >= 0.14, else unary_union."""
    try:
        return geoseries.union_all()
    except AttributeError:
        return geoseries.unary_union


def _cache_write(df, stem):
    try:
        df.to_parquet(f"{stem}.parquet", index=False)
        return f"{stem}.parquet"
    except Exception:
        df.to_pickle(f"{stem}.pkl")
        return f"{stem}.pkl"


def _cache_read(stem):
    for path, reader in ((f"{stem}.parquet", pd.read_parquet),
                         (f"{stem}.pkl", pd.read_pickle)):
        if os.path.exists(path):
            if (os.path.exists(NETWORK_SHAPEFILE) and
                    os.path.getmtime(path) < os.path.getmtime(NETWORK_SHAPEFILE)):
                print(f"  cache {os.path.basename(path)} is older than the "
                      "network file; recomputing.")
                return None
            print(f"  loaded cached pair table: {os.path.basename(path)}")
            return reader(path)
    return None


# =========================================================================
# STAGE 1 — canonicalise the network
# =========================================================================
def prepare_exposure_network(network, analysis_crs=ANALYSIS_CRS, verbose=True):
    """One row per undirected road segment, in `analysis_crs`, carrying
    per-truck-class VMT.

      1. VMT is computed in the network's native CRS (EPSG:32610) before
         reprojecting, so the total matches the network VMT used elsewhere.
      2. Zero-length and empty geometries are dropped: their buffers have zero
         area, which would make the VMT density divide by zero.
      3. Reciprocal (u,v)/(v,u) directed edges are dissolved onto one geometry
         with VMT summed, since both directions are traffic. This leaves the
         allocation unchanged and keeps the fraction-exposed statistic from
         counting a strip twice. Rounded length is part of the grouping key,
         so parallel edges between the same nodes stay separate.
    """
    net = network.copy()
    if net.crs is None:
        raise ValueError("network has no CRS; expected EPSG:32610 for v13.")

    net["length_mi"] = net.geometry.length / M_PER_MILE
    net["h10"] = pd.to_numeric(net.get("h10"), errors="coerce").fillna(0)
    net["h20"] = pd.to_numeric(net.get("h20"), errors="coerce").fillna(0)
    net["VMT10"] = net["h10"] * net["length_mi"]
    net["VMT20"] = net["h20"] * net["length_mi"]
    vmt_before = float(net["VMT10"].sum() + net["VMT20"].sum())

    net = net.to_crs(analysis_crs)   # areas and buffer distances from here on

    n0 = len(net)
    bad = net.geometry.isna() | net.geometry.is_empty | (net.geometry.length <= 0)
    dropped_vmt = float(net.loc[bad, ["VMT10", "VMT20"]].to_numpy().sum())
    net = net[~bad].copy()
    if verbose:
        print(f"  dropped {n0 - len(net):,} zero-length/empty edges "
              f"(carrying {dropped_vmt:,.1f} VMT)")

    if {"u", "v"}.issubset(net.columns):
        net["_pair"] = [(min(a, b), max(a, b)) for a, b in zip(net["u"], net["v"])]
        net["_len_key"] = net.geometry.length.round(1)
        agg = (net.groupby(["_pair", "_len_key"], sort=False)
                  .agg(VMT10=("VMT10", "sum"), VMT20=("VMT20", "sum"),
                       h10=("h10", "sum"), h20=("h20", "sum"),
                       geometry=("geometry", "first"))
                  .reset_index(drop=True))
        net = gpd.GeoDataFrame(agg, geometry="geometry", crs=analysis_crs)
        if verbose:
            print(f"  dissolved directed duplicates -> "
                  f"{len(net):,} undirected segments")

    vmt_after = float(net["VMT10"].sum() + net["VMT20"].sum())
    assert np.isclose(vmt_after, vmt_before - dropped_vmt, rtol=1e-9), (
        f"VMT not conserved by network prep: {vmt_before:,.2f} -> {vmt_after:,.2f}")
    if verbose:
        print(f"  VMT conserved: {vmt_after:,.1f} mi "
              f"(10t {net['VMT10'].sum():,.1f} / 20t {net['VMT20'].sum():,.1f})")
    return net


# =========================================================================
# BLOCK GROUPS — canonical loader
# =========================================================================
def load_block_groups(analysis_crs=ANALYSIS_CRS, verbose=True):
    """Block groups majority-inside the Water Board, with income and population.

    A block group is retained when more than half its area falls inside the
    Water Board boundary; its full unclipped geometry is then used for
    exposure and area, so the boundary defines the study area but does not
    mask it.
    """
    bg = gpd.read_file(BG_SHP_FILE)
    inc = pd.read_csv(BG_CSV_FILE, encoding="latin1")
    keep = ["GISJOIN", BG_INCOME_COL] + (
        [BG_INCOME_MOE_COL] if BG_INCOME_MOE_COL in inc.columns else [])
    for extra in (BG_INCOME_PC_COL, BG_INCOME_PC_MOE_COL):
        if extra and extra in inc.columns:
            keep.append(extra)
    inc = inc[keep].copy()
    for c in (BG_INCOME_COL, BG_INCOME_PC_COL):
        if c and c in inc.columns:
            inc[c] = pd.to_numeric(inc[c], errors="coerce")
    bg = bg.merge(inc, on="GISJOIN", how="inner")

    # --- population (needed for the per-capita normalization) ---
    pop_col = None
    if BG_POP_CSV_FILE and os.path.exists(BG_POP_CSV_FILE):
        pop = pd.read_csv(BG_POP_CSV_FILE, encoding="latin1")
        pop_col = BG_POP_COL
        if pop_col is None:
            # Auto-detect: NHGIS ACS estimate columns are <4-character table
            # code>E<digits> (B01003 is AUO6 in ds272). Take the first, other
            # than the income column, whose block-group median is a plausible
            # person count (50-20,000).
            cands = [c for c in pop.columns
                     if len(c) == 8 and c[0].isalpha() and c[:4].isalnum()
                     and c[4] == "E" and c[5:].isdigit() and c != BG_INCOME_COL]
            for c in cands:
                med = pd.to_numeric(pop[c], errors="coerce").median()
                if pd.notna(med) and 50 <= med <= 20000:
                    pop_col = c
                    break
            if pop_col:
                print(f"  auto-detected block-group population column: {pop_col}")
            else:
                print(f"  WARNING: no population column auto-detected among "
                      f"{cands[:8]}. Set BG_POP_COL explicitly.")
        if pop_col and pop_col in pop.columns:
            pop = pop[["GISJOIN", pop_col]].copy()
            pop[pop_col] = pd.to_numeric(pop[pop_col], errors="coerce")
            bg = bg.merge(pop.rename(columns={pop_col: "bg_pop"}),
                          on="GISJOIN", how="left")
    if "bg_pop" not in bg.columns:
        bg["bg_pop"] = np.nan
        print("  WARNING: no block-group population; per-capita normalization "
              "will be unavailable and only area-normalized results reported.")

    bgp = bg.to_crs(analysis_crs)
    wbp = water_board.to_crs(analysis_crs)
    bgp["orig_area"] = bgp.geometry.area
    clipped = gpd.clip(bgp, wbp)
    ca = (clipped.assign(_a=clipped.geometry.area)
                 .groupby("GISJOIN")["_a"].sum().rename("clipped_area"))
    bgp = bgp.merge(ca, on="GISJOIN", how="left")
    bgp["clipped_area"] = bgp["clipped_area"].fillna(0)
    bgp = bgp[(bgp["clipped_area"] / bgp["orig_area"]) > 0.5].copy()

    bgp["unit_area_mi2"] = bgp.geometry.area / M2_PER_MI2
    # Area denominator: TIGER ALAND (land area, m^2, excludes water) where
    # available, else geometric area.
    if "ALAND" in bgp.columns:
        bgp["land_area_mi2"] = (
            pd.to_numeric(bgp["ALAND"], errors="coerce") / M2_PER_MI2)
        bgp["land_area_mi2"] = bgp["land_area_mi2"].fillna(bgp["unit_area_mi2"])
    else:
        bgp["land_area_mi2"] = bgp["unit_area_mi2"]
    if verbose:
        print(f"  block groups retained (majority in Water Board): {len(bgp):,}")
        print(f"    with income > 0 : {int((bgp[BG_INCOME_COL] > 0).sum()):,}")
        print(f"    with population : {int(bgp['bg_pop'].notna().sum()):,}")
    return bgp.reset_index(drop=True)


# =========================================================================
# STAGE 2 — buffer x block-group intersection table (the expensive step)
# =========================================================================
def build_exposure_pairs(net, units, unit_id="GISJOIN", buffer_m=None,
                         verbose=True, unit_label="block group"):
    """Edge-by-unit pair table with `area_frac` = area(B_e & u_j) / area(B_e).

    Cached by the caller; exposure metrics downstream are derived from it.
    """
    buffer_m = BUFFER_DISTANCE_M if buffer_m is None else buffer_m
    buf = net[["VMT10", "VMT20", "h10", "h20"]].copy()
    # Round caps: buffers of connected segments overlap at junctions, which the
    # additive rule counts as the extra traffic there.
    buf["geometry"] = net.geometry.buffer(buffer_m)
    buf = gpd.GeoDataFrame(buf, geometry="geometry", crs=net.crs)
    buf["buffer_area"] = buf.geometry.area
    buf["edge_idx"] = np.arange(len(buf))
    if (buf["buffer_area"] <= 0).any():
        raise ValueError("zero-area buffer survived network prep; "
                         "cannot form a VMT density.")

    pairs = gpd.sjoin(
        buf[["edge_idx", "buffer_area", "geometry"]],
        units[[unit_id, "geometry"]], how="inner", predicate="intersects")
    if verbose:
        print(f"  candidate edge x {unit_label} pairs: {len(pairs):,}")

    unit_geom = units.set_index(unit_id).geometry
    left_geom = buf.set_index("edge_idx").geometry
    pairs = pairs.reset_index(drop=True)
    inter = (left_geom.loc[pairs["edge_idx"].to_numpy()]
             .reset_index(drop=True)
             .intersection(gpd.GeoSeries(
                 unit_geom.loc[pairs[unit_id].to_numpy()].to_numpy(),
                 crs=units.crs), align=False))
    pairs["inter_area"] = inter.area.to_numpy()
    pairs = pairs[pairs["inter_area"] > 0].copy()
    pairs["area_frac"] = pairs["inter_area"] / pairs["buffer_area"]
    return pairs.drop(columns=["geometry", "index_right"], errors="ignore")


def attach_edge_vmt(pairs, net):
    """Join per-edge VMT from `net` onto a cached geometric pair table.

    The cached table depends only on the network geometry and the units; VMT
    depends on the selected routes, so it is joined on each run. edge_idx is
    the positional index into `net`, so both must describe the same edges.
    """
    p = pairs.drop(columns=[c for c in ("VMT10", "VMT20", "h10", "h20")
                            if c in pairs.columns])
    assert int(p["edge_idx"].max()) < len(net), (
        f"pair table references edge {int(p['edge_idx'].max()):,} but the network "
        f"has {len(net):,} edges; delete the cache and recompute")
    src = net[["VMT10", "VMT20"]].reset_index(drop=True)
    src["edge_idx"] = np.arange(len(net))
    out = p.merge(src, on="edge_idx", how="left")
    assert out["VMT10"].notna().all(), "an edge in the pair table is not in the network"
    return out


def aggregate_exposure(pairs, units, unit_id="GISJOIN", verbose=True,
                       unit_label="block groups"):
    """Collapse the pair table to one row per unit and convert to pollutant
    mass with the EMFAC rates derived in the first cell.

    Pair-table VMT is loaded truck-mi; ROUND_TRIP_FACTOR is applied here.
    """
    p = pairs.copy()
    p["VMT10_alloc"] = p["VMT10"] * p["area_frac"] * ROUND_TRIP_FACTOR
    p["VMT20_alloc"] = p["VMT20"] * p["area_frac"] * ROUND_TRIP_FACTOR
    agg = (p.groupby(unit_id)[["VMT10_alloc", "VMT20_alloc"]].sum()
            .reset_index()
            .rename(columns={"VMT10_alloc": "VMT10_j",
                             "VMT20_alloc": "VMT20_j"}))
    out = units.merge(agg, on=unit_id, how="left")
    out[["VMT10_j", "VMT20_j"]] = out[["VMT10_j", "VMT20_j"]].fillna(0.0)
    out["VMT_j"] = out["VMT10_j"] + out["VMT20_j"]

    # Same per-class factors as network['PM2.5_emissions'] etc., so unit totals
    # reconcile with the network emissions.
    for pol in ("PM2.5", "NOx", "ROG"):
        out[f"{pol}_g_j"] = (out["VMT10_j"] * ef_c7_g_per_mi[pol] +
                             out["VMT20_j"] * ef_c8_g_per_mi[pol])
    if verbose:
        print(f"  {unit_label} with non-zero exposure: "
              f"{int((out['VMT_j'] > 0).sum()):,} / {len(out):,}")
    return out


def add_fraction_exposed(exposure, net, buffer_m=None, unit_id="GISJOIN",
                         verbose=True):
    """P_j = area((union of all buffers) & u_j) / area(u_j).

    Buffers are unioned first, so overlapping strips are counted once.
    """
    buffer_m = BUFFER_DISTANCE_M if buffer_m is None else buffer_m
    union = _union_all(net.geometry.buffer(buffer_m))
    if verbose:
        print(f"  buffer union area: {union.area / M2_PER_MI2:,.0f} mi2 "
              f"({100 * union.area / water_board.to_crs(net.crs).area.sum():.1f}% "
              "of the Water Board region)")
    union_gdf = gpd.GeoDataFrame(geometry=[union], crs=net.crs)
    clipped = gpd.overlay(exposure[[unit_id, "geometry"]], union_gdf,
                          how="intersection", keep_geom_type=True)
    exposed = (clipped.assign(_a=clipped.geometry.area / M2_PER_MI2)
                      .groupby(unit_id)["_a"].sum().rename("exposed_mi2"))
    out = exposure.merge(exposed, on=unit_id, how="left")
    out["exposed_mi2"] = out["exposed_mi2"].fillna(0.0)
    out["frac_exposed"] = out["exposed_mi2"] / out["unit_area_mi2"]
    return out


# =========================================================================
# WEIGHTED STATISTICS HELPERS
# =========================================================================
def weighted_quantile_bins(values, weights, n_bins=5):
    """Cut points placing an equal share of TOTAL WEIGHT in each bin.

    With population weights, each income quintile holds about a fifth of the
    region's residents.
    """
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    v, w = v[ok], w[ok]
    order = np.argsort(v)
    v, w = v[order], w[order]
    cw = np.cumsum(w) / w.sum()
    qs = np.linspace(0, 1, n_bins + 1)[1:-1]
    cuts = [v[np.searchsorted(cw, q)] for q in qs]
    return [-np.inf] + list(cuts) + [np.inf]


def weighted_mean(values, weights):
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    return np.average(v[ok], weights=w[ok]) if ok.any() else np.nan


def bootstrap_weighted_mean_ci(values, weights, n_boot=2000, alpha=0.05, seed=0):
    """Percentile CI for a population-weighted mean, resampling block groups."""
    v = np.asarray(values, dtype=float)
    w = np.asarray(weights, dtype=float)
    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    v, w = v[ok], w[ok]
    if len(v) < 2:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(v), size=(n_boot, len(v)))
    boots = (v[idx] * w[idx]).sum(axis=1) / w[idx].sum(axis=1)
    return np.percentile(boots, 100 * alpha / 2), np.percentile(boots, 100 * (1 - alpha / 2))


def bootstrap_spearman(x, y, n_boot=2000, seed=0):
    """95% percentile CI for Spearman's rho by resampling block groups.

    Reported beside the analytic p-value, which is near zero at n in the
    thousands for any non-trivial rho.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 3:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    n = len(x)
    draws = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, n)
        draws[b] = spearmanr(x[idx], y[idx]).statistic
    draws = draws[np.isfinite(draws)]
    if draws.size == 0:
        return np.nan, np.nan
    return tuple(np.percentile(draws, [2.5, 97.5]))

def concentration_curve(income, exposure, population):
    """Cumulative share of exposure against cumulative share of population,
    with block groups ranked by income ascending.

    Returns (pop_share, exposure_share, concentration_index). The index is
    twice the signed area between the curve and the 45-degree line: positive
    means exposure is concentrated among LOWER-income residents.
    """
    d = pd.DataFrame({"inc": income, "exp": exposure, "pop": population}).dropna()
    d = d[(d["pop"] > 0)].sort_values("inc")
    if d.empty or d["exp"].sum() <= 0:
        return np.array([0, 1]), np.array([0, 1]), np.nan
    cp = np.concatenate([[0], np.cumsum(d["pop"].to_numpy()) / d["pop"].sum()])
    ce = np.concatenate([[0], np.cumsum(d["exp"].to_numpy()) / d["exp"].sum()])
    # np.trapezoid in NumPy >= 2.0, np.trapz before.
    _trapz = getattr(np, "trapezoid", None) or np.trapz
    ci = 2.0 * (_trapz(ce, cp) - 0.5)
    return cp, ce, ci


# =========================================================================
# RUN
# =========================================================================
HAVE_BG_EXPOSURE = False
HAVE_PLACE_EXPOSURE = False

if not (os.path.exists(BG_SHP_FILE) and os.path.exists(BG_CSV_FILE)):
    print("  WARNING: Blockgroup shapefile or census CSV not found; "
          "skipping Figure 4e.")
else:
    print("\nStage 1: canonicalising the network...")
    exp_net = prepare_exposure_network(network)

    print("\nLoading block groups...")
    bg_units = load_block_groups()

    print(f"\nStage 2: buffering at {BUFFER_DISTANCE_M} m and intersecting...")
    _stem = os.path.join(CACHE_DIR, f"exposure_pairs_{BUFFER_DISTANCE_M}m")
    exp_pairs = _cache_read(_stem)
    if exp_pairs is None:
        exp_pairs = build_exposure_pairs(exp_net, bg_units)
        print(f"  cached -> {os.path.basename(_cache_write(exp_pairs, _stem))}")
    exp_pairs = attach_edge_vmt(exp_pairs, exp_net)
    # Area-weighted pair-table VMT should recover the loaded network VMT, less
    # buffer area outside the retained block groups.
    _cache_vmt = float(((exp_pairs["VMT10"] + exp_pairs["VMT20"])
                        * exp_pairs["area_frac"]).sum())
    _net_vmt = float(exp_net["VMT10"].sum() + exp_net["VMT20"].sum())
    assert 0.95 < _cache_vmt / _net_vmt <= 1.001, (
        f"pair table allocates {_cache_vmt:,.0f} mi against a loaded network "
        f"total of {_net_vmt:,.0f} mi ({_cache_vmt / _net_vmt:.2f}x)")

    print("\nStage 3: aggregating to block groups...")
    bg_exp = aggregate_exposure(exp_pairs, bg_units)
    bg_exp = add_fraction_exposed(bg_exp, exp_net)

    # --- normalizations ---------------------------------------------------
    bg_exp["VMT_per_mi2"] = bg_exp["VMT_j"] / bg_exp["land_area_mi2"]
    bg_exp["VMT_per_capita"] = np.where(
        bg_exp["bg_pop"] > 0, bg_exp["VMT_j"] / bg_exp["bg_pop"], np.nan)
    bg_exp["PM25_per_capita"] = np.where(
        bg_exp["bg_pop"] > 0, bg_exp["PM2.5_g_j"] / bg_exp["bg_pop"], np.nan)

    # --- analysis sample --------------------------------------------------
    # Income <= 0 is NHGIS suppression, not a real value.
    n_all = len(bg_exp)
    corr4 = bg_exp[bg_exp[BG_INCOME_COL].notna() & (bg_exp[BG_INCOME_COL] > 0)].copy()
    n_dropped_income = n_all - len(corr4)
    has_pop = corr4["bg_pop"].notna() & (corr4["bg_pop"] > 0)
    print(f"\n  analysis sample: {len(corr4):,} block groups "
          f"({n_dropped_income:,} dropped for missing/suppressed income)")
    print(f"    of these, with usable population: {int(has_pop.sum()):,}")

    USE_PER_CAPITA = bool(has_pop.sum() > 0.5 * len(corr4))
    EXPOSURE_COL = "VMT_per_capita" if USE_PER_CAPITA else "VMT_per_mi2"
    EXPOSURE_LABEL = ("Truck-mi within "
                      f"{BUFFER_DISTANCE_M} m per resident"
                      if USE_PER_CAPITA else
                      f"Truck-mi within {BUFFER_DISTANCE_M} m per {AREA_UNIT}")
    # The other normalization, shown beside the first in Figure 4e.
    ALT_EXPOSURE_COL = "VMT_per_mi2" if USE_PER_CAPITA else "VMT_per_capita"
    ALT_EXPOSURE_LABEL = (f"Truck-mi per {AREA_UNIT}" if USE_PER_CAPITA
                          else "Truck-mi per resident")
    EXPOSURE_SHORT = {"VMT_per_capita": "per resident", "VMT_per_mi2": "per mi2"}
    print(f"    primary exposure metric: {EXPOSURE_COL}")

    # --- income quintiles (population-weighted where possible) ------------
    if USE_PER_CAPITA:
        q_bins = weighted_quantile_bins(corr4[BG_INCOME_COL], corr4["bg_pop"], 5)
        q_weight = corr4["bg_pop"]
    else:
        q_bins = weighted_quantile_bins(corr4[BG_INCOME_COL],
                                        np.ones(len(corr4)), 5)
        q_weight = pd.Series(np.ones(len(corr4)), index=corr4.index)
    corr4["Income_Quintile"] = pd.cut(
        corr4[BG_INCOME_COL], bins=q_bins,
        labels=[f"Q{i + 1}" for i in range(5)], include_lowest=True)
    corr4["_w"] = q_weight.to_numpy()

    # Bootstrap CIs for both normalizations, with CORR_BOOTSTRAP_DRAWS draws.
    q_rows = []
    for q, sub in corr4.groupby("Income_Quintile", observed=True):
        lo, hi = bootstrap_weighted_mean_ci(sub[EXPOSURE_COL], sub["_w"],
                                            n_boot=CORR_BOOTSTRAP_DRAWS)
        alt_lo, alt_hi = bootstrap_weighted_mean_ci(sub[ALT_EXPOSURE_COL], sub["_w"],
                                                    n_boot=CORR_BOOTSTRAP_DRAWS)
        q_rows.append({
            "quintile": str(q),
            "n_bg": len(sub),
            "income_min": sub[BG_INCOME_COL].min(),
            "income_max": sub[BG_INCOME_COL].max(),
            "pop": sub["bg_pop"].sum(skipna=True),
            "mean_exposure": weighted_mean(sub[EXPOSURE_COL], sub["_w"]),
            "ci_lo": lo, "ci_hi": hi,
            "alt_mean": weighted_mean(sub[ALT_EXPOSURE_COL], sub["_w"]),
            "alt_ci_lo": alt_lo, "alt_ci_hi": alt_hi,
            "total_VMT": sub["VMT_j"].sum(),
            "total_PM25_g": sub["PM2.5_g_j"].sum(),
        })
    q_tab = pd.DataFrame(q_rows)
    q_tab["pop_share_pct"] = 100 * q_tab["pop"] / q_tab["pop"].sum()
    q_tab["vmt_share_pct"] = 100 * q_tab["total_VMT"] / q_tab["total_VMT"].sum()

    # --- statistics -------------------------------------------------------
    _s = corr4[corr4[EXPOSURE_COL].notna()]
    rho_all, p_all = spearmanr(_s[BG_INCOME_COL], _s[EXPOSURE_COL])
    _a = _s[_s[EXPOSURE_COL] > 0]
    rho_act, p_act = (spearmanr(_a[BG_INCOME_COL], _a[EXPOSURE_COL])
                      if len(_a) > 2 else (np.nan, np.nan))
    cp, ce, conc_idx = concentration_curve(
        corr4[BG_INCOME_COL], corr4["VMT_j"], corr4["bg_pop"])

    print("\n" + "-" * 70)
    print("EXPOSURE vs INCOME")
    print("-" * 70)
    print(f"  sample: all {len(corr4):,} block groups in the Water Board with a "
          f"reported income, including the "
          f"{int((corr4['VMT_j'] <= 0).sum()):,} carrying no trucking")
    print(f"  mean_exposure is {EXPOSURE_COL}, alt_mean is {ALT_EXPOSURE_COL}; "
          "both are population-weighted")
    print(f"  ci_lo/ci_hi and alt_ci_lo/alt_ci_hi: nonparametric bootstrap, "
          f"{CORR_BOOTSTRAP_DRAWS:,} draws.")
    print( "    Block groups are resampled with replacement WITHIN each quintile, "
           "at that")
    print( "    quintile's own n, so quintile membership is held fixed and only "
           "sampling")
    print( "    variation in the block groups is propagated. Each draw recomputes "
           "the")
    print( "    population-weighted mean; the interval is the 2.5th-97.5th "
           "percentile of")
    print( "    the draws. Seeded, so the interval is reproducible.")
    print(q_tab.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
    print(f"\n  Spearman rho (all BGs, n={len(_s):,}):      "
          f"{rho_all:+.3f}  (p={p_all:.3g})")
    print(f"  Spearman rho (exposed only, n={len(_a):,}):  "
          f"{rho_act:+.3f}  (p={p_act:.3g})")
    print(f"  Concentration index (pop-ranked by income): {conc_idx:+.3f}  "
          "(positive = exposure concentrated among lower incomes)")

    # --- rank correlation by income measure and sample --------------------
    # Unweighted (each block group counts once), for median household income
    # and, where available, per capita income.
    income_measures = [("median household", BG_INCOME_COL)]
    if BG_INCOME_PC_COL and BG_INCOME_PC_COL in corr4.columns:
        income_measures.append(("per capita", BG_INCOME_PC_COL))

    # Run for both normalizations (both plotted in Figure 4e). The exposed-only
    # sample is drawn per metric, so a block group missing one denominator does
    # not enter the other metric's sample.
    corr_rows = []
    for exp_col in (EXPOSURE_COL, ALT_EXPOSURE_COL):
        _e_all = corr4[corr4[exp_col].notna()]
        _e_exp = _e_all[_e_all[exp_col] > 0]
        for inc_label, inc_col in income_measures:
            for samp_label, samp in (("all BGs", _e_all), ("exposed only", _e_exp)):
                sub_df = samp[samp[inc_col].notna() & (samp[inc_col] > 0)]
                if len(sub_df) < 3:
                    continue
                xi = sub_df[inc_col].to_numpy(float)
                yi = sub_df[exp_col].to_numpy(float)
                rho_i, p_i = spearmanr(xi, yi)
                lo, hi = bootstrap_spearman(xi, yi, n_boot=CORR_BOOTSTRAP_DRAWS)
                corr_rows.append({
                    "exposure_metric": EXPOSURE_SHORT.get(exp_col, exp_col),
                    "exposure_col": exp_col,
                    "is_primary": exp_col == EXPOSURE_COL,
                    "income_measure": inc_label, "sample": samp_label,
                    "n_bg": len(sub_df), "rho": rho_i, "p_value": p_i,
                    "ci_lo": lo, "ci_hi": hi,
                })

    corr_tab = pd.DataFrame(corr_rows)
    if not corr_tab.empty:
        print(f"\n  Spearman rho of exposure with income "
              f"(bootstrap CI over block groups, {CORR_BOOTSTRAP_DRAWS:,} draws)")
        hdr = (f"    {'exposure':<14}{'income':<18}{'sample':<14}{'n_bg':>7}"
               f"{'rho':>10}{'p':>11}{'95% CI':>22}")
        print(hdr)
        print("    " + "-" * (len(hdr) - 4))
        for _, r in corr_tab.iterrows():
            ci = f"[{r['ci_lo']:+.2f}, {r['ci_hi']:+.2f}]"
            print(f"    {r['exposure_metric']:<14}{r['income_measure']:<18}"
                  f"{r['sample']:<14}{int(r['n_bg']):>7,}{r['rho']:>10.3f}"
                  f"{r['p_value']:>11.2g}{ci:>22}")

    if not q_tab.empty:
        _q1 = q_tab.iloc[0]
        print(f"  Lowest-income quintile holds {_q1['pop_share_pct']:.1f}% of "
              f"population and {_q1['vmt_share_pct']:.1f}% of near-road VMT.")

    # =====================================================================
    # VALIDATION
    # =====================================================================
    print("\n" + "-" * 70)
    print("VALIDATION")
    print("-" * 70)
    # exp_net holds loaded VMT and bg_exp includes ROUND_TRIP_FACTOR, so the
    # network total is scaled to the same basis.
    _vmt_net = float(exp_net["VMT10"].sum()
                     + exp_net["VMT20"].sum()) * ROUND_TRIP_FACTOR
    _vmt_alloc = float(bg_exp["VMT_j"].sum())
    print(f"  network VMT                  : {_vmt_net:,.1f} mi")
    print(f"  allocated to block groups    : {_vmt_alloc:,.1f} mi "
          f"({100 * _vmt_alloc / _vmt_net:.2f}% retained)")
    print( "  (shortfall = buffer area outside the retained block groups)")
    _pm_net = float(network["PM2.5_emissions"].sum())
    _pm_alloc = float(bg_exp["PM2.5_g_j"].sum())
    print(f"  network PM2.5                 : {_pm_net:,.1f} g")
    print(f"  allocated PM2.5               : {_pm_alloc:,.1f} g "
          f"({100 * _pm_alloc / _pm_net:.2f}% retained)")

    _per_edge = exp_pairs.groupby("edge_idx")["area_frac"].sum()
    print(f"  max sum(area_frac) per edge  : {_per_edge.max():.6f} (must be <= 1)")
    assert _per_edge.max() < 1 + 1e-6, "buffer area fractions exceed 1"
    assert bg_exp["frac_exposed"].max() <= 1 + 1e-6, "frac_exposed exceeds 1"
    print(f"  frac_exposed range           : {bg_exp['frac_exposed'].min():.3f} "
          f"- {bg_exp['frac_exposed'].max():.3f}")
    print(f"  block groups with zero exposure: "
          f"{int((bg_exp['VMT_j'] == 0).sum()):,} / {len(bg_exp):,}")


    # --- export -----------------------------------------------------------
    _export_cols = ["GISJOIN", BG_INCOME_COL, "bg_pop", "unit_area_mi2",
                    "land_area_mi2", "VMT10_j", "VMT20_j", "VMT_j",
                    "PM2.5_g_j", "NOx_g_j", "ROG_g_j", "exposed_mi2",
                    "frac_exposed", "VMT_per_mi2", "VMT_per_capita",
                    "PM25_per_capita"]
    (pd.DataFrame(bg_exp)[[c for c in _export_cols if c in bg_exp.columns]]
       .to_csv(out_path("BlockGroup_Exposure.csv"), index=False))
    # Correlations are appended as constant columns on every quintile row.
    q_out = q_tab.copy()
    for _, r in corr_tab.iterrows():
        pre = "rho" if r["is_primary"] else "rho_alt"
        tag = ("pc" if r["income_measure"] == "per capita" else "mhi")
        samp = ("all" if r["sample"] == "all BGs" else "exposed")
        q_out[f"{pre}_{tag}_{samp}"] = r["rho"]
        q_out[f"{pre}_{tag}_{samp}_p"] = r["p_value"]
        q_out[f"{pre}_{tag}_{samp}_ci_lo"] = r["ci_lo"]
        q_out[f"{pre}_{tag}_{samp}_ci_hi"] = r["ci_hi"]
    q_out.to_csv(out_path("Exposure_by_Income_Quintile.csv"), index=False)
    print("Saved BlockGroup_Exposure.csv and Exposure_by_Income_Quintile.csv.")
    HAVE_BG_EXPOSURE = True

# =========================================================================
# TABLES S8-S11: TRUCK EXPOSURE BY CENSUS PLACE
#   The block-group buffer allocation above, applied to 2024 Census place
#   boundaries. Tables S8-S10 rank the top PLACE_TOP_N places by total
#   truck-mi, truck-mi per capita and truck-mi per mi2 of land. Table S11
#   collapses the same figures by community type and population size class.
#
#   Places are cities, towns (incorporated in California, grouped with cities)
#   and census designated places. Population is the 2020-2024 ACS five-year
#   total. A place is retained when more than half its area falls inside the
#   Water Board.
# =========================================================================
PLACE_TOP_N = 20
PLACE_SIZE_BINS = [0, 500, 2500, 10000, 50000, np.inf]
PLACE_SIZE_LABELS = ["<0.5k", "0.5-2.5k", "2.5-10k", "10-50k", ">=50k"]
PLACE_COVERAGE_THRESHOLDS = (0.50, 0.75, 0.95)

if not (os.path.exists(PLACE_SHP_FILE) and os.path.exists(PLACE_POP_FILE)):
    print(f"\n  place boundaries or population not found; skipping Tables S8-S11")
else:
    print("\n" + "=" * 96)
    print("TABLES S8-S11: TRUCK EXPOSURE BY CENSUS PLACE")
    print("=" * 96)
    _pl = gpd.read_file(PLACE_SHP_FILE).to_crs(ANALYSIS_CRS)
    _ap = pd.read_csv(PLACE_POP_FILE, low_memory=False)
    # GEOID = NHGIS state code + place code, as concatenated in the shapefile.
    _ap["GEOID"] = (_ap["STATEA"].astype(str).str.zfill(2)
                    + _ap["PLACEA"].astype(str).str.zfill(5))
    _pl = _pl.merge(_ap[["GEOID", "AUO6E001"]].rename(
        columns={"AUO6E001": "place_pop"}), on="GEOID", how="left")
    _LSAD_LABEL = {"25": "city", "43": "town", "57": "CDP"}
    _pl["place_type"] = _pl["LSAD"].map(_LSAD_LABEL).fillna("other")

    _pl["orig_area"] = _pl.geometry.area
    _pclip = gpd.clip(_pl, water_board.to_crs(ANALYSIS_CRS))
    _pca = (_pclip.assign(_a=_pclip.geometry.area)
                  .groupby("GEOID")["_a"].sum().rename("clipped_area"))
    _pl = _pl.merge(_pca, on="GEOID", how="left")
    _pl["clipped_area"] = _pl["clipped_area"].fillna(0.0)
    _pl = _pl[(_pl["clipped_area"] / _pl["orig_area"]) > 0.5].copy()
    # Two area denominators: ALAND (excludes inland water) for VMT density, and
    # polygon area for add_fraction_exposed, so its coverage share cannot
    # exceed 1.
    _pl["unit_area_mi2"] = _pl.geometry.area / M2_PER_MI2
    _pl["land_area_mi2"] = pd.to_numeric(_pl["ALAND"], errors="coerce") / M2_PER_MI2
    _pl["land_area_mi2"] = _pl["land_area_mi2"].fillna(_pl["unit_area_mi2"])
    _pl = _pl.reset_index(drop=True)
    print(f"  places retained (majority in Water Board): {len(_pl):,}   "
          + ", ".join(f"{v} {k}" for k, v in
                      _pl['place_type'].value_counts().items()))

    _pstem = os.path.join(CACHE_DIR, f"exposure_pairs_place_{BUFFER_DISTANCE_M}m")
    _ppairs = _cache_read(_pstem)
    if _ppairs is None:
        _ppairs = build_exposure_pairs(exp_net, _pl, unit_id="GEOID",
                                       unit_label="place")
        print(f"  cached -> {os.path.basename(_cache_write(_ppairs, _pstem))}")
    # The cache is keyed on the network only; check it matches this place set.
    assert set(_ppairs["GEOID"]) <= set(_pl["GEOID"]), (
        "cached place pair table holds units outside the retained place set; "
        f"delete {_pstem}.pkl and re-run")
    _ppairs = attach_edge_vmt(_ppairs, exp_net)
    place_exp = aggregate_exposure(_ppairs, _pl, unit_id="GEOID",
                                   unit_label="places")
    place_exp = add_fraction_exposed(place_exp, exp_net, unit_id="GEOID")
    # Tolerance for places wholly inside the buffer union (1 plus rounding).
    assert place_exp["frac_exposed"].between(0.0, 1.0 + 1e-9).all(), \
        "frac_exposed outside [0, 1]; check the area denominator"

    # Rates are NaN where the denominator is zero.
    place_exp["vmt_per_capita"] = np.where(
        place_exp["place_pop"] > 0,
        place_exp["VMT_j"] / place_exp["place_pop"], np.nan)
    place_exp["vmt_per_mi2"] = np.where(
        place_exp["land_area_mi2"] > 0,
        place_exp["VMT_j"] / place_exp["land_area_mi2"], np.nan)
    place_exp["size_class"] = pd.cut(place_exp["place_pop"],
                                     bins=PLACE_SIZE_BINS,
                                     labels=PLACE_SIZE_LABELS, right=False)

    _in_place = float(place_exp["VMT_j"].sum())
    _net_vmt = float(exp_net["VMT10"].sum() + exp_net["VMT20"].sum()) * ROUND_TRIP_FACTOR
    print(f"  truck-mi falling inside any place: {_in_place:,.0f} of {_net_vmt:,.0f} "
          f"({100 * _in_place / _net_vmt:.1f}%)")
    print(f"  the remainder runs on roads outside every city, town and CDP")

    _PCOLS = {"NAMELSAD": "Place", "place_type": "Type", "place_pop": "Population",
              "land_area_mi2": "Land area (mi2)", "VMT_j": "Truck-mi",
              "vmt_per_capita": "Truck-mi per capita",
              "vmt_per_mi2": "Truck-mi per mi2"}
    _hit = place_exp[place_exp["VMT_j"] > 0].copy()

    for _n, _col, _fname, _desc in (
            ("S8", "VMT_j", "Table_S8_Place_VMT_Total.csv", "total truck-mi"),
            ("S9", "vmt_per_capita", "Table_S9_Place_VMT_Per_Capita.csv",
             "truck-mi per capita"),
            ("S10", "vmt_per_mi2", "Table_S10_Place_VMT_Per_Area.csv",
             "truck-mi per mi2")):
        _t = (_hit.dropna(subset=[_col]).nlargest(PLACE_TOP_N, _col)
              [list(_PCOLS)].rename(columns=_PCOLS).reset_index(drop=True))
        _t.insert(0, "Rank", range(1, len(_t) + 1))
        _t = _t.round({"Land area (mi2)": 2, "Truck-mi": 0,
                       "Truck-mi per capita": 2, "Truck-mi per mi2": 0})
        _t.to_csv(out_path(_fname), index=False)
        print(f"\n  Table {_n}: top {PLACE_TOP_N} by {_desc}  ->  {_fname}")
        print(_t.head(10).to_string(index=False))

    _small = _hit.dropna(subset=["vmt_per_capita"]).nlargest(
        PLACE_TOP_N, "vmt_per_capita")
    print(f"\n  Table S9 note: {int((_small['place_pop'] < 500).sum())} of the top "
          f"{PLACE_TOP_N} per-capita places have fewer than 500 residents, so those "
          f"rates rest on\n  small denominators. Population is carried in the table "
          f"so the reader can weigh them.")

    # ---------------------------------------------------------------------
    # Table S11: exposure by community type and population size class.
    # ---------------------------------------------------------------------
    def place_group_summary(units, group_col, label):
        """One row per group, with rates pooled over places rather than averaged.

        Place counts and total population cover every retained place in the
        group; rates use only places carrying traffic, so they describe a
        resident of an exposed place. Pooling makes per capita = per mi2 /
        people per mi2 hold exactly, and avoids weighting small places as
        heavily as large ones.
        """
        hit = units["VMT_j"] > 0
        a = units.assign(
            _exposed=hit.astype(int),
            _pop_exp=np.where(hit, units["place_pop"], 0.0),
            _area_exp=np.where(hit, units["land_area_mi2"], 0.0),
            _frac_exp=np.where(hit, units["frac_exposed"], np.nan))
        g = a.groupby(group_col, observed=False)
        out = pd.DataFrame({
            "Places in region": g.size(),
            "Places exposed": g["_exposed"].sum(),
            "Population": g["place_pop"].sum(),
            "Population in exposed places": g["_pop_exp"].sum(),
            "Truck-mi": g["VMT_j"].sum(),
            "_area": g["_area_exp"].sum(),
            "Mean share of area within 500 m (%)": 100 * g["_frac_exp"].mean(),
        })
        _pop_exp = out["Population in exposed places"]
        out["Incidence (%)"] = 100 * out["Places exposed"] / out["Places in region"]
        out["Population in exposed places (%)"] = np.where(
            out["Population"] > 0, 100 * _pop_exp / out["Population"], np.nan)
        out["Share of truck-mi (%)"] = 100 * out["Truck-mi"] / out["Truck-mi"].sum()
        out["Share of exposed population (%)"] = 100 * _pop_exp / _pop_exp.sum()
        out["Trucking exposure ratio"] = np.where(
            out["Share of exposed population (%)"] > 0,
            out["Share of truck-mi (%)"] / out["Share of exposed population (%)"],
            np.nan)
        out["Truck-mi per capita"] = np.where(
            _pop_exp > 0, out["Truck-mi"] / _pop_exp, np.nan)
        out["Truck-mi per mi2"] = np.where(
            out["_area"] > 0, out["Truck-mi"] / out["_area"], np.nan)
        out["People per mi2"] = np.where(
            out["_area"] > 0, _pop_exp / out["_area"], np.nan)
        return (out.drop(columns="_area").reset_index()
                   .rename(columns={group_col: "Group"})
                   .assign(Grouping=label))

    S10_COLS = ["Grouping", "Group", "Places in region", "Places exposed",
                "Incidence (%)", "Population",
                "Population in exposed places (%)", "Truck-mi",
                "Share of truck-mi (%)", "Share of exposed population (%)",
                "Trucking exposure ratio", "Truck-mi per capita",
                "Truck-mi per mi2", "People per mi2",
                "Mean share of area within 500 m (%)"]
    place_summary = pd.concat(
        [place_group_summary(place_exp, "place_type", "Community type"),
         place_group_summary(place_exp, "size_class", "Population size")],
        ignore_index=True)[S10_COLS]
    place_summary["Group"] = place_summary["Group"].astype(str)

    _ok = place_summary["Truck-mi per capita"].notna()
    assert np.allclose(
        place_summary.loc[_ok, "Truck-mi per capita"],
        place_summary.loc[_ok, "Truck-mi per mi2"]
        / place_summary.loc[_ok, "People per mi2"], rtol=1e-9), \
        "per-capita does not equal per-area divided by population density"
    for _grp, _sub in place_summary.groupby("Grouping"):
        assert abs(_sub["Share of truck-mi (%)"].sum() - 100.0) < 1e-6, \
            f"{_grp} truck-mi shares do not sum to 100%"

    place_summary.round(
        {"Incidence (%)": 1, "Population in exposed places (%)": 1,
         "Truck-mi": 0, "Share of truck-mi (%)": 2,
         "Share of exposed population (%)": 2, "Trucking exposure ratio": 2,
         "Truck-mi per capita": 3, "Truck-mi per mi2": 1, "People per mi2": 1,
         "Mean share of area within 500 m (%)": 1}
    ).to_csv(out_path("Table_S11_Place_Exposure_Summary.csv"), index=False)
    print("\n  Table S11: exposure by community type and population size  ->  "
          "Table_S11_Place_Exposure_Summary.csv")
    print(place_summary[["Grouping", "Group", "Places in region", "Places exposed",
                         "Incidence (%)", "Share of exposed population (%)",
                         "Share of truck-mi (%)", "Trucking exposure ratio",
                         "Truck-mi per capita", "Truck-mi per mi2",
                         "People per mi2"]]
          .to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
    print("  Rates are pooled (sum of truck-mi over sum of population), not means "
          "of place-level rates.")

    _sz = place_summary[place_summary["Grouping"] == "Population size"]
    print(f"\n  Per-capita spans {_sz['Truck-mi per capita'].max() / _sz['Truck-mi per capita'].min():.1f}x "
          f"across size classes, per-area {_sz['Truck-mi per mi2'].max() / _sz['Truck-mi per mi2'].min():.1f}x, "
          f"population density {_sz['People per mi2'].max() / _sz['People per mi2'].min():.1f}x.")

    print("\n  Community type, exposed places only (Mann-Whitney U, two-sided; "
          "r_rb is the rank-biserial correlation):")
    _cdp = _hit[_hit["place_type"] == "CDP"]
    _cty = _hit[_hit["place_type"] != "CDP"]
    for _lab, _col in (("truck-mi per capita", "vmt_per_capita"),
                       ("truck-mi per mi2", "vmt_per_mi2"),
                       ("share of area within 500 m", "frac_exposed")):
        _a, _b = _cdp[_col].dropna(), _cty[_col].dropna()
        _u, _p = mannwhitneyu(_a, _b, alternative="two-sided")
        print(f"    {_lab:28s} CDP n={len(_a):3d} median {_a.median():9.3f} | "
              f"city n={len(_b):3d} median {_b.median():9.3f} | "
              f"p={_p:.2e}  r_rb={2 * _u / (len(_a) * len(_b)) - 1:+.2f}")

    print("\n  Same comparison inside matched population bands, which separates "
          "community type from size:")
    for _lab in PLACE_SIZE_LABELS:
        _s = _hit[_hit["size_class"] == _lab]
        _a = _s[_s["place_type"] == "CDP"]["vmt_per_capita"].dropna()
        _b = _s[_s["place_type"] != "CDP"]["vmt_per_capita"].dropna()
        _med = (f"CDP n={len(_a):3d} median {_a.median():6.2f} | "
                f"city n={len(_b):3d} median "
                + (f"{_b.median():6.2f}" if len(_b) else "     -"))
        if len(_a) >= 3 and len(_b) >= 3:
            print(f"    {_lab:10s} {_med} | p={mannwhitneyu(_a, _b, alternative='two-sided')[1]:.2f}")
        else:
            print(f"    {_lab:10s} {_med} | too few for a test")

    print("\n  Across the five size classes (Kruskal-Wallis, exposed places):")
    for _lab, _col in (("truck-mi per capita", "vmt_per_capita"),
                       ("truck-mi per mi2", "vmt_per_mi2")):
        _grps = [_hit[_hit["size_class"] == s][_col].dropna()
                 for s in PLACE_SIZE_LABELS]
        _rho, _prho = spearmanr(_hit["place_pop"], _hit[_col],
                                nan_policy="omit")
        print(f"    {_lab:20s} H-test p={kruskal(*_grps)[1]:.2e}   "
              f"Spearman against population rho={_rho:+.3f} (p={_prho:.2e})")

    _cum = (_hit.sort_values("VMT_j", ascending=False)["VMT_j"].cumsum()
            / _hit["VMT_j"].sum()).to_numpy()
    print(f"\n  Concentration: {int((_cum < 0.5).sum()) + 1} places carry half of "
          f"the in-place truck-mi; the top 10 carry {100 * _cum[9]:.1f}% and the "
          f"top {PLACE_TOP_N} carry {100 * _cum[PLACE_TOP_N - 1]:.1f}%.")

    print("  Agreement between the three rankings over "
          f"{len(_hit):,} exposed places (Spearman, and shared members of the "
          f"top {PLACE_TOP_N}):")
    for _x, _y, _xl, _yl in (("VMT_j", "vmt_per_capita", "total", "per capita"),
                             ("VMT_j", "vmt_per_mi2", "total", "per mi2"),
                             ("vmt_per_capita", "vmt_per_mi2", "per capita", "per mi2")):
        _shared = len(set(_hit.nlargest(PLACE_TOP_N, _x)["GEOID"])
                      & set(_hit.nlargest(PLACE_TOP_N, _y)["GEOID"]))
        print(f"    {_xl:10s} vs {_yl:10s} rho={spearmanr(_hit[_x], _hit[_y], nan_policy='omit')[0]:+.3f}   "
              f"{_shared} of {PLACE_TOP_N} shared")

    # Rank places by per-capita exposure and accumulate residents (no size bins).
    _rk = _hit.dropna(subset=["vmt_per_capita"]).sort_values(
        "vmt_per_capita", ascending=False)
    _cum_pop = (_rk["place_pop"].cumsum() / _rk["place_pop"].sum()).to_numpy()
    _cum_vmt = (_rk["VMT_j"].cumsum() / _rk["VMT_j"].sum()).to_numpy()
    print("\n  Ranking places by per-capita exposure, the most-exposed share of "
          "exposed-place residents\n  carries:")
    for _q in (0.05, 0.10, 0.25, 0.50):
        _i = min(int(np.searchsorted(_cum_pop, _q)), len(_cum_vmt) - 1)
        print(f"    most-exposed {_q:.0%} of residents -> "
              f"{100 * _cum_vmt[_i]:.1f}% of in-place truck-mi")

    print("\n  Share of a community's area lying within "
          f"{BUFFER_DISTANCE_M} m of a haul route:")
    for _thr in PLACE_COVERAGE_THRESHOLDS:
        _s = _hit[_hit["frac_exposed"] >= _thr]
        print(f"    >= {_thr:.0%}: {len(_s):3d} places, "
              f"{int(_s['place_pop'].sum()):,} residents "
              f"({int((_s['place_type'] == 'CDP').sum())} CDP / "
              f"{int((_s['place_type'] != 'CDP').sum())} city)")
    print(f"  {int(_hit['place_pop'].sum()):,} of "
          f"{int(place_exp['place_pop'].sum()):,} residents of retained places "
          f"({100 * _hit['place_pop'].sum() / place_exp['place_pop'].sum():.1f}%) "
          "live in a place carrying some manure trucking.")

    # -----------------------------------------------------------------
    # County of each exposed place, overall and for the top 20 on each rate.
    # Places are assigned by centroid, so each counts once.
    # -----------------------------------------------------------------
    if os.path.exists(COUNTIES_FILE):
        _cty = gpd.read_file(COUNTIES_FILE)
        _ccol = next(c for c in ("NAME", "COUNTY", "NAMELSAD", "County")
                     if c in _cty.columns)
        _cty = _cty[[_ccol, "geometry"]].rename(columns={_ccol: "county"})
        _cen = _hit.copy()
        _cen["geometry"] = _cen.geometry.centroid
        _cen = gpd.sjoin(_cen, _cty.to_crs(_cen.crs), how="left",
                         predicate="intersects")
        _cen = _cen[~_cen.index.duplicated(keep="first")]
        _cen["county"] = _cen["county"].fillna("(unmatched)")

        PLACE_TOP_N_COUNTY = 20
        _top_pc = set(_cen.nlargest(PLACE_TOP_N_COUNTY, "vmt_per_capita").index)
        _top_ar = set(_cen.nlargest(PLACE_TOP_N_COUNTY, "vmt_per_mi2").index)
        _tbl = pd.DataFrame({
            "Exposed places": _cen.groupby("county").size(),
            "Truck-mi": _cen.groupby("county")["VMT_j"].sum(),
            f"Top {PLACE_TOP_N_COUNTY} per capita":
                _cen.loc[list(_top_pc)].groupby("county").size(),
            f"Top {PLACE_TOP_N_COUNTY} per mi2":
                _cen.loc[list(_top_ar)].groupby("county").size(),
        }).fillna(0)
        for _c in _tbl.columns:
            if _c != "Truck-mi":
                _tbl[_c] = _tbl[_c].astype(int)
        _tbl["% of in-place truck-mi"] = (
            100 * _tbl["Truck-mi"] / _tbl["Truck-mi"].sum())
        _tbl = _tbl.sort_values("Exposed places", ascending=False)

        print(f"\n  Counties holding the {len(_cen):,} exposed places, and how many "
              f"of the top {PLACE_TOP_N_COUNTY} on each rate fall in each:")
        print(_tbl.to_string(float_format=lambda v: f"{v:,.1f}"))
        assert _tbl["Exposed places"].sum() == len(_cen), \
            "the county breakdown does not cover every exposed place"
        for _lbl, _sel in ((f"top {PLACE_TOP_N_COUNTY} per capita", _top_pc),
                           (f"top {PLACE_TOP_N_COUNTY} per mi2", _top_ar)):
            _n_cty = _cen.loc[list(_sel), "county"].nunique()
            _lead = _cen.loc[list(_sel), "county"].value_counts()
            print(f"    {_lbl}: {_n_cty} counties, led by "
                  + ", ".join(f"{k} ({v})" for k, v in _lead.head(3).items()))
    else:
        print(f"\n  {COUNTIES_FILE} not found; county breakdown omitted")

    HAVE_PLACE_EXPOSURE = True


In [ ]:
# =========================================================================
# FIGURE 5: CARB GHG INVENTORY vs REPORTED NITROGEN BALANCE
#   Top    CARB manure-management system fractions for dairy cows
#   Bottom nitrogen endpoints from the annual reports, on the same
#          colour-keyed columns
#
# Nitrogen basis: the denominator is dry manure nitrogen after ammonia loss,
# fixed on the annual report form at 0.70 of gross dry manure N, so ammonia is
# outside the accounting. Process wastewater N is not a second source; the
# wastewater N a dairy applies or exports is an outflow of that denominator.
#
# Reconciliation is per dairy. Manifests replace the export figure in a
# dairy's annual report; an export reported with no manifest is booked as
# unaccounted-for.
#
# Outputs:
#   Figure_5_CARB_Manure_Management.png
#   plus sample selection, manifest attribution, nitrogen closure and
#   whole-file reported totals, printed to the console
# =========================================================================
print("\n" + "=" * 96)
print("FIGURE 5: CARB inventory vs reported nitrogen balance")
print("=" * 96)

# ---- CARB GHG inventory manure-management fractions (dairy cows) ---------
# CARB Documentation of California's 2000-2023 GHG Inventory.
CARB_GHG_MANURE_FRACTIONS = {
    "Anaerobic Digestors": 0.164,
    "Anaerobic Lagoons":   0.43,
    "Daily Spread":        0.106,
    "Deep Pit":            0.001035,
    "Liquid/Slurry":       0.202,
    "Pasture":             0.006712,
    "Solid Storage":       0.091,
}

# ---- Nitrogen concentration of exported material ------------------------
# UC Davis Manure Technical Guide, "Dairy Manure Nutrient Content and Forms"
# (Pettygrove et al.). Solids: 29 samples from 11 Central Valley dairies,
# %N dry basis. Lagoon water: per-dairy median total N at 9 dairies, mg/L.
C_DRY_PCT_LOW, C_DRY_PCT_MID, C_DRY_PCT_HIGH = 1.2, 2.1, 3.5
C_WW_MGL_LOW, C_WW_MGL_MID, C_WW_MGL_HIGH = 164.0, 353.0, 645.0
lbN_per_dry_ton = lambda pct: (pct / 100.0) * 2000.0
lbN_per_gal = lambda mgL: mgL * 3.785e-6 * 2.2046

MIN_HERD = 1              # dairies must exceed this head count

# Denominator: 0.70 x gross dry manure N, the after-ammonia factor fixed on the
# report form. The form's own after-ammonia column is used only to check the
# factor, since some dairies leave it blank.
NH3_RETAINED = 0.70
AFTER_NH3 = "Total Dry Manure Generated N After Ammonia Losses (lbs)"

N5_MANURE_CSV = ROOT_PATH + "outputs/review/manifests/processed_manure_manifests.csv"
N5_WW_CSV = ROOT_PATH + "outputs/review/manifests/processed_wastewater_manifests.csv"

GEN_DRY = "Total Dry Manure Generated N (lbs)"
GEN_WW = "Total Process Wastewater Generated N (lbs)"
APP_DRY = "Applied N Dry Manure (lbs)"
APP_WW = "Applied Process Wastewater N (lbs)"
EXP_N = "Total Exports N (lbs)"

# Columns shared by both panels, in x order: (colour, CARB label, report
# category). None means the column has no counterpart in that panel.
SLOTS = [
    ("black",   "Anaerobic\nDigestors", None),
    ("green",   "Anaerobic\nLagoons",   None),
    ("wheat",   "Daily Spread",         "Land Application"),
    ("#2ca7a4", "Liquid/Slurry",        "Liquid"),
    ("brown",   "Other",                "Other Dry Storage"),
    ("#adadad", None,                   "Unaccounted-for"),
]
SLOT_LABELS = ["Anaerobic\nDigesters", "Anaerobic\nLagoons",
               "Daily Spread /\nLand Application", "Liquid/Slurry",
               "Other Dry\nStorage", "Unknown"]
x_positions = list(range(len(SLOTS)))
CARB_OTHER_COMPONENTS = [
    ("Solid Storage", CARB_GHG_MANURE_FRACTIONS["Solid Storage"] * 100),
    ("Pasture",       CARB_GHG_MANURE_FRACTIONS["Pasture"] * 100),
    ("Deep Pit",      CARB_GHG_MANURE_FRACTIONS["Deep Pit"] * 100),
]
carb_other_total = sum(v for _, v in CARB_OTHER_COMPONENTS)
carb_slot_pct = {
    "Anaerobic\nDigestors": CARB_GHG_MANURE_FRACTIONS["Anaerobic Digestors"] * 100,
    "Anaerobic\nLagoons":   CARB_GHG_MANURE_FRACTIONS["Anaerobic Lagoons"] * 100,
    "Daily Spread":         CARB_GHG_MANURE_FRACTIONS["Daily Spread"] * 100,
    "Liquid/Slurry":        CARB_GHG_MANURE_FRACTIONS["Liquid/Slurry"] * 100,
}
# Manifest destination type to display column.
N_COL = {"Farmer": None, "Broker": "Other Dry Storage",
         "Composting Facility": "Other Dry Storage",
         "Blank": "Unaccounted-for", "Other": "Unaccounted-for"}

_num = lambda s: pd.to_numeric(s, errors="coerce")

# =========================================================================
# SAMPLE SELECTION
# =========================================================================
rep_all = pd.read_csv(DAIRY_REPORT_FILE)
sample_log = [("Dairy annual reports in the summary file", "", len(rep_all), len(rep_all))]
rep_n = rep_all.copy()

_herd = _num(rep_n["Total Herd Size"])
_drop = ~(_herd > MIN_HERD)
sample_log.append(("Dropped: herd size missing or <= %d head" % MIN_HERD,
                   "cannot be a producing dairy", int(_drop.sum()),
                   int((~_drop).sum())))
rep_n = rep_n[~_drop].copy()

_gen = _num(rep_n[GEN_DRY]).fillna(0) + _num(rep_n[GEN_WW]).fillna(0)
_drop = ~(_gen > 0)
sample_log.append(("Dropped: no generated nitrogen reported",
                   "no denominator to apportion", int(_drop.sum()),
                   int((~_drop).sum())))
rep_n = rep_n[~_drop].copy()

# The CARB fractions describe dairy cows, so other livestock is out of scope.
_nondairy = ~dairy_cow_mask(rep_n)
nondairy_rows = [(r["Dairy Name"], r["Source PDF"], r["Total Herd Size"])
                 for _, r in rep_n[_nondairy].iterrows()]
sample_log.append(("Dropped: not a dairy-cow operation",
                   "CARB fractions are dairy-cow specific", int(_nondairy.sum()),
                   int((~_nondairy).sum())))
rep_n = rep_n[~_nondairy].copy().reset_index(drop=True)

# Livingston Dairy Consulting forms report nitrogen applications only per field
# and exports only as mass, so these reports are left out of the nitrogen
# accounting: the Figure 5 sample here, the Figure 3a waterfall and the
# nitrogen lines of the reported totals below. Their herds stay in the herd,
# manure and wastewater totals, and their manifests in the export totals.
LIVINGSTON_FORM = "Livingston"
_livingston = rep_n["Source Form Type"].eq(LIVINGSTON_FORM)
sample_log.append(("Dropped: Livingston Dairy Consulting form",
                   "applications reported only per field, exports only as mass",
                   int(_livingston.sum()), int((~_livingston).sum())))
rep_n = rep_n[~_livingston].copy().reset_index(drop=True)

rep_n["Source PDF_Cleaned"] = rep_n["Source PDF"].map(normalize_source_pdf)
rep_n["_gen_N"] = NH3_RETAINED * _num(rep_n[GEN_DRY]).fillna(0)
TOTAL_N = float(rep_n["_gen_N"].sum())
_gross_N = float(_num(rep_n[GEN_DRY]).fillna(0).sum())
# Where stated, the form's after-ammonia column must equal the factor above.
_af = _num(rep_n[AFTER_NH3])
_af_ok = _af.notna() & (_num(rep_n[GEN_DRY]).fillna(0) > 0)
assert np.allclose(_af[_af_ok], NH3_RETAINED * _num(rep_n[GEN_DRY])[_af_ok],
                   rtol=1e-4, atol=1.0), \
    "the form's after-ammonia column is not 0.70 x gross dry-manure nitrogen"
assert rep_n["Source PDF_Cleaned"].is_unique or True, "report keys need not be unique here"
N_DAIRIES = len(rep_n)
print(f"  dairies in the sample: {N_DAIRIES:,}   "
      f"N after ammonia loss: {TOTAL_N:,.0f} lb "
      f"({100 * TOTAL_N / _gross_N:.1f}% of the {_gross_N:,.0f} lb of dry manure N generated; "
      f"{_gross_N - TOTAL_N:,.0f} lb lost as ammonia)")
print(f"    after-ammonia column stated by {int(_af_ok.sum()):,} of "
      f"{N_DAIRIES:,} dairies and consistent with the {NH3_RETAINED:.2f} factor")

# =========================================================================
# MANIFESTS, ATTRIBUTED TO SAMPLE DAIRIES
# =========================================================================
def load_and_attribute(path, mass_col):
    """Attach each manifest to a sample dairy and report how the match was made."""
    df = pd.read_csv(path)
    df["Source PDF_Cleaned"] = df["Source PDF"].map(normalize_source_pdf)
    idx, how = resolve_report_rows(df, rep_n)
    df["_rep"] = idx
    df["_how"] = how
    df["_mass"] = _num(df[mass_col]).fillna(0).clip(lower=0)
    return df

man_n = load_and_attribute(N5_MANURE_CSV, "Total Manure Amount (tons)")
man_n["dest"] = standardize_dest_type(man_n["Destination Type (Standardized)"])

# Solids fraction: each manifest's own value where present, else the
# mass-weighted mean over manifests that report one.
_sp = _num(man_n["Manure Solids (%)"]) / 100.0
_ok = _sp.notna() & (man_n["_mass"] > 0)
SOLIDS_DEFAULT = float((man_n.loc[_ok, "_mass"] * _sp[_ok]).sum() / man_n.loc[_ok, "_mass"].sum())
man_n["_solids"] = _sp.fillna(SOLIDS_DEFAULT)
man_n["_dry_tons"] = man_n["_mass"] * man_n["_solids"]
print(f"  manure manifests: {len(man_n):,}  attributed {int(man_n['_rep'].notna().sum()):,}"
      f"  default solids fraction {SOLIDS_DEFAULT:.3f}")

if INCLUDE_WASTEWATER:
    ww_n = load_and_attribute(N5_WW_CSV, "Total Process Wastewater Exports (Gallons)")
    ww_n["dest"] = standardize_dest_type(ww_n["Destination Type (Standardized)"])
    print(f"  wastewater manifests: {len(ww_n):,}  attributed {int(ww_n['_rep'].notna().sum()):,}")
else:
    ww_n = man_n.iloc[:0].copy()

# Every attributed manifest's mass must survive into the per-dairy totals.
for _lbl, _df, _mc in (("manure", man_n, "Total Manure Amount (tons)"),
                       ("wastewater", ww_n, "Total Process Wastewater Exports (Gallons)")):
    if not len(_df):
        continue
    _att = _df[_df["_rep"].notna()]
    _direct = float(_num(_att[_mc]).fillna(0).clip(lower=0).sum())
    _viagrp = float(_att.groupby("_rep")["_mass"].sum().sum())
    assert abs(_direct - _viagrp) < 1.0, \
        f"{_lbl} mass not conserved through attribution: {_direct:,.0f} vs {_viagrp:,.0f}"
    print(f"  {_lbl} attributed mass conserved: {_viagrp:,.0f}")

# A dairy is manifested if either stream is attributed to it; its manifests
# then replace its reported export figure.
manifested = set(man_n.loc[man_n["_rep"].notna(), "_rep"].astype(int)) | \
             set(ww_n.loc[ww_n["_rep"].notna(), "_rep"].astype(int))
rep_n["_manifested"] = rep_n.index.isin(manifested)

# =========================================================================
# APPLIED-TO-REMOVED RATIO AND NITROGEN PRESSURE
#   The reporting form's applied-to-removed ratio is N applied to the land
#   application areas from all sources over N removed by crop harvest (MRP
#   R5-2013-0122 item 5). Nitrogen pressure is available N over the reported
#   facility-wide removal (Total Crop-Removed N).
#
#   Writes ratio_panel, one row per sample dairy, read by Figure 3b-d.
# =========================================================================
RATIO_N = "Applied to Remove Ratio N"
APP_TOT = "Total Applied N (lbs)"
UNACC_COL = "Unaccounted-for N (lbs)"
CROP_REMOVED = "Total Crop-Removed N"
RATIO_BOOT_DRAWS = 2000
RATIO_WINDOW = 90                    # dairies per sliding window in Figure 3b

assert "_dry_tons" in man_n.columns, "manure solids fraction not applied yet"
man_n["_N"] = man_n["_dry_tons"] * lbN_per_dry_ton(C_DRY_PCT_MID)
ww_n["_N"] = ww_n["_mass"] * lbN_per_gal(C_WW_MGL_MID)

ratio_panel = pd.DataFrame(index=rep_n.index)
ratio_panel["ratio"] = _num(rep_n[RATIO_N])
ratio_panel["avail"] = rep_n["_gen_N"]
ratio_panel["applied"] = _num(rep_n[APP_TOT])
ratio_panel["exported"] = _num(rep_n[EXP_N])
# The form's unaccounted-for column counts process wastewater N as a source;
# subtracting it gives 0.70 x dry manure N less applied and exported N.
ratio_panel["unacc"] = _num(rep_n[UNACC_COL]) - _num(rep_n[GEN_WW]).fillna(0)
ratio_panel["herd"] = _num(rep_n["Total Herd Size"])
ratio_panel["manifest_n"] = (
    man_n.groupby("_rep")["_N"].sum().reindex(rep_n.index).fillna(0.0)
    + ww_n.groupby("_rep")["_N"].sum().reindex(rep_n.index).fillna(0.0))

# Crop uptake is the reported removal; pressure is undefined where it is
# missing or zero.
ratio_panel["uptake"] = _num(rep_n[CROP_REMOVED]).where(lambda s: s > 0)
ratio_panel["pressure"] = ratio_panel["avail"] / ratio_panel["uptake"]

# Every fraction is a share of available N except the unmanifested one, which
# is a share of what the dairy itself reported exporting.
ratio_panel["f_unacc"] = ratio_panel["unacc"] / ratio_panel["avail"]
ratio_panel["f_exported"] = ratio_panel["exported"] / ratio_panel["avail"]
ratio_panel["f_unmanifested"] = np.where(
    ratio_panel["exported"] > 0,
    1.0 - ratio_panel["manifest_n"] / ratio_panel["exported"].replace(0, np.nan),
    np.nan)
ratio_panel = ratio_panel.replace([np.inf, -np.inf], np.nan)


def ratio_pfmt(p):
    """Format a p value for Figure 3."""
    return "p < 0.001" if p < 0.001 else f"p = {p:.2f}"


def boot_rho_ci(x, y, n_boot=RATIO_BOOT_DRAWS, seed=0):
    """Spearman rho with a 95% percentile bootstrap interval, resampling dairies.

    Returns (rho, lo, hi, n, p); NaNs when fewer than 20 complete pairs.
    """
    x, y = np.asarray(x, float), np.asarray(y, float)
    keep = np.isfinite(x) & np.isfinite(y)
    x, y = x[keep], y[keep]
    if len(x) < 20:
        return np.nan, np.nan, np.nan, len(x), np.nan
    rho, p = spearmanr(x, y)
    rs = np.random.default_rng(seed)
    draws = np.empty(n_boot)
    for i in range(n_boot):
        k = rs.integers(0, len(x), len(x))   # one index set, applied to both
        draws[i] = spearmanr(x[k], y[k])[0]
    lo, hi = np.nanpercentile(draws, [2.5, 97.5])
    return rho, lo, hi, len(x), p


def moving_median(x, y, k=RATIO_WINDOW):
    """Median and quartiles of y over a sliding window of k dairies ordered by x.

    Each window holds k consecutive dairies and is placed at the median x of
    those dairies, so the line starts and ends half a window inside the data.
    Returns rows (x, median, 25th, 75th percentile). A median is used because a
    few dairies report more nitrogen leaving than was generated, which would
    pull a mean.
    """
    x, y = np.asarray(x, float), np.asarray(y, float)
    o = np.argsort(x)
    x, y = x[o], y[o]
    if len(x) < k:
        return np.empty((4, 0))
    out = [(np.median(x[i:i + k]), np.median(y[i:i + k]),
            np.percentile(y[i:i + k], 25), np.percentile(y[i:i + k], 75))
           for i in range(len(x) - k + 1)]
    return np.array(out).T


_rp = ratio_panel
_has = _rp["ratio"].notna()
_pos = _rp["ratio"] > 0
print(f"\n  applied-to-removed ratio stated by {int(_has.sum()):,} of "
      f"{len(_rp):,} sample dairies, above zero for {int(_pos.sum()):,}")
print(f"    ratio median {_rp.loc[_pos, 'ratio'].median():.2f}, "
      f"above 1.4 for {int((_rp['ratio'] > 1.4).sum()):,} dairies")
_rem = _num(rep_n[CROP_REMOVED])
print(f"    crop N removal reported for {int((_rem > 0).sum()):,} of {len(_rp):,} sample "
      f"dairies ({int((_rem == 0).sum())} report zero, {int(_rem.isna().sum())} carry none)")
print(f"    nitrogen pressure median "
      f"{_rp.loc[_pos, 'pressure'].median():.2f} over "
      f"{int((_pos & _rp['pressure'].notna()).sum()):,} dairies "
      f"(available N over the crop N removal the report states)")
# Removal implied by applied manure and wastewater N over the ratio, as a share
# of the reported removal. The ratio's numerator also counts soil, fertilizer,
# water and atmospheric N, so this share is expected below one.
_recov = ((_rp["applied"] / _rp["ratio"].where(_rp["ratio"] > 0)) / _rp["uptake"]
          ).replace([np.inf, -np.inf], np.nan).dropna()
print(f"    removal backed out as applied manure and wastewater N / ratio would be "
      f"{_recov.median():.2f} of the reported figure at the median "
      f"(IQR {_recov.quantile(0.25):.2f}-{_recov.quantile(0.75):.2f}, n={len(_recov):,})")
_cov = man_n.loc[man_n["_rep"].map(_has).fillna(False), "_mass"].sum()
print(f"    those dairies carry {100 * _cov / man_n['_mass'].sum():.0f}% of "
      f"manifested manure tonnage")

# =========================================================================
# NITROGEN BY DISPLAY COLUMN
# =========================================================================
# segment tuple: (label, pct at the median N contents, pct at the low N
# contents, pct at the high N contents, hatch)
nbasis = {"Land Application": [], "Liquid": [],
          "Other Dry Storage": [], "Unaccounted-for": []}
pct = lambda lb: lb / TOTAL_N * 100.0

N_app_dry = float(_num(rep_n[APP_DRY]).fillna(0).clip(lower=0).sum())
N_app_ww = float(_num(rep_n[APP_WW]).fillna(0).clip(lower=0).sum())
nbasis["Land Application"].append(
    ("Applied (manure)", pct(N_app_dry), pct(N_app_dry), pct(N_app_dry), None))
nbasis["Liquid"].append(
    ("Applied (wastewater)", pct(N_app_ww), pct(N_app_ww), pct(N_app_ww), None))

def add_export_segments(df, kind, tons_col, conc):
    """Aggregate attributed manifests into display columns by destination."""
    lo_f, mid_f, hi_f = conc
    g = (df[df["_rep"].notna()]
         .groupby("dest")[tons_col].sum().reset_index())
    for _, row in g.iterrows():
        dest, q = row["dest"], float(row[tons_col])
        if q <= 0:
            continue
        col = N_COL.get(dest, "Unaccounted-for")
        if col is None:                      # Farmer: land application or liquid column
            col = "Land Application" if kind == "Solid" else "Liquid"
        nbasis[col].append((f"Export ({kind}): {dest}",
                            pct(q * mid_f), pct(q * lo_f), pct(q * hi_f),
                            "////" if dest == "Broker" else None))

add_export_segments(man_n, "Solid", "_dry_tons",
                    (lbN_per_dry_ton(C_DRY_PCT_LOW), lbN_per_dry_ton(C_DRY_PCT_MID),
                     lbN_per_dry_ton(C_DRY_PCT_HIGH)))
if INCLUDE_WASTEWATER:
    add_export_segments(ww_n, "Liquid", "_mass",
                        (lbN_per_gal(C_WW_MGL_LOW), lbN_per_gal(C_WW_MGL_MID),
                         lbN_per_gal(C_WW_MGL_HIGH)))

# Exports reported by dairies with no manifest: no destination is recorded.
_orphan_mask = (~rep_n["_manifested"]) & (_num(rep_n[EXP_N]).fillna(0) > 0)
N_exp_orphan = float(_num(rep_n.loc[_orphan_mask, EXP_N]).fillna(0).sum())
if N_exp_orphan > 0:
    nbasis["Unaccounted-for"].append(
        ("Export Reported,\nno manifest", pct(N_exp_orphan),
         pct(N_exp_orphan), pct(N_exp_orphan), None))

# Export N reported by manifested dairies, which their manifests replace, and
# the manifest export N in lb, for comparison.
N_exp_displaced = float(_num(rep_n.loc[rep_n["_manifested"], EXP_N]).fillna(0).sum())
N_exp_manifest = sum(
    seg[1] for col in nbasis for seg in nbasis[col] if seg[0].startswith("Export ("))
N_exp_manifest_lb = N_exp_manifest / 100.0 * TOTAL_N

# Tracked nitrogen and manifest export N at the median, low and high N
# contents (segment tuple positions 1, 2 and 3).
N_SCENARIOS = (1, 2, 3)
tracked_by = {k: sum(seg[k] for col in nbasis for seg in nbasis[col]) for k in N_SCENARIOS}
manifest_by = {k: sum(seg[k] for col in nbasis for seg in nbasis[col]
                      if seg[0].startswith("Export (")) for k in N_SCENARIOS}
tracked_pct = tracked_by[1]
remainder_raw = 100.0 - tracked_pct

# Reported export N of manifested dairies in excess of their manifest export
# N, summed over all manifested dairies and shown as its own segment rather
# than as on-farm loss, at each N content. Capped at the remainder so the
# columns close at 100% at each N content.
GAP_LABEL = "Export reported,\nexceeds manifest tonnage"
gap_by = {k: min(max(0.0, pct(N_exp_displaced) - manifest_by[k]),
                 max(0.0, 100.0 - tracked_by[k])) for k in N_SCENARIOS}
gap_shown = gap_by[1]
if any(gap_by[k] > 0 for k in N_SCENARIOS):
    nbasis["Unaccounted-for"].append(
        (GAP_LABEL, gap_by[1], gap_by[2], gap_by[3], None))

# Everything left over: volatilisation, storage, on-farm losses. It is the
# same at each N content unless the cap above applies.
REMAINDER_LABEL = "Unaccounted-for"
remainder_by = {k: max(0.0, 100.0 - tracked_by[k] - gap_by[k]) for k in N_SCENARIOS}
remainder_shown = remainder_by[1]
nbasis["Unaccounted-for"].append(
    (REMAINDER_LABEL, remainder_by[1], remainder_by[2], remainder_by[3], None))

def _ods_key(t):
    if "Composting Facility" in t[0]:
        return 0
    if "Broker" in t[0]:
        return 1
    return 2
nbasis["Other Dry Storage"].sort(key=_ods_key)
for k in ("Land Application", "Liquid", "Unaccounted-for"):
    nbasis[k].sort(key=lambda t: t[1], reverse=True)

_closure = sum(seg[1] for col in nbasis for seg in nbasis[col])
assert abs(_closure - 100.0) < 1e-6 or remainder_raw < 0, \
    f"nitrogen columns sum to {_closure:.4f}%, expected 100%"
for _k in N_SCENARIOS:
    _ck = sum(seg[_k] for col in nbasis for seg in nbasis[col])
    assert abs(_ck - 100.0) < 1e-6 or tracked_by[_k] > 100.0, \
        f"nitrogen columns sum to {_ck:.4f}% at N content {_k}, expected 100%"
# Unknown column total at the low, median and high N contents.
UNKNOWN_PCT_LMH = [sum(seg[k] for seg in nbasis["Unaccounted-for"]) for k in (2, 1, 3)]
# Export figures quoted in the Results: reported minus manifested export N,
# and the share of reported export N the manifests account for.
_exp_gap_lb = max(0.0, N_exp_displaced - N_exp_manifest_lb)
_exp_cov = (N_exp_manifest_lb / N_exp_displaced) if N_exp_displaced else float("nan")
print(f"  reported exports {N_exp_displaced:,.0f} lb N, manifested "
      f"{N_exp_manifest_lb:,.0f} lb N, gap {_exp_gap_lb:,.0f} lb "
      f"({_exp_gap_lb / 2000:,.0f} tons); manifests account for {_exp_cov:.1%} "
      f"of reported export nitrogen")
# The same share at the low, mid and high N contents (export segment bounds).
_exp_cov_lmh = [sum(seg[k] for col in nbasis for seg in nbasis[col]
                    if seg[0].startswith("Export (")) / 100.0 * TOTAL_N / N_exp_displaced
                for k in (2, 1, 3)]
print("    at the low, mid and high nitrogen contents: "
      + " / ".join(f"{_c:.1%}" for _c in _exp_cov_lmh))
# Reported export N in excess of the manifests at the low, median and high
# N contents (lb).
_exp_gap_lb_lmh = [max(0.0, N_exp_displaced - manifest_by[k] / 100.0 * TOTAL_N) for k in (2, 1, 3)]
print(f"  tracked {tracked_pct:.1f}%  reported-export gap {gap_shown:.1f}%"
      f"  unaccounted-for {remainder_shown:.1f}%  total {_closure:.1f}%")
print("    reported-export gap at the low, median and high nitrogen contents: "
      + " / ".join(f"{gap_by[k]:.1f}%" for k in (2, 1, 3))
      + "  (" + " / ".join(f"{_g / 2000:,.0f}" for _g in _exp_gap_lb_lmh) + " tons)")
print("    Unknown column at the low, median and high nitrogen contents: "
      + " / ".join(f"{_u:.1f}%" for _u in UNKNOWN_PCT_LMH))
if remainder_raw < 0:
    print(f"  NOTE: reported streams exceed generated nitrogen by "
          f"{-remainder_raw:.1f}%; the remainder is clipped at zero")

# =========================================================================
# FIGURE 5
# =========================================================================
BAR_W = 0.62
# Per-segment colour overrides (none): segments take their column colour and
# are separated by black edges.
SEGMENT_SHADE = {}
INLINE_MIN_PCT = 4.5      # thinner segments get a leader label instead
LABEL_MIN_PCT = 0.05      # below this a segment is drawn but not labelled
leader_labels = []
column_totals = []
plt.rcParams["hatch.linewidth"] = 3
fig6, (ax6a, ax6b) = plt.subplots(
    2, 1, figsize=(13.5, 14), sharex=True,
    gridspec_kw={"height_ratios": [1.0, 1.25], "hspace": 0.20})

for xi, (color, carb_label, _rep_cat) in zip(x_positions, SLOTS):
    if carb_label is None:
        continue
    if carb_label == "Other":
        bottom = 0.0
        for name, val in CARB_OTHER_COMPONENTS:
            ax6a.bar(xi, val, bottom=bottom, width=BAR_W, color=color, edgecolor="black")
            if val > 0.8:
                ax6a.text(xi, bottom + val / 2, f"{name}\n{val:.1f}%",
                          ha="center", va="center", fontsize=8.5)
            bottom += val
        ax6a.text(xi, bottom + 0.8, f"{carb_other_total:.1f}%",
                  ha="center", va="bottom", fontsize=12)
    else:
        val = carb_slot_pct[carb_label]
        ax6a.bar(xi, val, width=BAR_W, color=color, edgecolor="black")
        ax6a.text(xi, val + 0.8, f"{val:.1f}%", ha="center", va="bottom", fontsize=12)
ax6a.set_ylabel("Manure Management Fractions (%)", fontsize=13)
ax6a.set_title("CARB Dairy Greenhouse Gas Management Fractions", fontsize=15, pad=10)
add_panel_label(ax6a, "a)", fontsize=24, x=-0.09, y=1.07)
ax6a.set_ylim(0, max(list(carb_slot_pct.values()) + [carb_other_total]) * 1.20)
ax6a.grid(axis="y", alpha=0.3)

empty_slots = []
for xi, (color, _carb, rep_cat) in zip(x_positions, SLOTS):
    if rep_cat is None:
        empty_slots.append(xi)
        continue
    segs = nbasis.get(rep_cat, [])
    if not segs:
        empty_slots.append(xi)
        continue
    bottom = 0.0
    # Error bar from the column totals at the low and high N contents. A higher
    # content moves nitrogen out of the Unknown column, so there the
    # low-content total is the larger.
    _col_low_n = sum(s[2] for s in segs)
    _col_high_n = sum(s[3] for s in segs)
    col_lo, col_hi = min(_col_low_n, _col_high_n), max(_col_low_n, _col_high_n)
    col_mid = sum(s[1] for s in segs)
    small = []
    for label, seg, s_lo, s_hi, hatch in segs:
        face = SEGMENT_SHADE.get(label, color)
        if hatch:
            ax6b.bar(xi, seg, bottom=bottom, width=BAR_W, color=face,
                     edgecolor="white", linewidth=0, hatch=hatch, zorder=2)
            ax6b.bar(xi, seg, bottom=bottom, width=BAR_W, color="none",
                     edgecolor="black", linewidth=0.8, zorder=3)
        else:
            ax6b.bar(xi, seg, bottom=bottom, width=BAR_W, color=face,
                     edgecolor="black", linewidth=0.8, zorder=2)
        if seg >= INLINE_MIN_PCT:
            ax6b.text(xi, bottom + seg / 2, f"{label}\n{seg:.1f}%",
                      ha="center", va="center", fontsize=8.5, zorder=6)
        elif seg >= LABEL_MIN_PCT:
            small.append((label, bottom + seg / 2, seg))
        bottom += seg
    leader_labels.append((xi, small, max(col_hi, col_mid)))
    if col_hi > col_lo:
        ax6b.errorbar(xi, col_mid, yerr=[[col_mid - col_lo], [col_hi - col_mid]],
                      fmt="none", ecolor="black", elinewidth=2, capsize=8, capthick=2,
                      zorder=7)
    column_totals.append((xi, col_mid, max(col_hi, col_mid), len(small)))
# Segments too thin for inline text are labelled in a stack above their column.
LABEL_GAP = 3.4
LABEL_BASE = 5.2
LEADER_WRAP_CHARS = 34

def leader_text(label, seg):
    """Leader-label text, wrapped after the stream prefix when longer than
    LEADER_WRAP_CHARS."""
    txt = f"{label.replace(chr(10), ' ')}  {seg:.1f}%"
    if len(txt) > LEADER_WRAP_CHARS and ": " in txt:
        head, tail = txt.split(": ", 1)
        txt = f"{head}:\n{tail}"
    return txt

_ymax = max(max(sum(seg[2] for seg in nbasis[k]), sum(seg[3] for seg in nbasis[k]))
            for k in nbasis if nbasis[k]) * 1.08
_stack_top = 0.0
stack_end = {}
for xi, small, col_top in leader_labels:
    off = LABEL_BASE
    for label, ymid, seg in small:
        txt = leader_text(label, seg)
        ytxt = col_top + off
        _stack_top = max(_stack_top, ytxt)
        ax6b.annotate(txt,
                      xy=(xi, ymid), xytext=(xi, ytxt),
                      ha="center", va="center", fontsize=8.2, zorder=8,
                      bbox=dict(boxstyle="round,pad=0.22", fc="white",
                                ec="0.75", lw=0.5),
                      arrowprops=dict(arrowstyle="-", lw=0.6, color="0.45",
                                      shrinkA=1, shrinkB=1))
        off += LABEL_GAP * (1 + 0.6 * txt.count("\n"))
    stack_end[xi] = off

# The column total sits above whatever label stack the column carries.
for xi, col_mid, col_top, n_small in column_totals:
    ytxt = col_top + (stack_end.get(xi, LABEL_BASE) if n_small else 1.6)
    _stack_top = max(_stack_top, ytxt + LABEL_GAP * 0.6)
    ax6b.text(xi, ytxt, f"{col_mid:.1f}%", ha="center", va="bottom",
              fontsize=12.5, fontweight="semibold", zorder=9)

ax6b.set_ylabel("Nitrogen Endpoints Fractions (%)", fontsize=13)
ax6b.set_title("Dairy Annual Report Nitrogen Endpoint Fractions", fontsize=15, pad=10)
add_panel_label(ax6b, "b)", fontsize=24, x=-0.09, y=1.07)
ax6b.set_ylim(0, max(_ymax, _stack_top + LABEL_GAP))
ax6b.set_xlim(-0.62, len(SLOTS) - 1 + 0.62)
ax6b.set_xticks(x_positions)
ax6b.set_xticklabels(SLOT_LABELS, rotation=18, ha="right", fontsize=11.5)
ax6b.grid(axis="y", alpha=0.3)
# Columns with no report counterpart (digesters, lagoons) get a question mark:
# unreported, not zero.
for xi in empty_slots:
    ax6b.text(xi, ax6b.get_ylim()[1] * 0.015, "?", ha="center", va="bottom",
              fontsize=26, fontweight="bold", color="0.35", zorder=9)

save_figure(fig6, "Figure_5_CARB_Manure_Management.png", dpi=200, bbox_inches="tight")
plt.show()

# =========================================================================
# SAMPLE SELECTION, MANIFEST ATTRIBUTION AND NITROGEN CLOSURE
#   Printed to the console; these values are quoted in the text.
# =========================================================================
rows = [dict(section="Sample selection", item=a, reason=b, n_dropped=c, n_remaining=d)
        for a, b, c, d in sample_log]

_exp = _num(rep_n[EXP_N])
comp = [
    ("Dairies with at least one manifest", int(rep_n["_manifested"].sum())),
    ("Dairies reporting exports with no manifest", int(_orphan_mask.sum())),
    ("Dairies reporting zero exports and no manifest",
     int(((~rep_n["_manifested"]) & (_exp == 0)).sum())),
    ("Dairies with no export figure and no manifest",
     int(((~rep_n["_manifested"]) & _exp.isna()).sum())),
]
for item, n in comp:
    rows.append(dict(section="Composition of the retained sample", item=item,
                     reason="", n_dropped="", n_remaining=n))

_mt = _num(man_n["Total Manure Amount (tons)"]).fillna(0)
for kind, df, mass in (("manure", man_n, _mt),
                       ("wastewater", ww_n, _num(ww_n["_mass"]).fillna(0))):
    if not len(df):
        continue
    for how in ("Source PDF", "dairy name", "street address", "unmatched"):
        m = df["_how"] == how
        rows.append(dict(section=f"Manifest attribution ({kind})",
                         item=f"matched by {how}" if how != "unmatched" else "unmatched, excluded",
                         reason="", n_dropped="",
                         n_remaining=f"{int(m.sum())} manifests, "
                                     f"{mass[m].sum():,.0f} {'tons' if kind=='manure' else 'gal'}"))

for nm, pdf, hd in nondairy_rows:
    rows.append(dict(section="Dropped: not a dairy-cow operation", item=str(nm),
                     reason=f"filed as {'goat' if 'goat' in str(pdf).lower() else ('poultry' if 'poultry' in str(pdf).lower() else 'bovine')}"
                            f"; {hd:,.0f} head",
                     n_dropped=1, n_remaining=""))

_missing_app = _num(rep_n[APP_DRY]).isna() | _num(rep_n[APP_WW]).isna()
_missing_gen_N = float(rep_n.loc[_missing_app, "_gen_N"].sum())
_over = (_num(rep_n[APP_DRY]).fillna(0) + _num(rep_n[APP_WW]).fillna(0)
         + _num(rep_n[EXP_N]).fillna(0)) > rep_n["_gen_N"] * 1.001
diag = [
    ("Kept: missing an applied-nitrogen field", int(_missing_app.sum()),
     f"{_missing_gen_N/TOTAL_N*100:.1f}% of the denominator; contributes to "
     f"Unaccounted-for without reporting any application"),
    ("Kept: applied + exported exceeds generated nitrogen", int(_over.sum()),
     "internally inconsistent reports, retained as filed"),
    ("Kept: manure reported on a non as-excreted basis",
     int((rep_n["Manure Basis"].notna() & (rep_n["Manure Basis"] != "as-excreted")).sum())
     if "Manure Basis" in rep_n.columns else 0,
     "nitrogen basis is unaffected: numerator and denominator come from the same report"),
]
for item, n, why in diag:
    rows.append(dict(section="Diagnostics (kept in the sample)", item=item,
                     reason=why, n_dropped="", n_remaining=n))

for item, val in (
        ("Dry manure nitrogen generated, gross (lb)", f"{_gross_N:,.0f}"),
        ("Nitrogen after ammonia loss (denominator, lb)", f"{TOTAL_N:,.0f}"),
        ("Tracked nitrogen (% of available after ammonia loss)",
         f"{tracked_pct:.1f}%"),
        ("Remainder after tracked streams (%)", f"{remainder_raw:+.1f}%"),
        ("...shown as 'Export reported, exceeds manifest tonnage' (%)", f"{gap_shown:.1f}%"),
        ("...shown as 'Unaccounted-for' (%)", f"{remainder_shown:.1f}%"),
        ("Export N reported by manifested dairies, displaced by the manifests (lb)",
         f"{N_exp_displaced:,.0f}  ({N_exp_displaced/TOTAL_N*100:.1f}% of available)"),
        ("Export N the manifests account for instead (lb)",
         f"{N_exp_manifest_lb:,.0f}  ({N_exp_manifest:.1f}% of available; "
         f"{100 * N_exp_manifest_lb / _gross_N:.1f}% of gross generated)"),
        ("Difference, shown as its own Unaccounted-for segment (lb)",
         f"{N_exp_displaced - N_exp_manifest_lb:,.0f}  "
         f"({(N_exp_displaced - N_exp_manifest_lb)/TOTAL_N*100:.1f}% of available)")):
    rows.append(dict(section="Nitrogen closure", item=item, reason="",
                     n_dropped="", n_remaining=val))

sample_summary = pd.DataFrame(rows)
print("\n" + sample_summary.to_string(index=False, max_colwidth=58))
print(f"\n  wrote {out_path('Figure_5_CARB_Manure_Management.png')}")

# =========================================================================
# REPORTED TOTALS ACROSS THE ANNUAL REPORT SUMMARY
#   Whole-file totals from the annual reports, independent of the manifests
#   and of the Figure 5 sample. The herd, manure and wastewater lines count
#   every dairy-cow report with at least TOTALS_MIN_HERD cattle and a dry
#   manure nitrogen figure, and sum wastewater where it is stated. The nitrogen
#   lines count those reports that also state wastewater nitrogen, without the
#   Livingston reports (see SAMPLE SELECTION). Nitrogen generated is the dry
#   manure figure; unaccounted-for is the form's column less process wastewater
#   N (the basis of Figures 3, 5).
# =========================================================================
TOTALS_MIN_HERD = 2

_t = rep_all.copy()
_t_herd = _num(_t["Total Herd Size"])
_t_dry = _num(_t[GEN_DRY])
_t_ww = _num(_t[GEN_WW])

_small = ~(_t_herd >= TOTALS_MIN_HERD)
_not_dairy = ~dairy_cow_mask(_t)

# Reports in the herd, manure and wastewater lines.
_keep_all = ~_small & _t_dry.notna() & ~_not_dairy
_kept_all = _t[_keep_all]
# Reports in the nitrogen lines also state wastewater nitrogen.
_keep = _keep_all & _t_ww.notna()
_kept = _t[_keep].assign(
    _unacc_dry=_num(_t.loc[_keep, "Unaccounted-for N (lbs)"])
    - _num(_t.loc[_keep, GEN_WW]).fillna(0))
# The same reports without the Livingston forms, for the nitrogen lines.
_kept_n = _kept[~_kept["Source Form Type"].eq(LIVINGSTON_FORM)]
TOTALS_ALL_REPORT_COLS = {"Total Herd Size", "Total Manure Excreted (tons)",
                          "Total Process Wastewater Generated (gals)"}
UNACC_TOTAL_LB = float(_kept_n["_unacc_dry"].sum())

TOTALS_COLS = [
    ("Total herd size (head)",            "Total Herd Size",                        "{:>16,.0f}"),
    ("Manure generated (tons)",           "Total Manure Excreted (tons)",           "{:>16,.0f}"),
    ("Wastewater generated (gal)",        "Total Process Wastewater Generated (gals)", "{:>16,.0f}"),
    ("Nitrogen generated, dry (lb)",      None,                                     "{:>16,.0f}"),
    ("Nitrogen applied (lb)",             "Total Applied N (lbs)",                  "{:>16,.0f}"),
    ("Nitrogen exported (lb)",            "Total Exports N (lbs)",                  "{:>16,.0f}"),
    ("Nitrogen unaccounted-for (lb)",     "_unacc_dry",                             "{:>16,.0f}"),
]

print("\n" + "=" * 96)
print("REPORTED TOTALS ACROSS THE ANNUAL REPORT SUMMARY")
print("=" * 96)
print(f"  dairies in {os.path.basename(DAIRY_REPORT_FILE)}: {len(_t):,}")
print(f"    dropped: fewer than {TOTALS_MIN_HERD} cattle reported        "
      f"{int(_small.sum()):>5}")
print(f"    dropped: dry manure N blank                      "
      f"{int((_t_dry.isna() & ~_small).sum()):>5}")
print(f"    dropped: not a dairy-cow herd on review          "
      f"{int((_not_dairy & ~_small & _t_dry.notna()).sum()):>5}   "
      + ", ".join(str(n) for n in _t.loc[_not_dairy & ~_small & _t_dry.notna(), "Dairy Name"]))
print(f"    in the herd, manure and wastewater lines         {len(_kept_all):>5}"
      f"   ({100.0 * len(_kept_all) / len(_t):.1f}% of the file; wastewater stated on "
      f"{int(_num(_kept_all['Total Process Wastewater Generated (gals)']).notna().sum())})")
print(f"    of which state wastewater N                      {len(_kept):>5}"
      f"   ({len(_kept_all) - len(_kept)} without it: "
      f"{int(_kept_all.loc[_t_ww[_keep_all].isna(), 'Source Form Type'].eq(LIVINGSTON_FORM).sum())} Livingston)")
print(f"    of which in the nitrogen lines                   {len(_kept_n):>5}"
      f"   ({len(_kept) - len(_kept_n)} Livingston reports left out)")

_gen_kept = _num(_kept_n[GEN_DRY]).fillna(0)
print()
for _label, _col, _fmt in TOTALS_COLS:
    _frame = _kept_all if _col in TOTALS_ALL_REPORT_COLS else _kept_n
    _v = _gen_kept.sum() if _col is None else float(_num(_frame[_col]).sum())
    print(f"    {_label:<32}" + _fmt.format(_v))
for _sub_label, _sub_col in (("  generated, dry manure (lb)", GEN_DRY),
                             ("  generated, wastewater (lb)", GEN_WW),
                             ("  applied, dry manure (lb)", APP_DRY),
                             ("  applied, wastewater (lb)", APP_WW)):
    print(f"    {_sub_label:<32}" + "{:>16,.0f}".format(float(_num(_kept_n[_sub_col]).sum())))

# Total Applied N is the sum of the two applied streams on the form; dairies
# whose stated total differs from their components by over 1 lb are listed.
_app_comp = _num(_kept_n[APP_DRY]).fillna(0) + _num(_kept_n[APP_WW]).fillna(0)
_app_tot = _num(_kept_n["Total Applied N (lbs)"])
_app_bad = _app_tot.notna() & ((_app_tot - _app_comp).abs() > 1.0)
print(f"    {'  applied, components summed (lb)':<32}"
      + "{:>16,.0f}".format(float(_app_comp.sum())))
if _app_bad.any():
    print(f"    {int(_app_bad.sum())} dairy(s) state a Total Applied N that differs from their own "
          f"components by {float((_app_tot - _app_comp)[_app_bad].sum()):,.0f} lb in total:")
    for _, _r in _kept_n[_app_bad].iterrows():
        _c = float(_num(pd.Series([_r[APP_DRY]])).fillna(0).iloc[0]) + \
             float(_num(pd.Series([_r[APP_WW]])).fillna(0).iloc[0])
        print(f"      {str(_r['Dairy Name'])[:34]:<34} stated {float(_r['Total Applied N (lbs)']):>14,.0f}"
              f"   components {_c:>12,.2f}   ratio {float(_r['Total Applied N (lbs)']) / _c:>8,.1f}x")

# Residual on the after-ammonia basis (0.70 x dry manure N), the basis on which
# the form states applied and exported N.
_after_nh3 = NH3_RETAINED * _num(_kept_n[GEN_DRY]).fillna(0)
_resid = (_after_nh3.sum() - float(_num(_kept_n["Total Applied N (lbs)"]).sum())
          - float(_num(_kept_n["Total Exports N (lbs)"]).sum()))
_reported_unacc = UNACC_TOTAL_LB
# Dairies with a blank applied or exported figure also have a blank
# unaccounted-for column, so they enter the computed residual but not the
# summed column.
_blank_u = _kept_n["_unacc_dry"].isna()
_blank_lb = float((_after_nh3[_blank_u]
                   - _num(_kept_n.loc[_blank_u, "Total Applied N (lbs)"]).fillna(0)
                   - _num(_kept_n.loc[_blank_u, "Total Exports N (lbs)"]).fillna(0)).sum())
assert abs((_resid - _reported_unacc) - _blank_lb) < len(_kept_n), \
    "the residual and the summed column differ by more than the blank rows explain"
print()
for _label, _v in (("available N after ammonia loss (lb)", float(_after_nh3.sum())),
                   ("less applied and exported (lb)", _resid),
                   ("unaccounted-for as reported (lb)", _reported_unacc),
                   ("difference (lb)", _resid - _reported_unacc)):
    print(f"    {_label:<40}" + "{:>16,.0f}".format(_v))
print(f"    the two disagree by "
      f"{abs(_resid - _reported_unacc) / _after_nh3.sum() * 100:.1f}% of available nitrogen: "
      f"{int(_blank_u.sum())} dairies leave an applied or exported\n    figure blank, so their "
      f"unaccounted-for column is blank too ({_blank_lb:,.0f} lb of the difference)")

_looser = int(((_t_herd >= TOTALS_MIN_HERD) & (_t_dry.fillna(0) + _t_ww.fillna(0) > 0)
               & ~_not_dairy).sum())
print(f"\n    requiring only a positive nitrogen total rather than both figures stated "
      f"would keep {_looser:,}")

# =========================================================================
# MANIFEST-REPORTED EXPORTS
#   Tonnage and volume stated on every manifest in the two manifest files,
#   whether or not the origin dairy is in the report sample above. The
#   attributed line is the subset resolved to a dairy in the nitrogen sample,
#   which Figure 5b converts to nitrogen.
# =========================================================================
print("\n" + "=" * 96)
print("MANIFEST-REPORTED EXPORTS")
print("=" * 96)
print(f"  Sample: all manifests in processed_manure_manifests.csv and")
print(f"  processed_wastewater_manifests.csv, independent of the "
      f"{len(_kept_all):,}-dairy report sample above.")
print()
for _lbl, _df in (("Manure exported (tons)", man_n),
                  ("Wastewater exported (gal)", ww_n)):
    if not len(_df):
        continue
    _all = float(_df["_mass"].sum())
    _att = float(_df.loc[_df["_rep"].notna(), "_mass"].sum())
    _n_att = int(_df["_rep"].notna().sum())
    print(f"    {_lbl:<38}" + "{:>15,.0f}".format(_all)
          + f"   {len(_df):>5,} manifests")
    print(f"      {'attributed to the %d-dairy sample' % N_DAIRIES:<36}"
          + "{:>15,.0f}".format(_att)
          + f"   {_n_att:>5,} manifests   ({100 * _att / _all:.1f}%)")

# =========================================================================
# NITROGEN WATERFALL (Figure 3a)
#   Sums the reports' nitrogen columns over dairies with all seven nitrogen
#   fields stated (including the after-ammonia column, asserted against the
#   gross figure), at least TOTALS_MIN_HERD cattle and a dairy-cow herd, and
#   not on the Livingston form; a narrower sample than Figure 5's. Dry manure is the only source: the
#   unaccounted-for bar is the form's column less process wastewater N, and
#   applied wastewater is an outflow like applied manure.
#
#   The ammonia bar removes 30% of gross dry manure N (the form's 0.70
#   after-ammonia factor) before application and export, which closes the
#   sequence to zero.
# =========================================================================
UNACC_N = "Unaccounted-for N (lbs)"
NH3_LOSS_FRACTION = 1.0 - NH3_RETAINED

_wf_cols = [GEN_DRY, GEN_WW, APP_DRY, APP_WW, EXP_N, UNACC_N, AFTER_NH3]
_wf_ok = (_t_herd >= TOTALS_MIN_HERD) & ~_not_dairy & \
    ~_t["Source Form Type"].eq(LIVINGSTON_FORM) & \
    _t[_wf_cols].apply(_num).notna().all(axis=1)
wf = _t[_wf_ok]
_v = {c: _num(wf[c]) for c in _wf_cols}

# The form's after-ammonia column must be 0.70 of gross for the ammonia bar to
# be sized correctly; the tolerance covers the reports' two-decimal rounding.
assert np.allclose(_v[AFTER_NH3], (1 - NH3_LOSS_FRACTION) * _v[GEN_DRY],
                   rtol=1e-4, atol=1.0), \
    "the after-ammonia column is not 0.70 x gross dry-manure nitrogen"
_v[UNACC_N] = _v[UNACC_N] - _v[GEN_WW]

WATERFALL = [
    ("Manure",                  +float(_v[GEN_DRY].sum())),
    ("Ammonia\nloss",           -NH3_LOSS_FRACTION * float(_v[GEN_DRY].sum())),
    ("Applied\ndry manure",     -float(_v[APP_DRY].sum())),
    ("Applied\nwastewater",     -float(_v[APP_WW].sum())),
    ("Exported",                 -float(_v[EXP_N].sum())),
    ("Unaccounted\n-for",       -float(_v[UNACC_N].sum())),
]
WATERFALL_N_DAIRIES = len(wf)
# Reported process wastewater N, marked as a line across Figure 3a's first bar.
WATERFALL_WW = float(_v[GEN_WW].sum())
# Dairies reporting more N applied and exported than available; their negative
# unaccounted-for figures offset the positive ones in the net.
WATERFALL_NEG_N = int((_v[UNACC_N] < 0).sum())
WATERFALL_NEG_LB = float(_v[UNACC_N][_v[UNACC_N] < 0].sum())

_close = sum(v for _, v in WATERFALL)
assert abs(_close) < 1.0, f"waterfall does not close: {_close:,.2f} lb"

print("\n" + "=" * 96)
print("FIGURE 3a: NITROGEN WATERFALL")
print("=" * 96)
print(f"  dairies with all seven nitrogen fields stated: {WATERFALL_N_DAIRIES:,}"
      f"   (Figure 5 sample is {N_DAIRIES:,})")
_run = 0.0
for _lbl, _val in WATERFALL:
    _run += _val
    print(f"    {_lbl.replace(chr(10), ' '):<34}{_val:>16,.0f}   running {_run:>16,.0f}")
print(f"    {'closes to':<34}{_close:>16,.0f}")
print(f"    {'process wastewater, in first bar':<34}{WATERFALL_WW:>16,.0f}")
print(f"  {WATERFALL_NEG_N} dairies report applying and exporting more nitrogen than they "
      f"had available,\n  totalling {WATERFALL_NEG_LB:,.0f} lb, which offsets the positive "
      f"unaccounted figures")
# Median per-dairy unaccounted-for share of available N vs the aggregate share.
_f_wf = (_v[UNACC_N] / (NH3_RETAINED * _v[GEN_DRY])).replace([np.inf, -np.inf], np.nan).dropna()
print(f"  median dairy leaves {_f_wf.median():.3f} of its available N unaccounted-for "
      f"(n={len(_f_wf):,}), against {_v[UNACC_N].sum() / (NH3_RETAINED * _v[GEN_DRY].sum()):.3f} "
      f"in aggregate")


In [ ]:
# =========================================================================
# FIGURE S7: DO DROPPED MANIFESTS DIFFER FROM ROUTED ONES?
#   Row 1 (a-c)  manure tonnage per manifest, routed vs each drop group
#   Row 2 (d-f)  animal units of the origin dairy, same comparison
#
#   Each panel overlays the empirical CDFs of the drop group and the routed
#   set (no binning or smoothing), with medians as dotted lines and a two-sided
#   Mann-Whitney U test with rank-biserial r. Log x-axis; both fields span
#   three orders of magnitude.
#
#   Animal units repeat across a dairy's manifests, so bottom-row p-values
#   count one facility many times and are optimistic.
#
# Outputs: Figure_S7_Routing_Failure_Modes.png
# =========================================================================
from scipy.stats import mannwhitneyu

print("\n" + "=" * 96)
print("FIGURE S7: routed against each failure mode, tonnage and herd size")
print("=" * 96)

S3_MODES = [
    ("Dropped: Failed geocode",          "Failed geocode",           "#1f78b4"),
    ("Dropped: Bad Destination Geocode",  "Bad destination geocode",  "#d6604d"),
    ("Dropped: ALL",                      "All dropped",              "#762a83"),
]
S3_VARS = [
    ("Total Manure Amount (tons)", "Manure per manifest (tons)"),
    ("Animal Units",               "Animal units of origin dairy"),
]
S3_ROW_TITLES = ["Manure Amount per Manifest",
                 "Herd Size of Origin Dairy (Animal Units)"]

S3_REF_COLOR = "0.25"
S3_LETTERS = "abcdefgh"


def ecdf(x):
    """Sorted values and their cumulative proportion, for a step plot."""
    xs = np.sort(np.asarray(x, dtype=float))
    return xs, np.arange(1, len(xs) + 1) / len(xs)


def s3_compare(vals, mask_a, mask_b):
    """Mann-Whitney U of group a against group b, with medians and effect size."""
    a = vals[mask_a].dropna()
    b = vals[mask_b].dropna()
    if len(a) < 2 or len(b) < 2:
        return None
    u, p = mannwhitneyu(a, b, alternative="two-sided")
    return dict(a=a, b=b, n_a=len(a), n_b=len(b),
                med_a=float(a.median()), med_b=float(b.median()),
                rb=2.0 * u / (len(a) * len(b)) - 1.0, p=float(p))


figS7 = plt.figure(figsize=(15.8, 10.6))
gsS3 = GridSpec(len(S3_VARS), len(S3_MODES), figure=figS7,
                hspace=0.46, wspace=0.16, top=0.905, bottom=0.075,
                left=0.052, right=0.988)

s3_rows = []
_row_lead_ax = {}
for _vi, (_col, _vlabel) in enumerate(S3_VARS):
    _vals = pd.to_numeric(manure_df[_col], errors="coerce")
    _ax0 = None
    for _mi, (_key, _mlabel, _colr) in enumerate(S3_MODES):
        ax = figS7.add_subplot(gsS3[_vi, _mi], sharey=_ax0)
        if _ax0 is None:
            _ax0 = ax
            _row_lead_ax[_vi] = ax
        st = s3_compare(_vals, balance_groups[_key], balance_groups["Routed"])
        if st is None:
            ax.text(0.5, 0.5, "too few observations", ha="center", va="center",
                    transform=ax.transAxes, fontsize=11)
            ax.set_xticks([]); ax.set_yticks([])
            continue

        # Log axis: plot positive values only; non-positive tonnage or animal
        # units are treated as missing.
        _pa = st["a"][st["a"] > 0]
        _pb = st["b"][st["b"] > 0]
        _excl = (len(st["a"]) - len(_pa)) + (len(st["b"]) - len(_pb))
        for _v, _c, _lab, _lw in ((_pb, S3_REF_COLOR, f"Routed (n={st['n_b']:,})", 2.4),
                                  (_pa, _colr, f"{_mlabel} (n={st['n_a']:,})", 2.4)):
            _x, _y = ecdf(_v)
            ax.step(_x, _y, where="post", color=_c, lw=_lw, label=_lab)
            ax.axvline(float(np.median(_v)), color=_c, ls=":", lw=1.5, alpha=0.85)

        ax.set_xscale("log")
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.28)
        ax.set_title(f"{S3_LETTERS[_vi * len(S3_MODES) + _mi]}) {_mlabel}",
                     fontsize=12.5, loc="left", pad=8)
        ax.set_xlabel(_vlabel, fontsize=10.5)
        if _mi == 0:
            ax.set_ylabel("Cumulative share of manifests", fontsize=11)
        else:
            ax.tick_params(labelleft=False)
        ax.legend(loc="lower right", fontsize=8.6, framealpha=0.93)

        _sig = st["p"] < 0.05
        _pt = ("p < 0.001" if st["p"] < 0.001 else f"p = {st['p']:.3f}")
        ax.text(0.035, 0.965,
                f"Mann-Whitney {_pt}\n"
                f"median {st['med_a']:,.0f} vs {st['med_b']:,.0f}\n"
                f"rank-biserial r = {st['rb']:+.2f}",
                transform=ax.transAxes, ha="left", va="top", fontsize=8.8,
                color="#b2182b" if _sig else "0.25",
                bbox=dict(boxstyle="round,pad=0.32", fc="white",
                          ec="#b2182b" if _sig else "0.72", lw=1.1))
        s3_rows.append(dict(variable=_vlabel, mode=_mlabel, n_mode=st["n_a"],
                            n_routed=st["n_b"], median_mode=st["med_a"],
                            median_routed=st["med_b"], rank_biserial=st["rb"],
                            p_value=st["p"], excluded_nonpositive=_excl))

# Row headings above each row, anchored to the row's first axes.
for _vi, _lead in _row_lead_ax.items():
    figS7.text(0.5, _lead.get_position().y1 + 0.052, S3_ROW_TITLES[_vi],
               ha="center", va="bottom", fontsize=16, fontweight="bold")

save_figure(figS7, "Figure_S7_Routing_Failure_Modes.png",
            dpi=200, bbox_inches="tight")
plt.show()

s3_summary = pd.DataFrame(s3_rows)
print()
print(s3_summary.round({"median_mode": 1, "median_routed": 1,
                        "rank_biserial": 3, "p_value": 4}).to_string(index=False))
# Exclusions counted per variable over all manifests; the routed group recurs
# in every panel of a row, so summing excluded_nonpositive would overcount.
print("\n  non-positive values excluded from the log axis:")
for _c, _l in S3_VARS:
    _v = pd.to_numeric(manure_df[_c], errors="coerce")
    print(f"    {_l:<34}{int((_v.notna() & (_v <= 0)).sum())} of "
          f"{int(_v.notna().sum())} manifests")
print(f"\n  wrote {out_path('Figure_S7_Routing_Failure_Modes.png')}")


In [ ]:
# =========================================================================
# WHERE THE REGULATORY RECORD ENDS
#   Builds Figures 2, 3 and 4, SI Figures S1, S4, S5 and S9, and Table S6.
#   First measures how often a manifest destination is itself a permitted
#   dairy in the report set. "Traceable" means traceable within the dairy
#   nutrient-accounting record; composting facilities and brokers hold no
#   permit under this programme.
#
# Outputs: Figure_2_Manifest_Characteristics.png
#          Figure_3_Nitrogen_Accounting.png
#          Figure_4_Network_And_Exposure.png
#          Figure_S1_Reporting_Formats.png
#          Figure_S4_Manure_Trucked_Per_Month.png
#          Figure_S5_Applied_Ratio.png
#          Figure_S9_Unaccounted_Nitrogen.png
#          Table_S6_Top_Receivers.csv
# =========================================================================
print("\n" + "=" * 96)
print("FIGURES 2, 3 and 4: manifests, nitrogen balance and the trucking network")
print("=" * 96)

T6_BOOTSTRAP = 1000
PO_BOX_RE = r"\bP\.?\s*O\.?\s+BOX\b"


def norm_address(s):
    """Address reduced to uppercase alphanumeric words for exact comparison."""
    if pd.isna(s):
        return ""
    return " ".join(re.sub(r"[^A-Z0-9]+", " ", str(s).upper()).split())


ROAD_WORDS = {"AVE", "AVENUE", "RD", "ROAD", "HWY", "HIGHWAY", "ST", "CR"}
# County names as token lists, longest first, so "SAN JOAQUIN" is tested
# before any one-word name.
COUNTY_TOKENS = sorted((n.upper().split() for n in CA_COUNTY_NAMES), key=len, reverse=True)


def receiver_key(addr_norm):
    """Receiver key: normalised address without state, ZIP code, trailing
    county names or a leading name.

    A ZIP is a 5-digit code in 90000-96199, with any +4 suffix, that is neither
    the first token (a house number) nor after BOX (a PO box number). Trailing
    county names are removed one after another while at least one word
    remains, so an address ending "TULARE MERCED" loses both and a lone
    "FRESNO" is kept. Words before the first number are removed unless one is
    a road word ("AVE 152", "STUHR ROAD NEWMAN 85340"), the number is the last
    word or is followed by another number (as in a phone number), or the
    address is a PO box.
    """
    toks = addr_norm.split()
    out, after_zip = [], False
    for i, t in enumerate(toks):
        if t in ("CA", "CALIFORNIA"):
            after_zip = False
            continue
        if (i > 0 and len(t) == 5 and t.isdigit() and 90000 <= int(t) <= 96199
                and toks[i - 1] != "BOX"):
            after_zip = True
            continue
        if after_zip and len(t) == 4 and t.isdigit():
            after_zip = False
            continue
        after_zip = False
        out.append(t)
    while True:
        tail = next((c for c in COUNTY_TOKENS
                     if len(out) > len(c) and out[-len(c):] == c), None)
        if tail is None:
            break
        del out[-len(tail):]
    if not re.search(PO_BOX_RE, " ".join(out)):
        k = next((j for j, t in enumerate(out) if t.isdigit()), None)
        if (k and not ROAD_WORDS & set(out[:k])
                and k + 1 < len(out) and not out[k + 1].isdigit()):
            out = out[k:]
    return " ".join(out)


# ---- the manifest universe ----------------------------------------------
# All manure manifests, including those whose origin dairy is outside the
# Figure 5 sample.
t6 = man_n.copy()
t6["_addr_raw"] = t6["Destination Address Final"].fillna("").astype(str).str.strip()
t6["_addr"] = t6["Destination Address Final"].map(norm_address)
# Receivers are counted and ranked on this key; address forms use _addr.
t6["_recv"] = t6["_addr"].map(receiver_key)
T6_TONS = float(t6["_mass"].sum())

_raw_distinct = t6.loc[t6["_addr_raw"].ne(""), "_addr_raw"].nunique()
_norm_distinct = t6.loc[t6["_addr"].ne(""), "_addr"].nunique()
_recv_distinct = t6.loc[t6["_recv"].ne(""), "_recv"].nunique()


def top_share(addr, mass, k):
    """Percentage of tonnage held by the k largest receivers.

    Blank addresses are excluded from both numerator and denominator.
    """
    s = pd.Series(np.asarray(mass, dtype=float))
    a = pd.Series(np.asarray(addr, dtype=object))
    keep = a.notna() & a.astype(str).str.strip().ne("")
    g = s[keep].groupby(a[keep].values).sum().sort_values(ascending=False)
    return 100.0 * g.head(k).sum() / g.sum() if g.sum() > 0 else np.nan


print(f"  manifests {len(t6):,}   tonnage {T6_TONS:,.0f}")
print(f"  distinct destination addresses: {_raw_distinct} as written, "
      f"{_norm_distinct} after normalising case and punctuation, "
      f"{_recv_distinct} after dropping state, ZIP, county and leading names")
print(f"  top-10 share of tonnage: {top_share(t6['_addr_raw'], t6['_mass'], 10):.1f}% as written, "
      f"{top_share(t6['_recv'], t6['_mass'], 10):.1f}% normalised")
# Receivers that receiver_key joins from more than one normalised spelling.
_merged = t6[t6["_recv"].ne("")].groupby("_recv")["_addr"].unique()
_merged = _merged[_merged.map(len) > 1]
print(f"  receivers joined by dropping state, ZIP, county and leading names: {len(_merged)}")
for _k, _forms in _merged.items():
    print(f"    {_k}  <-  " + " | ".join(sorted(_forms)))

# ---- can the receiver be found in the report set? -----------------------
# Roster: every dairy that filed a 2024 annual report, including those outside
# the Figure 5 sample.
roster = rep_all.assign(
    _sk=rep_all["Dairy Address"].map(lambda s: street_key(s, anywhere=True)))
roster_keys = set(roster["_sk"].dropna())
roster_numcity = set()
for a in rep_all["Dairy Address"].dropna():
    _n = norm_address(a)
    _m = re.match(r"^(\d+)\s", _n)
    if _m:
        roster_numcity |= {(_m.group(1), w) for w in _n.split()[1:]}

t6["_sk"] = t6["Destination Address Final"].map(
    lambda s: street_key(s, anywhere=True))
t6["_is_pobox"] = t6["_addr"].str.contains(PO_BOX_RE, regex=True, na=False)
t6["_strict"] = t6["_sk"].notna() & t6["_sk"].isin(roster_keys)


def relaxed_match(addr_norm):
    """True when the house number and any later word both appear in one roster address.

    Looser than the street-key match: it tolerates variant street spellings and
    abbreviations but admits coincidences.
    """
    m = re.match(r"^(\d+)\s", addr_norm)
    if not m:
        return False
    num = m.group(1)
    return any((num, w) in roster_numcity for w in addr_norm.split()[1:])


t6["_relaxed"] = t6["_strict"] | t6["_addr"].map(relaxed_match)
# Non-blank, non-PO-box destination whose street key matches a roster dairy.
# Used for the traceability printouts and the review sample; not an address class.
t6["_in_roster"] = t6["_strict"] & ~t6["_is_pobox"] & t6["_addr"].ne("")

TRACE_ORDER = ["Street address",
               "Place description,\nno street number",
               "PO box only",
               "No address given"]
# Two-line names for the same four classes, for the panel b axis labels.
TRACE_TICKS = ["Street\nAddress",
               "Place\nDescription",
               "PO Box\nOnly",
               "No Address\nGiven"]
# The last three classes have no address that resolves to a parcel.
TRACE_UNPLACEABLE = TRACE_ORDER[1:]


def trace_class(r):
    """Address class of a record, by the form of its destination address.

    Tested in order: blank, PO box, no street key, then street address. The
    street key is searched anywhere in the address, so a facility name followed
    by a street address counts as a street address.
    """
    if r["_addr"] == "":
        return TRACE_ORDER[3]
    if r["_is_pobox"]:
        return TRACE_ORDER[2]
    if pd.isna(r["_sk"]):
        return TRACE_ORDER[1]
    return TRACE_ORDER[0]


t6["_trace"] = t6.apply(trace_class, axis=1)

# ---- nitrogen on the same rows ------------------------------------------
# Nitrogen as dry tons times lb N per dry ton, at the low, mid and high N
# contents used for the Figure 5 export segments.
for _tag, _pct in (("lo", C_DRY_PCT_LOW), ("mid", C_DRY_PCT_MID), ("hi", C_DRY_PCT_HIGH)):
    t6[f"_N_{_tag}"] = t6["_dry_tons"] * lbN_per_dry_ton(_pct)

by_trace = t6.groupby("_trace").agg(
    tons=("_mass", "sum"), manifests=("_mass", "size"),
    dairies=("Source PDF", "nunique"),
    N_lo=("_N_lo", "sum"), N_mid=("_N_mid", "sum"), N_hi=("_N_hi", "sum"),
).reindex(TRACE_ORDER).fillna(0)
by_trace["pct_tons"] = 100.0 * by_trace["tons"] / T6_TONS
by_trace["pct_N"] = 100.0 * by_trace["N_mid"] / by_trace["N_mid"].sum()

assert abs(by_trace["tons"].sum() - T6_TONS) < 1.0, "address classes do not partition tonnage"
assert by_trace["manifests"].sum() == len(t6), "address classes do not partition records"

by_dest = t6.groupby("dest").agg(
    tons=("_mass", "sum"), manifests=("_mass", "size"), dairies=("Source PDF", "nunique"),
).sort_values("tons", ascending=False)
by_dest["pct_tons"] = 100.0 * by_dest["tons"] / T6_TONS
assert abs(by_dest["tons"].sum() - T6_TONS) < 1.0, "destination types do not partition tonnage"

print("\n  tonnage by destination type")
for k, r in by_dest.iterrows():
    print(f"    {k:<22}{r.tons:>12,.0f} t  {r.pct_tons:>5.1f}%   "
          f"{int(r.manifests):>4} manifests  {int(r.dairies):>4} dairies")
print("\n  tonnage and nitrogen by destination address form")
for k, r in by_trace.iterrows():
    print(f"    {k.replace(chr(10), ' '):<40}{r.tons:>12,.0f} t  {r.pct_tons:>5.1f}%   "
          f"{r.N_mid:>11,.0f} lb N  {r.pct_N:>5.1f}%   {int(r.manifests):>4} manifests")

_strict_pct = 100.0 * t6.loc[t6["_strict"], "_mass"].sum() / T6_TONS
_relaxed_pct = 100.0 * t6.loc[t6["_relaxed"], "_mass"].sum() / T6_TONS
print(f"\n  traceable to a permitted dairy: {_strict_pct:.1f}% of tonnage on the street-key "
      f"match, {_relaxed_pct:.1f}% on the looser number-and-word match")
# The three largest destinations found by the loose match only.
_loose_only = (t6[t6["_relaxed"] & ~t6["_strict"]]
               .groupby("_recv")["_mass"].sum().nlargest(3))
print("  largest destinations the loose key finds and the strict key misses:")
for _a, _m in _loose_only.items():
    print(f"    {_m:>10,.0f} t   {_a}")
print("  the pair bounds the figure: the strict match misses spelling variants, the loose "
      "one admits coincidences")

# ---- receiver concentration ---------------------------------------------
named = t6[t6["_recv"].ne("")].copy()
recv = (named.groupby("_recv")
        .agg(tons=("_mass", "sum"), manifests=("_mass", "size"),
             dairies=("Source PDF", "nunique"), N_mid=("_N_mid", "sum"))
        .sort_values("tons", ascending=False))
recv["pct"] = 100.0 * recv["tons"] / recv["tons"].sum()
recv["cum_pct"] = recv["pct"].cumsum()


def hhi(shares_pct):
    """Herfindahl-Hirschman index and its equivalent number of receivers.

    HHI is the sum of squared percentage shares (0 to 10,000); 10,000 / HHI is
    the number of equal-sized receivers giving the same concentration.
    """
    h = float(np.sum(np.square(shares_pct)))
    return h, (10000.0 / h if h > 0 else np.nan)


HHI, HHI_EQ = hhi(recv["pct"].to_numpy())
TOP10, TOP20 = recv["pct"].head(10).sum(), recv["pct"].head(20).sum()

# Bootstrap over whole origin dairies, since one dairy's manifests share
# destinations. Each draw selects rows of the dairy-by-receiver tonnage matrix
# and sums the columns.
_d_codes, _d_uniq = pd.factorize(named["Source PDF"].fillna("(none)").astype(str))
_r_codes, _r_uniq = pd.factorize(named["_recv"])
_M = np.zeros((len(_d_uniq), len(_r_uniq)))
np.add.at(_M, (_d_codes, _r_codes), named["_mass"].to_numpy(float))
assert abs(_M.sum() - named["_mass"].sum()) < 1.0, "matrix loses tonnage"
_rng = np.random.default_rng(0)
_boot = np.empty((T6_BOOTSTRAP, 2))
for _b in range(T6_BOOTSTRAP):
    tot = _M[_rng.integers(0, len(_d_uniq), len(_d_uniq))].sum(axis=0)
    if tot.sum() <= 0:
        _boot[_b] = (np.nan, np.nan)
        continue
    p = 100.0 * tot / tot.sum()
    _boot[_b] = (float(np.square(p).sum()), float(np.sort(p)[::-1][:10].sum()))
HHI_CI = np.nanpercentile(_boot[:, 0], [2.5, 97.5])
TOP10_CI = np.nanpercentile(_boot[:, 1], [2.5, 97.5])

_unplaceable = float(by_trace.loc[TRACE_UNPLACEABLE, "tons"].sum())
_pob = t6[t6["_is_pobox"]]
print(f"\n  receivers with an address: {len(recv)}   "
      f"top 10 hold {TOP10:.1f}% [{TOP10_CI[0]:.1f}-{TOP10_CI[1]:.1f}], top 20 {TOP20:.1f}%")
print(f"  HHI {HHI:,.0f} [{HHI_CI[0]:,.0f}-{HHI_CI[1]:,.0f}]  "
      f"equivalent to {HHI_EQ:.1f} equal-sized receivers")
print("  an HHI this low is an unconcentrated market on the 1,500 threshold in the "
      "DOJ/FTC merger guidelines:")
print("  the head is heavy but the tail is long, so the claim is that a few dozen parties "
      "hold most of the tonnage,")
print("  not that any one of them dominates")
print(f"  PO box destinations: {len(_pob)} manifests, {_pob['_addr'].nunique()} boxes, "
      f"{_pob['_mass'].sum():,.0f} t ({100.0 * _pob['_mass'].sum() / T6_TONS:.1f}%), "
      f"{_pob['Source PDF'].nunique()} dairies")
print(f"  no address that resolves to a parcel (a mailbox, a locality, or nothing): "
      f"{_unplaceable:,.0f} t ({100.0 * _unplaceable / T6_TONS:.1f}%)")

_untr = float(t6.loc[~t6["_in_roster"], "_N_mid"].sum())
print(f"\n  nitrogen leaving for a receiver with no report counterpart: {_untr:,.0f} lb"
      f"  ({100.0 * _untr / TOTAL_N:.1f}% of the {TOTAL_N:,.0f} lb generated by the "
      f"Figure 5 sample)")

# =========================================================================
# FIGURE 2 AND FIGURE S4: SHARED STYLE
#   All four Figure 2 panels are horizontal bars ranked by size, each
#   labelled with its share of the panel total. Figure S4 is the monthly
#   manure tonnage.
# =========================================================================
# Bars with no usable information (blank destination type, no transport mode,
# or an address that does not resolve to a parcel) take Figure 3a's
# unaccounted-for grey; all others are light grey.
FIG2_DARK_GREY, FIG2_LIGHT_GREY = FIG1_GREY, "#CFCFCF"
DEST_BAR_COLORS = {"Farmer": FIG2_LIGHT_GREY, "Broker": FIG2_LIGHT_GREY,
                   "Composting Facility": FIG2_LIGHT_GREY, "Blank": FIG2_DARK_GREY,
                   "Other": FIG2_LIGHT_GREY}
TRANSPORT_BAR_COLORS = {"Pipeline": FIG2_LIGHT_GREY, "Trucked": FIG2_LIGHT_GREY,
                        "Unsure": FIG2_DARK_GREY}
MONTH_BAR_COLOR = "#8c6d4f"
TRACE_COLORS = [FIG2_LIGHT_GREY, FIG2_DARK_GREY, FIG2_DARK_GREY, FIG2_DARK_GREY]   # TRACE_ORDER


PANEL_TITLE_FS = 12
F3_TITLE_FS = 14                   # Figure 3 is drawn a step larger
F3_LABEL_FS = F3_TITLE_FS          # its axis and tick labels match the panel titles
F3_SUB_FS = 11.5                   # second line of a two-line title or axis label


def panel_letter(ax, text, x=-0.14, fs=13, y=1.04):
    """Bold panel letter above the top-left corner of an axes.

    Raise y for a panel with a two-line title.
    """
    ax.text(x, y, text, transform=ax.transAxes, fontsize=fs,
            fontweight="bold", ha="left", va="bottom")

def scale_axes_fonts(ax, k, extra=()):
    """Multiply every text size on an axes, and on any extra axes, by k."""
    for a in (ax, *extra):
        items = [a.title, a.xaxis.label, a.yaxis.label]
        items += list(a.get_xticklabels()) + list(a.get_yticklabels())
        items += list(a.texts)
        leg = a.get_legend()
        if leg is not None:
            items += list(leg.get_texts())
            if leg.get_title() is not None:
                items.append(leg.get_title())
        for t in items:
            t.set_fontsize(t.get_fontsize() * k)


def frame_map_axes(ax, like_ax):
    """Remove a map panel's ticks but keep its frame, at like_ax's spine width."""
    ax.set_xticks([])
    ax.set_yticks([])
    lw = like_ax.spines["left"].get_linewidth()
    for side in ax.spines.values():
        side.set_visible(True)
        side.set_linewidth(lw)


def share_barh(ax, totals, colors, title, xlabel, divisor):
    """Horizontal bars ranked smallest to largest, each labelled with its share."""
    o = totals.sort_values()
    tot = float(o.sum())
    ax.barh(range(len(o)), o.to_numpy(float) / divisor,
            color=[colors.get(k, "0.6") for k in o.index],
            edgecolor="black", linewidth=0.8)
    span = float(o.max()) / divisor
    for i, (k, v) in enumerate(o.items()):
        ax.text(v / divisor + span * 0.015, i, f"{100.0 * v / tot:.1f}%",
                va="center", fontsize=10)
    ax.set_yticks(range(len(o)))
    ax.set_yticklabels([str(k).replace(" Facility", "") for k in o.index], fontsize=10.5)
    ax.set_title(title, fontsize=PANEL_TITLE_FS, pad=9)
    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_xlim(0, span * 1.22)
    ax.grid(axis="x", alpha=0.3)
    return o


# Haul-date coverage for Figure S4. distribute_monthly puts a manifest with one
# haul date in that month and spreads one with both dates evenly over the range.
# Months ignore the year, so records dated outside 2023-2025 fold into their
# calendar month.
_d1 = pd.to_datetime(manure_df["First Haul Date"], errors="coerce")
_d2 = pd.to_datetime(manure_df["Last Haul Date"], errors="coerce")
_dated = _d1.notna() | _d2.notna()
_last_only = _d2.notna() & _d1.isna()
_offyear = ((_d1.notna() & ~_d1.dt.year.between(2023, 2025))
            | (_d2.notna() & ~_d2.dt.year.between(2023, 2025)))
_mass_all = pd.to_numeric(manure_df["Total Manure Amount (tons)"], errors="coerce").fillna(0)
_dated_pct = 100.0 * _mass_all[_dated].sum() / _mass_all.sum()


# =========================================================================
# FIGURE 2: WHAT THE MANIFESTS RECORD
#   a) exported manure by destination type   b) exported manure by destination address
#   c) exported wastewater by destination type   d) wastewater by transport mode
# =========================================================================
fig2 = plt.figure(figsize=(13.0, 8.4))
gs2 = GridSpec(2, 2, figure=fig2, hspace=0.42, wspace=0.24,
               top=0.945, bottom=0.085)
ax2a = fig2.add_subplot(gs2[0, 0])
ax2b = fig2.add_subplot(gs2[0, 1])
ax2c = fig2.add_subplot(gs2[1, 0])
ax2d = fig2.add_subplot(gs2[1, 1])

# a) exported manure tonnage by destination type
share_barh(ax2a, by_dest["tons"], DEST_BAR_COLORS,
           "Exported Manure by Destination Type", "Thousand Tons", 1000.0)

# b) exported manure tonnage by the form of its destination address
share_barh(ax2b, by_trace["tons"].rename(index=dict(zip(TRACE_ORDER, TRACE_TICKS))),
           dict(zip(TRACE_TICKS, TRACE_COLORS)),
           "Exported Manure by Destination Address", "Thousand Tons", 1000.0)

# c, d) exported wastewater volume by destination type and by transport mode
if INCLUDE_WASTEWATER:
    _wd = ww_df.groupby("Dest_Type")[WW_VOL_COL].sum()
    _we = ww_df.groupby("Transport_Mode")[WW_VOL_COL].sum()
    share_barh(ax2c, _wd[_wd > 0], DEST_BAR_COLORS,
               "Exported Wastewater by Destination Type", "Million Gallons", 1e6)
    share_barh(ax2d, _we[_we > 0], TRANSPORT_BAR_COLORS,
               "Exported Wastewater by Transport Mode", "Million Gallons", 1e6)
    WW_TOTAL = float(ww_df[WW_VOL_COL].sum())
    _pipe_pct = 100.0 * _we.get("Pipeline", 0.0) / WW_TOTAL
    _truck_pct = 100.0 * _we.get("Trucked", 0.0) / WW_TOTAL
    _unsure_pct = 100.0 * _we.get("Unsure", 0.0) / WW_TOTAL
else:
    WW_TOTAL = 0.0
    _pipe_pct = _truck_pct = _unsure_pct = np.nan
    for _ax in (ax2c, ax2d):
        _ax.text(0.5, 0.5, "Wastewater not included", ha="center", va="center", fontsize=12)
        _ax.axis("off")

# Panel letters.
for _ax, _ltr in ((ax2a, "a)"), (ax2b, "b)"), (ax2c, "c)"), (ax2d, "d)")):
    panel_letter(_ax, _ltr)

# Values quoted in the Figure 2 and Figure S4 captions.
print(f"\n  haul dates: {int(_dated.sum())} of {len(manure_df):,} manifests carry one "
      f"({_dated_pct:.1f}% of tonnage); {int(_last_only.sum())} give only a last haul date; "
      f"{int(_offyear.sum())} fall outside 2023-2025")
print(f"  wastewater: {len(ww_df):,} manifests, {WW_TOTAL:,.0f} gallons  "
      f"trucked {_truck_pct:.1f}%  pipeline {_pipe_pct:.1f}%  unrecorded {_unsure_pct:.1f}%")
save_figure(fig2, "Figure_2_Manifest_Characteristics.png", dpi=200,
            bbox_inches="tight")
plt.show()

# =========================================================================
# FIGURE S1: WHAT THE TWO SOURCE DOCUMENT SETS LOOK LIKE
#   a) annual report form per dairy, from Annual_Report_Formats.csv
#   b) manifest template per analysed manifest, from
#      2024_manifest_template_counts.csv
#
#   One labelled bar per document type, smallest to largest down the axis;
#   a label that would overrun the axis is drawn inside its bar. Panel a
#   counts dairies and panel b manifests, so the x axes differ.
# =========================================================================
# One neutral fill for every bar; each category is named by its label.
BAR_GREY = "#CFCFCF"


def format_bars(ax, labels, counts, xlabel):
    """Horizontal tally, one labelled bar per document type."""
    y = np.arange(len(counts))
    ax.barh(y, counts, color=BAR_GREY,
            edgecolor="black", linewidth=0.6, height=0.74, zorder=2)
    ax.set_xlim(0, max(counts) * 1.02)
    ax.set_ylim(len(counts) - 0.5, -0.5)
    ax.set_yticks([])
    ax.set_xlabel(xlabel, fontsize=13)
    ax.tick_params(axis="x", labelsize=11)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    pad = max(counts) * 0.012
    texts = [ax.text(v + pad, i, f"{t} ({v:,.0f})", va="center", ha="left",
                     fontsize=11, fontweight="bold", zorder=5)
             for i, (t, v) in enumerate(zip(labels, counts))]
    ax.figure.canvas.draw()
    inv = ax.transData.inverted()
    for i, (tx, v) in enumerate(zip(texts, counts)):
        bb = tx.get_window_extent(ax.figure.canvas.get_renderer())
        w = inv.transform((bb.x1, 0))[0] - inv.transform((bb.x0, 0))[0]
        if v + pad + w > ax.get_xlim()[1]:
            tx.set_position((pad, i))


if os.path.exists(MANIFEST_FORMATS_FILE):
    _mf = pd.read_csv(MANIFEST_FORMATS_FILE)
    _mf = _mf[_mf["template_id"].astype(str).str.strip() != "Total"]
    _mf_n = pd.to_numeric(_mf["analysis_manifests"], errors="coerce").fillna(0)
    _fmt = (pd.DataFrame({"template": _mf["template_id"].astype(str).str.strip(),
                          "n": _mf_n})
            .query("n > 0").sort_values("n").reset_index(drop=True))
    figS1 = plt.figure(figsize=(17.0, 7.2))
    gsS1 = GridSpec(1, 2, figure=figS1, wspace=0.08, left=0.015, right=0.985,
                    top=0.94, bottom=0.095)
    axS1a, axS1b = figS1.add_subplot(gsS1[0, 0]), figS1.add_subplot(gsS1[0, 1])

    if os.path.exists(REPORT_FORMATS_FILE):
        _rfall = pd.read_csv(REPORT_FORMATS_FILE)["Form Type"]
        # Reports not matched to a PDF have no known form; they are counted in
        # the printout and not plotted.
        _nomatch = int((_rfall == "Not matched to a file").sum())
        _rf = _rfall[_rfall != "Not matched to a file"].value_counts().sort_values()
        format_bars(axS1a, _rf.index, _rf.to_numpy(float), "Number of Dairies")
    else:
        axS1a.text(0.5, 0.5, "Annual_Report_Formats.csv not found",
                   ha="center", va="center", fontsize=12)
        axS1a.axis("off")
        _rf, _nomatch = pd.Series(dtype=float), 0
    format_bars(axS1b, _fmt["template"], _fmt["n"].to_numpy(float),
                "Number of Manifests")
    panel_letter(axS1a, "a)", x=-0.02)
    panel_letter(axS1b, "b)", x=-0.02)
    save_figure(figS1, "Figure_S1_Reporting_Formats.png",
                dpi=200, bbox_inches="tight")
    plt.show()
    if len(_rf):
        print(f"\n  annual report forms: {_rf.sum():,.0f} dairies over "
              f"{len(_rf)} form types; {_nomatch} further reports could not be "
              f"matched to a filed PDF and are not plotted")
        for _t, _v in _rf.iloc[::-1].items():
            print(f"    {_t:<36} {_v:4,.0f}  ({100 * _v / _rf.sum():4.1f}%)")
    _sw = _fmt.loc[_fmt["template"].str.contains("Software", case=False), "n"].sum()
    print(f"\n  manifest templates: {len(_mf):,} listed, {len(_fmt):,} with 2024 "
          f"manifests, {_fmt['n'].sum():,.0f} manifests tallied")
    print(f"    largest: " + ", ".join(
        f"{t} {v:,.0f}" for t, v in zip(_fmt["template"].iloc[::-1].head(3),
                                        _fmt["n"].iloc[::-1].head(3))))
    print(f"    software template: {_sw:,.0f} "
          f"({100 * _sw / _fmt['n'].sum():.1f}% of the tally)")
    # Template counts are per manifest document; the processed files hold one
    # row per haul record, so a document with several hauls gives several rows.
    if abs(_fmt["n"].sum() - len(manure_df) - len(ww_df)) > 0:
        print(f"    NOTE: the template counts give {_fmt['n'].sum():,.0f} manifest documents "
              f"against {len(manure_df) + len(ww_df):,} transport records in the "
              f"processed files")
else:
    print(f"\n  {MANIFEST_FORMATS_FILE} not found; Figure S1 omitted")

# =========================================================================
# FIGURE S4: MANURE TRUCKED PER MONTH
#   Monthly tonnage from manifest haul dates, spread by distribute_monthly
#   as noted above the Figure 2 block.
# =========================================================================
_monthly = distribute_monthly(manure_df, "Total Manure Amount (tons)")
_months = [calendar.month_abbr[i] for i in range(1, 13)]
_tons_m = np.array([_monthly[i] for i in range(1, 13)])
assert abs(_tons_m.sum() - _mass_all[_dated].sum()) < 1.0, \
    "monthly totals do not match the tonnage that carries a haul date"
figS4, axS4 = plt.subplots(figsize=(8.2, 5.0))
axS4.bar(range(12), _tons_m / 1000.0, width=0.72, color=MONTH_BAR_COLOR,
         edgecolor="black", linewidth=0.7)
axS4.set_xticks(range(12))
axS4.set_xticklabels(_months, fontsize=10.5)
axS4.set_ylabel("Thousand Tons", fontsize=11)
axS4.set_ylim(0, _tons_m.max() / 1000.0 * 1.16)
axS4.grid(axis="y", alpha=0.3)
save_figure(figS4, "Figure_S4_Manure_Trucked_Per_Month.png",
            dpi=200, bbox_inches="tight")
plt.show()

# =========================================================================
# TABLES
#   Table S6 (top 20 receivers) and traceability by destination type
# =========================================================================
s6 = recv.head(20).reset_index().rename(columns={"_recv": "Destination (normalised)"})
s6.insert(0, "Rank", range(1, len(s6) + 1))
# A receiver's address form is the Figure 2b class of its records, by tonnage.
_recv_class = (t6[t6["_recv"].ne("")].groupby(["_recv", "_trace"])["_mass"].sum()
               .reset_index().sort_values("_mass", ascending=False)
               .drop_duplicates("_recv").set_index("_recv")["_trace"])
ADDRESS_FORM_LABELS = {TRACE_ORDER[0]: "street", TRACE_ORDER[1]: "place", TRACE_ORDER[2]: "PO box"}
s6["Address form"] = s6["Destination (normalised)"].map(_recv_class).map(ADDRESS_FORM_LABELS)
assert s6["Address form"].notna().all(), "a Table S6 receiver has no address class"
s6["In report set"] = s6["Destination (normalised)"].isin(
    set(t6.loc[t6["_strict"], "_recv"])).map({True: "yes", False: "no"})
s6 = s6.rename(columns={"tons": "Tons", "manifests": "Manifests",
                          "dairies": "Origin dairies", "N_mid": "Nitrogen (lb)",
                          "pct": "% of tonnage", "cum_pct": "Cumulative %"})
s6 = s6.round({"Tons": 0, "Nitrogen (lb)": 0, "% of tonnage": 2, "Cumulative %": 2})
s6.to_csv(out_path("Table_S6_Top_Receivers.csv"), index=False)

# Share of tonnage traceable to a permitted dairy, by destination type and
# overall, printed for the text.
trace_by_type = pd.DataFrame({
    "In report set": t6[t6["_in_roster"]].groupby("dest")["_mass"].sum(),
    "Total tons": t6.groupby("dest")["_mass"].sum()}).fillna(0)
trace_by_type.loc["All"] = trace_by_type.sum()
print("\n  traceable to a permitted dairy, by destination type, as a share of "
      "tonnage:")
for _d, _r in trace_by_type.iterrows():
    _tt = float(_r["Total tons"])
    _pc = 0.0 if _tt == 0 else 100.0 * float(_r["In report set"]) / _tt
    print(f"    {str(_d):22s} {_pc:5.1f}%   of {_tt:>12,.0f} tons")

print(f"\n  wrote {out_path('Figure_2_Manifest_Characteristics.png')}")


# =========================================================================
# FIGURE 3: WHERE THE NITROGEN GOES, AND WHAT THE RATIO TRACKS
#   a) nitrogen accounting waterfall across the Water Board
#   b) exported and unaccounted-for fractions against nitrogen pressure
#   c) correlates of the applied-to-removed nitrogen ratio
#   d) correlates of nitrogen pressure
#
#   b-d use ratio_panel, built with the CARB sample in the previous cell.
# =========================================================================
print("\nGenerating Figure 3...")
fig3 = plt.figure(figsize=(16.0, 11.2))
gs3 = GridSpec(2, 2, figure=fig3, hspace=0.52, wspace=0.52,
               left=0.095, right=0.955, top=0.95, bottom=0.085)
# Panel-letter x, in axes fractions, for the left and right columns.
F3_LETTER_X = (-0.14, -0.13)
ax3a = fig3.add_subplot(gs3[0, 0])
ax3b = fig3.add_subplot(gs3[0, 1])
ax3c = fig3.add_subplot(gs3[1, 0])
ax3d = fig3.add_subplot(gs3[1, 1])

# a) nitrogen accounting waterfall: each bar starts where the previous one
# ended, and the final bar closes on zero.
# Nitrogen is in lb, shown in thousand US short tons.
LB_PER_KTON = 2.0e6
# Inputs green, known fates red, unaccounted-for remainder in the Figure 2
# "no information" grey.
WF_POS, WF_NEG = "#1a9850", "#b2182b"
_run = 0.0
for _i, (_lbl, _val) in enumerate(WATERFALL):
    _last = _i == len(WATERFALL) - 1
    ax3a.bar(_i, _val, bottom=_run, width=0.66,
             color=FIG1_GREY if _last else (WF_POS if _val > 0 else WF_NEG),
             edgecolor="black", linewidth=0.8, zorder=2)
    ax3a.text(_i, _run + _val / 2.0, f"{abs(_val) / LB_PER_KTON:,.0f}",
              ha="center", va="center", fontsize=10, fontweight="semibold",
              color="white", zorder=3)
    _run += _val
    if not _last:
        ax3a.plot([_i - 0.33, _i + 1 + 0.33], [_run, _run], color="0.45",
                  lw=0.9, ls=":", zorder=1)
ax3a.axhline(0, color="black", lw=1.0, zorder=1)
# Process wastewater share of the first bar: a dotted line at the wastewater
# nitrogen and a leader from the middle of that share to its label.
ax3a.plot([-0.33, 0.33], [WATERFALL_WW, WATERFALL_WW], color="black", lw=1.5,
          ls=":", zorder=3)
ax3a.plot([0.33, 0.58], [WATERFALL_WW / 2] * 2, color="black", lw=0.8, zorder=3)
ax3a.text(0.62, WATERFALL_WW / 2, "Process Wastewater", ha="left", va="center",
          fontsize=F3_SUB_FS, zorder=3)
ax3a.set_xticks(range(len(WATERFALL)))
ax3a.set_xticklabels([lbl.replace("\n-", "-").replace("\n", " ")
                      for lbl, _ in WATERFALL],
                     fontsize=F3_LABEL_FS, rotation=28, ha="right",
                     rotation_mode="anchor")
ax3a.tick_params(axis="y", labelsize=F3_LABEL_FS)
panel_letter(ax3a, "a)", x=F3_LETTER_X[0], fs=15)
ax3a.set_ylabel("Thousand Tons", fontsize=F3_LABEL_FS)
ax3a.yaxis.set_major_formatter(
    ticker.FuncFormatter(lambda v, _p: f"{v / LB_PER_KTON:,.0f}"))
ax3a.set_ylim(0, max(_c for _c in np.cumsum([v for _, v in WATERFALL])) * 1.16)
ax3a.grid(axis="y", alpha=0.3)
ax3a.set_title("Nitrogen Accounting Across Dairies", fontsize=F3_TITLE_FS, pad=10)

# b) both fractions against nitrogen pressure. They share a denominator, so
# both y axes use one range. Lines are moving medians; bands are the
# interquartile range of dairies in each window, not a CI on the median.
RATIO_SB, RATIO_BLACK = "saddlebrown", "black"
_rp3 = ratio_panel[(ratio_panel["ratio"] > 0)
                   & np.isfinite(ratio_panel["pressure"])]
F3B_YL, F3B_XL = (-0.2, 1.0), (0.5, 10)
F3B_XTICKS = (0.5, 1, 2, 5, 10)
ax3b.grid(alpha=0.3, which="both")
ax3b.axvline(1, color="0.5", lw=1.0, ls=(0, (4, 3)), zorder=1)
ax3b.axhline(0, color="0.8", lw=0.9, zorder=1)
_h3b, _f3b_clip = [], []
for _lbl, _col, _c, _ax, _side in (
        ("Unaccounted-for N Fraction", "f_unacc", RATIO_SB, ax3b, "left"),
        ("Exported N Fraction", "f_exported", RATIO_BLACK, ax3b.twinx(), "right")):
    _v = _rp3[["pressure", _col]].dropna()
    _x, _y = _v["pressure"].to_numpy(float), _v[_col].to_numpy(float)
    _r, _p = spearmanr(_x, _y)
    _in = (_y >= F3B_YL[0]) & (_y <= F3B_YL[1])
    _ax.scatter(_x[_in], _y[_in], s=9, c=_c, alpha=0.16, lw=0, zorder=3)
    _f3b_clip.append((_lbl, int((_y > F3B_YL[1]).sum()), int((_y < F3B_YL[0]).sum())))
    _bx, _bm, _q1, _q3 = moving_median(_x, _y)
    _ax.fill_between(_bx, _q1, _q3, color=_c, alpha=0.14, lw=0, zorder=5)
    _ax.plot(_bx, _bm, color=_c, lw=2.4, zorder=6)
    _ax.set_xscale("log"); _ax.set_xlim(*F3B_XL); _ax.set_ylim(*F3B_YL)
    _ax.set_ylabel(_lbl, color=_c, fontsize=F3_LABEL_FS)
    _ax.tick_params(axis="y", colors=_c, labelsize=F3_LABEL_FS)
    _ax.spines[_side].set_color(_c)
    # Each series' Spearman statistics follow its name in the legend.
    _h3b.append(Line2D([], [], color=_c, lw=2.4,
                       label=f"{_lbl} ($\\rho$ = {_r:+.3f}, {ratio_pfmt(_p)}, n = {len(_v)})"))
ax3b.set_xticks(F3B_XTICKS)
ax3b.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _p: f"{v:g}"))
ax3b.xaxis.set_minor_formatter(ticker.NullFormatter())
ax3b.tick_params(axis="x", labelsize=F3_LABEL_FS)
ax3b.set_xlabel("Nitrogen Pressure", fontsize=F3_LABEL_FS)
# The definition sits under the axis label, a size smaller.
ax3b.annotate("(available N / crop-removed N)", xy=(0.5, 0.0), xycoords=ax3b.xaxis.label,
              xytext=(0, -1), textcoords="offset points", ha="center", va="top",
              fontsize=F3_SUB_FS)
ax3b.set_title("Nitrogen Fate versus Nitrogen Pressure", fontsize=F3_TITLE_FS, pad=10)
ax3b.set_zorder(2); ax3b.patch.set_visible(False)
_lg3b = ax3b.legend(handles=_h3b, loc="upper left", fontsize=9.5, framealpha=0.95)
for _t, _h in zip(_lg3b.get_texts(), _h3b):
    _t.set_color(_h.get_color())
panel_letter(ax3b, "b)", x=F3_LETTER_X[1], fs=15)
# Dairies outside the plotted y range are dropped from the scatter rather than
# stacked on the axis; the moving median and its quartiles still use them.
for _lbl, _hi, _lo in _f3b_clip:
    print(f"  Figure 3b: {_lbl} — {_hi} dairies above {F3B_YL[1]:.2f} and "
          f"{_lo} below {F3B_YL[0]:.2f} are off the plotted range")
# The median of each plotted fraction, over every dairy behind its line.
for _lbl, _col in (("Unaccounted-for N Fraction", "f_unacc"),
                   ("Exported N Fraction", "f_exported")):
    _s = _rp3[["pressure", _col]].dropna()[_col]
    print(f"  Figure 3b: {_lbl} — median {_s.median():.3f} of available N "
          f"over {len(_s)} dairies")

# c, d) Spearman rho with bootstrap CI for each outcome against the ratio (c)
# and nitrogen pressure (d), on a shared x range. Filled markers have p < 0.05;
# the unmanifested fraction uses only dairies with exports.
RATIO_OUT = [("Unaccounted-for\nN Fraction", "f_unacc"),
             ("Exported\nN Fraction", "f_exported"),
             ("Unmanifested\nN Export Fraction", "f_unmanifested"),
             ("Herd Size", "herd")]
RHO_XLIM = (-0.2, 0.72)
# (x, outcome) -> (rho, CI low, CI high, n, p), read by the manuscript parameters cell.
RHO_FIG3 = {}
for _ax, _xcol, _ttl, _sub, _ltr in (
        (ax3c, "ratio", "Correlates of the Applied-to-Removed N Ratio", None, "c)"),
        (ax3d, "pressure", "Correlates of Nitrogen Pressure", "(available N / crop-removed N)", "d)")):
    _ax.grid(axis="x", alpha=0.3)
    _ax.axvline(0, color="0.65", lw=1.1, zorder=2)
    for _yi, (_lbl, _col) in enumerate(RATIO_OUT):
        _d = _rp3[_rp3["exported"] > 0] if _col == "f_unmanifested" else _rp3
        _v = _d[[_xcol, _col]].dropna()
        _r, _lo, _hi, _n, _p = boot_rho_ci(_v[_xcol], _v[_col], seed=_yi)
        RHO_FIG3[(_xcol, _col)] = (_r, _lo, _hi, _n, _p)
        _y = -_yi
        _ax.plot([_lo, _hi], [_y, _y], color=RATIO_SB, lw=2.0, solid_capstyle="butt", zorder=3)
        for _e in (_lo, _hi):
            _ax.plot([_e, _e], [_y - 0.12, _y + 0.12], color=RATIO_SB, lw=1.6, zorder=3)
        _ax.plot([_r], [_y], marker="o", ms=7.5, mec=RATIO_SB, mew=1.9, zorder=4,
                 mfc=RATIO_SB if _p < 0.05 else "white")
        if _p < 0.05:
            _pad = 0.16 * (RHO_XLIM[1] - RHO_XLIM[0])
            _cx = min(max(_r, RHO_XLIM[0] + _pad), RHO_XLIM[1] - _pad)
            _ax.text(_cx, _y + 0.26, f"$\\rho$ = {_r:+.3f}, {ratio_pfmt(_p)}",
                     ha="center", va="bottom", fontsize=9.5, color="0.15")
        _ax.text(0.985, _y, f"n={_n}", transform=_ax.get_yaxis_transform(),
                 ha="right", va="center", fontsize=9.5, color="0.25")
    _ax.set_xlim(*RHO_XLIM); _ax.set_ylim(-len(RATIO_OUT) + 0.5, 0.6)
    _ax.set_yticks([-_i for _i in range(len(RATIO_OUT))])
    _ax.set_yticklabels([_l for _l, _ in RATIO_OUT], fontsize=F3_LABEL_FS)
    _ax.tick_params(axis="x", labelsize=F3_LABEL_FS)
    _ax.set_xlabel("Spearman $\\rho$", fontsize=F3_LABEL_FS)
    if _sub:
        # The definition sits under the title, a size smaller.
        _ax.annotate(_sub, xy=(0.5, 1.0), xycoords="axes fraction", xytext=(0, 10),
                     textcoords="offset points", ha="center", va="bottom", fontsize=F3_SUB_FS)
        _ax.set_title(_ttl, fontsize=F3_TITLE_FS, pad=10 + F3_SUB_FS * 1.35)
    else:
        _ax.set_title(_ttl, fontsize=F3_TITLE_FS, pad=10)
    panel_letter(_ax, _ltr, x=F3_LETTER_X[0 if _ax is ax3c else 1], fs=15)

save_figure(fig3, "Figure_3_Nitrogen_Accounting.png", dpi=200,
            bbox_inches="tight")
plt.show()
print("Saved Figure 3.")


# Dairy-marker scale and the county overlay threshold, used by Figure 4a.
AU_AREA_PER_UNIT = 0.0105          # points^2 per animal unit
AU_LEGEND_VALUES = (500, 2500, 10000)
COUNTY_MIN_OVERLAP = 0.05
DAIRY_POINT_COLOR = "#d95f02"

# =========================================================================
# FIGURE S5: REPORTED AND COUNTERFACTUAL APPLIED-TO-REMOVED RATIOS
#   Bars (0.2 bins): counterfactual applied-to-removed N ratio, with the
#   unaccounted-for N counted as applied, for dairies with a reported crop
#   removal. Outline: the same dairies' reported ratios. Dashed line: the
#   Dairy Order threshold of 1.4.
#
#   The printout tests for bunching of reported ratios below the threshold
#   and compares the unaccounted-for N fraction of the 1.2-1.4 band with the
#   rest of the sample.
#
#   Reads ratio_panel, as Figure 3 does.
# =========================================================================
RATIO_THR = 1.40
RATIO_BAND = (1.20, 1.40)
RATIO_WIN = 0.20                      # half-width of the bunching test window
RATIO_BW = 0.20                       # Figure S5 bin width
RATIO_CF_XMAX = 6.0                   # Figure S5 x-axis limit
RATIO_GREY = "#9E9E9E"

_rt = ratio_panel[ratio_panel["ratio"].notna()]
_in_band = _rt["ratio"].between(*RATIO_BAND)

# =========================================================================
# APPLIED-TO-REMOVED RATIO IF THE UNACCOUNTED-FOR NITROGEN WERE APPLIED
#   Counterfactual ratio = reported ratio + max(unaccounted-for N, 0) /
#   reported crop removal: the ratio the report would show had the
#   unaccounted-for N also been applied. Dairies with no or zero reported
#   removal, or no unaccounted-for figure, are left out.
# =========================================================================
_wi = ratio_panel[ratio_panel["ratio"].notna() & (ratio_panel["uptake"] > 0)
                  & ratio_panel["unacc"].notna()]
_wi_ratio = _wi["ratio"] + _wi["unacc"].clip(lower=0) / _wi["uptake"]

figS5, axS5 = plt.subplots(figsize=(11.0, 5.2))
# Rounded so np.arange float drift does not shift the bin edge at the threshold.
_cf_edges = np.round(np.arange(0, RATIO_CF_XMAX + RATIO_BW, RATIO_BW), 6)
_cf_cnt, _ = np.histogram(_wi_ratio[_wi_ratio < RATIO_CF_XMAX], bins=_cf_edges)
_as_cnt, _ = np.histogram(_wi.loc[_wi["ratio"] < RATIO_CF_XMAX, "ratio"], bins=_cf_edges)
_bS5 = axS5.bar(_cf_edges[:-1] + RATIO_BW / 2, _cf_cnt, width=RATIO_BW * 0.92,
                color=RATIO_GREY, edgecolor="white", linewidth=0.5, zorder=3,
                label="Counterfactual Applied-to-Removed Ratio")
_sS5 = axS5.stairs(_as_cnt, _cf_edges, color="#24282B", lw=1.4, zorder=4,
                   label="Reported Applied-to-Removed Ratio")
_ymax = int(max(_cf_cnt.max(), _as_cnt.max()))
axS5.axvline(RATIO_THR, color="#24282B", lw=1.6, ls="--", zorder=5)
axS5.set_xlim(0, RATIO_CF_XMAX)
axS5.set_ylim(0, _ymax * 1.06)
axS5.set_xlabel("Applied-to-removed N ratio", fontsize=13)
axS5.set_ylabel("Dairies", fontsize=13)
axS5.tick_params(labelsize=11)
axS5.grid(axis="y", alpha=0.3)
axS5.set_axisbelow(True)
axS5.annotate("Dairy Order Threshold", xy=(RATIO_THR, _ymax * 0.985),
              xytext=(RATIO_THR + 0.2, _ymax * 0.985), fontsize=11, va="center",
              ha="left", color="#24282B",
              arrowprops=dict(arrowstyle="-", color="#24282B", lw=1.0))
# Reported outline first in the legend.
axS5.legend(handles=[_sS5, _bS5], loc="upper right", fontsize=11, frameon=False)
save_figure(figS5, "Figure_S5_Applied_Ratio.png", dpi=200,
            bbox_inches="tight")
plt.show()

# Bunching: exact binomial test on dairies within RATIO_WIN below versus above
# the threshold. Band versus rest: Mann-Whitney U with rank-biserial
# correlation, and a bootstrap CI on the difference in medians.
_nlo = int(((_rt["ratio"] >= RATIO_THR - RATIO_WIN) & (_rt["ratio"] < RATIO_THR)).sum())
_nhi = int(((_rt["ratio"] >= RATIO_THR) & (_rt["ratio"] < RATIO_THR + RATIO_WIN)).sum())
_bp = binomtest(_nhi, _nlo + _nhi, 0.5).pvalue
_ra = _rt.loc[_in_band, "f_unacc"].replace([np.inf, -np.inf], np.nan).dropna()
_rb = _rt.loc[~_in_band, "f_unacc"].replace([np.inf, -np.inf], np.nan).dropna()
_ru, _rp = mannwhitneyu(_ra, _rb, alternative="two-sided")
_rbc = 2 * _ru / (len(_ra) * len(_rb)) - 1
_rrng = np.random.default_rng(0)
_rd = np.array([np.median(_rrng.choice(_ra.values, len(_ra)))
                - np.median(_rrng.choice(_rb.values, len(_rb)))
                for _ in range(RATIO_BOOT_DRAWS)])
_rdlo, _rdhi = np.percentile(_rd, [2.5, 97.5])
print(f"\n  reported ratios: {len(_rt):,} dairies report one; "
      f"{int((_rt['ratio'] >= RATIO_THR).sum())} "
      f"({(_rt['ratio'] >= RATIO_THR).mean():.1%}) are at or above {RATIO_THR}; "
      f"{int((_rt['ratio'] <= 0).sum())} report zero")
print(f"    pile-up: within +/-{RATIO_WIN} of the threshold {_nlo} fall below and "
      f"{_nhi} above, {_nlo / max(_nhi, 1):.1f}x, exact binomial p={_bp:.3g}")
print(f"    unaccounted-for N fraction, band vs rest: medians {_ra.median():.3f} "
      f"(n={len(_ra)}) and {_rb.median():.3f} (n={len(_rb)}); Mann-Whitney "
      f"p={_rp:.4g}, rank-biserial {_rbc:+.3f} "
      f"(band dairy ranks higher in {(_rbc + 1) / 2:.0%} of pairs); median "
      f"difference {_ra.median() - _rb.median():+.3f} [{_rdlo:+.3f}, {_rdhi:+.3f}]")

print(f"\n  applied-to-removed ratio above {RATIO_THR}: "
      f"{int((_rt['ratio'] > RATIO_THR).sum())} of {len(_rt):,} dairies reporting a ratio "
      f"({int(np.isclose(_rt['ratio'], RATIO_THR).sum())} more report exactly {RATIO_THR:.2f})")
print(f"    if all unaccounted-for N were applied: "
      f"{int((_wi_ratio > RATIO_THR).sum())} of the {len(_wi):,} dairies with a reported "
      f"crop removal ({(_wi_ratio > RATIO_THR).mean():.0%}), against "
      f"{int((_wi['ratio'] > RATIO_THR).sum())} of them as reported; median counterfactual "
      f"ratio {_wi_ratio.median():.2f}")
_rt_rem = _num(rep_n.loc[_rt.index, CROP_REMOVED])
_n_none, _n_zero = int(_rt_rem.isna().sum()), int((_rt_rem == 0).sum())
print(f"    left out of the {len(_rt):,} reporting a ratio: {_n_none} with no reported "
      f"removal, {_n_zero} with a removal of zero, "
      f"{len(_rt) - len(_wi) - _n_none - _n_zero} with no unaccounted-for figure")
print(f"    Figure S5: {int((_wi_ratio >= RATIO_CF_XMAX).sum())} counterfactual ratios at or "
      f"above {RATIO_CF_XMAX:g} and {int((_wi['ratio'] >= RATIO_CF_XMAX).sum())} reported ratios "
      f"are off the axis")

# =========================================================================
# FIGURE S9: UNACCOUNTED-FOR NITROGEN BY DAIRY, AND AGAINST DAIRY SIZE
#   a) unaccounted-for nitrogen per dairy, in lb
#   b) the same as a fraction of the dairy's available nitrogen
#   c) unaccounted-for nitrogen against dairy size in animal units
#   d) the unaccounted-for fraction against dairy size
#
#   Dairies are those behind the unaccounted-for line of Figure 3b. Animal
#   units are computed as in Figure 4a, so c and d hold only dairies whose
#   report fills all six herd classes.
# =========================================================================
print("\nGenerating Figure S9...")
S9_LB_LIM = (-1.0e6, 2.0e6)           # a) x axis and c) y axis, lb
S9_LB_BW = 5.0e4                      # a) bin width, lb
S9_FR_LIM = (-1.0, 1.0)               # b) x axis
S9_FR_BW = 0.05                       # b) bin width
S9_AU_LIM = (10, 2.0e4)               # c, d) x axis, animal units
S9_AU_TICKS = (10, 100, 1000, 10000)

_s9 = _rp3.loc[_rp3[["pressure", "f_unacc"]].dropna().index].copy()
_hd9 = rep_n.loc[_s9.index, list(AU_FACTORS)].apply(pd.to_numeric, errors="coerce")
_s9["au"] = ((_hd9 * pd.Series(AU_FACTORS)).sum(axis=1)
             .where(_hd9.notna().all(axis=1)))

# Pounds are written with thousands separators and a true minus sign.
_lb_fmt9 = ticker.FuncFormatter(lambda v, _p: f"{v:,.0f}".replace("-", "\u2212"))

figS9 = plt.figure(figsize=(16.0, 11.2))
gsS9 = GridSpec(2, 2, figure=figS9, hspace=0.45, wspace=0.40,
                left=0.095, right=0.955, top=0.95, bottom=0.085)
axS9a = figS9.add_subplot(gsS9[0, 0])
axS9b = figS9.add_subplot(gsS9[0, 1])
axS9c = figS9.add_subplot(gsS9[1, 0])
axS9d = figS9.add_subplot(gsS9[1, 1])

# a, b) histograms in Figure 3a's unaccounted-for grey. Dairies outside the
# axis range are left out of the bars and counted in the printout.
for _ax9, _col9, _lim9, _bw9, _xl9, _sub9, _ttl9, _ltr9 in (
        (axS9a, "unacc", S9_LB_LIM, S9_LB_BW, "Unaccounted-for N (lb)", None,
         "Unaccounted-for Nitrogen per Dairy", "a)"),
        (axS9b, "f_unacc", S9_FR_LIM, S9_FR_BW, "Unaccounted-for N Fraction",
         "(unaccounted-for N / available N)",
         "Unaccounted-for Nitrogen Fraction per Dairy", "b)")):
    # Rounded so np.arange float drift does not move the bin edge at zero.
    _e9 = np.round(np.arange(_lim9[0], _lim9[1] + _bw9 / 2, _bw9), 6)
    _n9, _ = np.histogram(_s9[_col9], bins=_e9)
    _ax9.bar(_e9[:-1] + _bw9 / 2, _n9, width=_bw9, color=FIG1_GREY,
             edgecolor="black", linewidth=0.6, zorder=2)
    _ax9.axvline(0, color="0.5", lw=1.0, ls=(0, (4, 3)), zorder=1)
    _ax9.set_xlim(*_lim9)
    _ax9.set_ylim(0, _n9.max() * 1.08)
    _ax9.set_xlabel(_xl9, fontsize=F3_LABEL_FS)
    _ax9.set_ylabel("Dairies", fontsize=F3_LABEL_FS)
    _ax9.tick_params(labelsize=F3_LABEL_FS)
    _ax9.grid(axis="y", alpha=0.3)
    _ax9.set_axisbelow(True)
    if _sub9:
        # The definition sits under the axis label, a size smaller.
        _ax9.annotate(_sub9, xy=(0.5, 0.0), xycoords=_ax9.xaxis.label,
                      xytext=(0, -1), textcoords="offset points", ha="center",
                      va="top", fontsize=F3_SUB_FS)
    _ax9.set_title(_ttl9, fontsize=F3_TITLE_FS, pad=10)
    panel_letter(_ax9, _ltr9, x=F3_LETTER_X[0 if _ax9 is axS9a else 1], fs=15)
axS9a.xaxis.set_major_locator(ticker.MultipleLocator(1.0e6))
axS9a.xaxis.set_minor_locator(ticker.MultipleLocator(2.5e5))
axS9a.xaxis.set_major_formatter(_lb_fmt9)
axS9b.xaxis.set_major_locator(ticker.MultipleLocator(0.5))
axS9b.xaxis.set_minor_locator(ticker.MultipleLocator(0.1))

# c, d) each measure against dairy size, styled as Figure 3b: points, moving
# median with interquartile band, Spearman statistics in the legend. Dairies
# off the plotted range are left out of the scatter but kept in the line, band
# and statistics.
_s9_stats = []
for _ax9, _col9, _yl9, _f9, _ylab9, _nm9, _ttl9, _ltr9 in (
        (axS9c, "unacc", S9_LB_LIM, ",.0f", "Unaccounted-for N (lb)",
         "Unaccounted-for N", "Unaccounted-for Nitrogen versus Dairy Size", "c)"),
        (axS9d, "f_unacc", F3B_YL, ".2f", "Unaccounted-for N Fraction",
         "Unaccounted-for N Fraction",
         "Unaccounted-for Nitrogen Fraction versus Dairy Size", "d)")):
    _v9 = _s9[["au", _col9]].dropna()
    _x9, _y9 = _v9["au"].to_numpy(float), _v9[_col9].to_numpy(float)
    _r9, _p9 = spearmanr(_x9, _y9)
    _inx9 = (_x9 >= S9_AU_LIM[0]) & (_x9 <= S9_AU_LIM[1])
    _iny9 = (_y9 >= _yl9[0]) & (_y9 <= _yl9[1])
    _in9 = _inx9 & _iny9
    _ax9.grid(alpha=0.3, which="both")
    _ax9.axhline(0, color="0.8", lw=0.9, zorder=1)
    _ax9.scatter(_x9[_in9], _y9[_in9], s=9, c=RATIO_SB, alpha=0.16, lw=0, zorder=3)
    _bx9, _bm9, _q19, _q39 = moving_median(_x9, _y9)
    _ax9.fill_between(_bx9, _q19, _q39, color=RATIO_SB, alpha=0.14, lw=0, zorder=5)
    _ax9.plot(_bx9, _bm9, color=RATIO_SB, lw=2.4, zorder=6)
    _ax9.set_xscale("log"); _ax9.set_xlim(*S9_AU_LIM); _ax9.set_ylim(*_yl9)
    _ax9.set_xticks(S9_AU_TICKS)
    _ax9.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _p: f"{v:,.0f}"))
    _ax9.xaxis.set_minor_formatter(ticker.NullFormatter())
    _ax9.tick_params(axis="x", labelsize=F3_LABEL_FS)
    _ax9.set_xlabel("Dairy Size (Animal Units)", fontsize=F3_LABEL_FS)
    _ax9.set_ylabel(_ylab9, color=RATIO_SB, fontsize=F3_LABEL_FS)
    _ax9.tick_params(axis="y", colors=RATIO_SB, labelsize=F3_LABEL_FS)
    _ax9.spines["left"].set_color(RATIO_SB)
    _h9 = Line2D([], [], color=RATIO_SB, lw=2.4,
                 label=f"{_nm9} ($\\rho$ = {_r9:+.3f}, {ratio_pfmt(_p9)}, n = {len(_v9)})")
    _lg9 = _ax9.legend(handles=[_h9], loc="upper left", fontsize=9.5, framealpha=0.95)
    _lg9.get_texts()[0].set_color(RATIO_SB)
    _ax9.set_title(_ttl9, fontsize=F3_TITLE_FS, pad=10)
    panel_letter(_ax9, _ltr9, x=F3_LETTER_X[0 if _ax9 is axS9c else 1], fs=15)
    _s9_stats.append((_ltr9, _ylab9, _yl9, _f9, _r9, _p9, len(_v9),
                      int((~_inx9).sum()), int((_inx9 & ~_iny9).sum())))
axS9c.yaxis.set_major_formatter(_lb_fmt9)

save_figure(figS9, "Figure_S9_Unaccounted_Nitrogen.png", dpi=200,
            bbox_inches="tight")
plt.show()
print("Saved Figure S9.")
print(f"  Figure S9: {len(_s9)} dairies; {int(_s9['au'].notna().sum())} have all six "
      f"herd classes for animal units")
for _ltr9, _lbl9, _col9, _lim9, _fm9, _f9 in (
        ("a)", "unaccounted-for N (lb)", "unacc", S9_LB_LIM, ",.0f", ",.0f"),
        ("b)", "unaccounted-for N fraction", "f_unacc", S9_FR_LIM, ".3f", ".2f")):
    _c9 = _s9[_col9]
    print(f"    {_ltr9} {_lbl9}: median {_c9.median():{_fm9}}; "
          f"{int((_c9 < 0).sum())} dairies below zero; "
          f"{int((_c9 < _lim9[0]).sum())} dairies below {_lim9[0]:{_f9}} and "
          f"{int((_c9 > _lim9[1]).sum())} above {_lim9[1]:{_f9}} are off the axis")
for _ltr9, _lbl9, _yl9, _f9, _r9, _p9, _nn9, _offx9, _offy9 in _s9_stats:
    print(f"    {_ltr9} {_lbl9} against animal units: Spearman rho = {_r9:+.3f}, "
          f"p = {_p9:.3g}, n = {_nn9}; off the plotted range, {_offx9} below "
          f"{S9_AU_LIM[0]:,} animal units and {_offy9} more outside "
          f"{_yl9[0]:{_f9}} to {_yl9[1]:{_f9}}")

# =========================================================================
# FIGURE 4: THE TRUCKING NETWORK AND WHO LIVES BESIDE IT
#   a) dairies, sized by animal units
#   b) truck passes over the routed network, in the same frame as a
#   c) block-group median household income, in the same frame
#   d) loaded hauls against one-way route distance
#   e) near-road exposure by block-group income quintile
#   f) exposure rate against community size, one point per place
#
#   Each panel's text is rescaled to this figure after it is drawn.
# =========================================================================
print("\nGenerating Figure 4...")
fig4 = plt.figure(figsize=(21, 19))
gs4 = GridSpec(2, 6, figure=fig4, height_ratios=[2.1, 1.0],
               hspace=0.20, wspace=1.70, left=0.075, right=0.925)
# Each map cell is split into the map and a colourbar slot, so the three maps
# keep the same width.
_gs_top = [GridSpecFromSubplotSpec(1, 2, subplot_spec=gs4[0, _i:_i + 2],
                                   width_ratios=[1.0, 0.045], wspace=0.05)
           for _i in (0, 2, 4)]
ax4a = fig4.add_subplot(_gs_top[0][0, 0])
ax4b = fig4.add_subplot(_gs_top[1][0, 0])
ax4c = fig4.add_subplot(_gs_top[2][0, 0])
cax4a = fig4.add_subplot(_gs_top[0][0, 1])
cax4b = fig4.add_subplot(_gs_top[1][0, 1])
cax4c = fig4.add_subplot(_gs_top[2][0, 1])
# a has a legend instead of a colourbar; its slot stays empty.
cax4a.axis("off")
ax4d = fig4.add_subplot(gs4[1, 0:2])
ax4e = fig4.add_subplot(gs4[1, 2:4])
ax4f = fig4.add_subplot(gs4[1, 4:6])

# a) dairies, zoomed to the Water Board. Animal units are the report herd
# counts weighted by the Order R5-2008-0066 factors; geocodes are joined on
# Source PDF. Only dairies with every herd class filled are drawn, since a
# blank class would understate the weighted sum.
if os.path.exists(CA_STATE_FILE) and os.path.exists(DAIRY_GEOCODE_FILE):
    _ca = gpd.read_file(CA_STATE_FILE).to_crs(ANALYSIS_CRS)
    _wbp = water_board.to_crs(ANALYSIS_CRS)
    _geo = pd.read_csv(DAIRY_GEOCODE_FILE, low_memory=False)
    _rep_au = df_report[["Source PDF"]].copy()
    _hd = df_report[list(AU_FACTORS)].apply(pd.to_numeric, errors="coerce")
    _rep_au["au"] = ((_hd * pd.Series(AU_FACTORS)).sum(axis=1)
                     .where(_hd.notna().all(axis=1)))
    _dairies = (_geo.dropna(subset=["lat", "lon"])
                    .merge(_rep_au, on="Source PDF", how="left"))
    assert len(_dairies) == _geo["lat"].notna().sum(), \
        "joining animal units to the geocodes changed the row count"
    _dpts = gpd.GeoDataFrame(
        _dairies, geometry=gpd.points_from_xy(_dairies["lon"], _dairies["lat"]),
        crs="EPSG:4326").to_crs(ANALYSIS_CRS)
    _sized = _dpts[_dpts["au"].notna()]

    _ca.plot(ax=ax4a, facecolor="0.96", edgecolor="0.55", linewidth=0.8, zorder=1)
    # Counties dissolved from the block-group file, kept where at least
    # COUNTY_MIN_OVERLAP of their area lies in the Water Board.
    _counties = None
    if os.path.exists(BG_SHP_FILE):
        _counties = (gpd.read_file(BG_SHP_FILE)[["COUNTYFP", "geometry"]]
                     .dissolve("COUNTYFP").to_crs(ANALYSIS_CRS).reset_index())
        _counties["county"] = _counties["COUNTYFP"].map(COUNTY_FIPS)
        _wb_geom = _wbp.geometry.union_all()
        _inter = _counties.geometry.intersection(_wb_geom)
        _counties["wb_frac"] = _inter.area / _counties.geometry.area
        # Each county is labelled at a representative point of its part inside
        # the Water Board.
        _counties["label_pt"] = _inter.representative_point()
        _counties = _counties[_counties["wb_frac"] >= COUNTY_MIN_OVERLAP].copy()
        _counties.plot(ax=ax4a, facecolor="none", edgecolor="0.62",
                       linewidth=0.55, zorder=2)
        print(f"  Figure 4a: {len(_counties)} counties with at least "
              f"{COUNTY_MIN_OVERLAP:.0%} of their area in the Water Board")
    _wbp.plot(ax=ax4a, facecolor="none", edgecolor="dodgerblue", linewidth=1.6,
              linestyle="--", zorder=3)
    _sized.plot(ax=ax4a, color=DAIRY_POINT_COLOR, alpha=0.62,
                markersize=_sized["au"] * AU_AREA_PER_UNIT,
                edgecolor="black", linewidth=0.35, zorder=5)

    # Zoom to the Water Board with a margin, and label the counties whose label
    # point falls inside the frame.
    _x0, _y0, _x1, _y1 = _wbp.total_bounds
    _mx, _my = (_x1 - _x0) * 0.03, (_y1 - _y0) * 0.02
    ax4a.set_xlim(_x0 - _mx, _x1 + _mx)
    ax4a.set_ylim(_y0 - _my, _y1 + _my)
    if _counties is not None:
        for _r in _counties.itertuples():
            _c = _r.label_pt
            if _x0 - _mx < _c.x < _x1 + _mx and _y0 - _my < _c.y < _y1 + _my:
                ax4a.annotate(_r.county, (_c.x, _c.y), ha="center", va="center",
                              fontsize=5.4, color="0.25", zorder=6,
                              path_effects=[path_effects.withStroke(
                                  linewidth=1.8, foreground="white")])
    frame_map_axes(ax4a, ax4b)
    panel_letter(ax4a, "a)", x=-0.11, y=1.05)
    _handles = [Line2D([], [], marker="o", ls="", color=DAIRY_POINT_COLOR,
                       alpha=0.62, mec="black", mew=0.35,
                       markersize=np.sqrt(_v * AU_AREA_PER_UNIT),
                       label=f"{_v:,}")
                for _v in AU_LEGEND_VALUES]
    _handles.append(Line2D([], [], color="dodgerblue", ls="--", lw=1.6,
                           label="Central Valley\nWater Board"))
    # Legend in the upper right, the corner of the frame with no dairies.
    _leg = ax4a.legend(handles=_handles,
                       loc="upper right", fontsize=7.4,
                       labelspacing=0.85, borderpad=0.55, frameon=True,
                       handletextpad=1.0)
    # Legend above the county labels.
    _leg.set_zorder(8)
    print(f"    {len(_sized):,} of {len(_geo):,} report dairies drawn, holding "
          f"{_sized['au'].sum():,.0f} of {_rep_au['au'].sum():,.0f} animal units "
          f"({100 * _sized['au'].sum() / _rep_au['au'].sum():.1f}%)")
else:
    ax4a.text(0.5, 0.5, "Dairy geocodes not found.\nRun "
              "review/annual-reports/geocode_dairy_addresses.py", ha="center", va="center",
              fontsize=10)
    frame_map_axes(ax4a, ax4b)
    panel_letter(ax4a, "a)", x=-0.11, y=1.05)

ax4a.set_title("Dairy Size\nin Animal Units (2024)",
               fontsize=PANEL_TITLE_FS, pad=9)

# b) truck passes on the routed network, in a's frame, without county borders.
if "positive_trips" in globals() and not positive_trips.empty and "_wbp" in dir():
    _net4 = positive_trips.to_crs(ANALYSIS_CRS)
    _ca.plot(ax=ax4b, facecolor="0.96", edgecolor="0.55", linewidth=0.8, zorder=1)
    _wbp.plot(ax=ax4b, facecolor="none", edgecolor="dodgerblue", linewidth=1.6,
              linestyle="--", zorder=3)
    _norm4 = LogNorm(vmin=vmin_t, vmax=vmax_t)
    _net4.plot(ax=ax4b, column="htot", cmap="YlOrRd", norm=_norm4,
               linewidth=1.7, zorder=5)
    _cb4 = fig4.colorbar(cm.ScalarMappable(norm=_norm4, cmap="YlOrRd"),
                         cax=cax4b)
    _cb4.ax.tick_params(labelsize=15)
    _cb4.ax.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f}"))
    ax4b.set_xlim(_x0 - _mx, _x1 + _mx)
    ax4b.set_ylim(_y0 - _my, _y1 + _my)
else:
    ax4b.text(0.5, 0.5, "Routed network not available",
              ha="center", va="center", fontsize=14)
frame_map_axes(ax4b, ax4d)
ax4b.set_title("Truck Passes\n(10t + 20t, loaded + return)", fontsize=23, pad=12)
panel_letter(ax4b, "b)", x=-0.11, fs=26, y=1.05)

# c) block-group median household income in a's frame, with the labelled
# cities; no network or county borders.
if "bg_exp" in globals() and "_wbp" in dir():
    _bgm = bg_exp.to_crs(ANALYSIS_CRS)
    _ca.plot(ax=ax4c, facecolor="0.96", edgecolor="0.55", linewidth=0.8, zorder=1)
    _bgm.plot(ax=ax4c, color="#f0f0f0", edgecolor="white", linewidth=0.1, zorder=2)
    _incm = _bgm[_bgm[BG_INCOME_COL].notna() & (_bgm[BG_INCOME_COL] > 0)]
    if not _incm.empty:
        _inorm4 = Normalize(vmin=_incm[BG_INCOME_COL].min(),
                            vmax=_incm[BG_INCOME_COL].max())
        _incm.plot(ax=ax4c, column=BG_INCOME_COL, cmap="Greens", norm=_inorm4,
                   edgecolor="black", linewidth=0.1, zorder=4)
        _cbx = fig4.colorbar(cm.ScalarMappable(norm=_inorm4, cmap="Greens"),
                             cax=cax4c)
        _cbx.ax.tick_params(labelsize=15)
        _cbx.ax.yaxis.set_major_formatter(ticker.StrMethodFormatter("{x:,.0f}"))
    _wbp.plot(ax=ax4c, facecolor="none", edgecolor="dodgerblue", linewidth=1.6,
              linestyle="--", zorder=5)
    _cty4 = cities.to_crs(ANALYSIS_CRS)
    _cty4.plot(ax=ax4c, color="black", markersize=16, zorder=7)
    for _, _cr in _cty4.iterrows():
        ax4c.text(_cr.geometry.x, _cr.geometry.y, _cr["city"], fontsize=9,
                  ha="left", va="bottom", weight="bold", zorder=8,
                  path_effects=CITY_LABEL_HALO)
    ax4c.set_xlim(_x0 - _mx, _x1 + _mx)
    ax4c.set_ylim(_y0 - _my, _y1 + _my)
else:
    ax4c.text(0.5, 0.5, "Block-group income not available",
              ha="center", va="center", fontsize=14)
frame_map_axes(ax4c, ax4d)
ax4c.set_title("Median Household\nIncome ($)", fontsize=23, pad=12)
panel_letter(ax4c, "c)", x=-0.11, fs=26, y=1.05)

# --- 4d: Hauls vs route distance ---
if os.path.exists(OD_LINES_FILE):
    od_lines_4 = gpd.read_file(OD_LINES_FILE)
    if 'route_km' in od_lines_4.columns:
        od_lines_4['route_mi'] = (
            pd.to_numeric(od_lines_4['route_km'], errors='coerce') * MI_PER_KM)
        # Validated routes only; routing_status == 'success' when VALID_COL is
        # absent.
        if VALID_COL in od_lines_4.columns:
            success_lines = od_lines_4[as_bool_series(od_lines_4[VALID_COL])].copy()
            f_basis = VALID_COL
        else:
            success_lines = od_lines_4[od_lines_4['routing_status'] == 'success'].copy()
            f_basis = "routing_status == 'success'"
        melted = success_lines.melt(
            id_vars=['RouteID', 'route_mi'],
            value_vars=['Hauls_10ton', 'Hauls_20ton'],
            var_name='Haul_Type', value_name='Num_Hauls')
        melted = melted[melted['Num_Hauls'] > 0]
        melted['Haul_Type'] = melted['Haul_Type'].replace({
            'Hauls_10ton': '10-Ton Hauls',
            'Hauls_20ton': '20-Ton Hauls',
        })
        # Bin width and printed cutoffs in DIST_UNIT. The x axis ends at the
        # first bin edge at or above the longest plotted route.
        FIG4F_BINWIDTH = 5.0
        FIG4F_LONG_CUT = 50.0
        FIG4F_CUTS = (10, 25, 50)
        max_mi = melted['route_mi'].max()
        FIG4F_XMAX = float(np.ceil(max_mi / FIG4F_BINWIDTH) * FIG4F_BINWIDTH)
        assert max_mi <= FIG4F_XMAX, "the longest route falls outside the x range"
        FIG4F_LABEL_FS = 29
        bin_kw = dict(binwidth=FIG4F_BINWIDTH, binrange=(0, FIG4F_XMAX),
                      edgecolor='white', alpha=0.6)
        sns.histplot(data=melted[melted['Haul_Type'] == '10-Ton Hauls'],
                     x='route_mi', weights='Num_Hauls',
                     color='dodgerblue', label='10-Ton Hauls',
                     shrink=1.0, ax=ax4d, **bin_kw)
        sns.histplot(data=melted[melted['Haul_Type'] == '20-Ton Hauls'],
                     x='route_mi', weights='Num_Hauls',
                     color='darkorange', label='20-Ton Hauls',
                     shrink=0.7, ax=ax4d, **bin_kw)
        ax4d.tick_params(axis='both', labelsize=22)
        ax4d.set_xlim(0, FIG4F_XMAX)
        ax4d.xaxis.set_major_locator(
            ticker.MaxNLocator(nbins=6, steps=[1, 2, 2.5, 5, 10]))
        ax4d.set_ylabel("Number of Loaded Hauls", fontsize=FIG4F_LABEL_FS)
        ax4d.set_xlabel(f"One-way Route Distance ({DIST_UNIT})",
                        fontsize=FIG4F_LABEL_FS)
        ax4d.legend(title='Truck Type', title_fontsize=22,
                    fontsize=19, loc='upper right')
        print(f"  Figure 4d: {len(success_lines):,} routes on {f_basis}, "
              f"0 to {max_mi:,.1f} {DIST_UNIT}, "
              f"x range 0 to {FIG4F_XMAX:,.0f} {DIST_UNIT}")
        # Distance summaries quoted in the text.
        _fk, _ft = melted["route_mi"], melted["Num_Hauls"]
        print(f"      longest validated route: {max_mi:,.1f} {DIST_UNIT}; "
              f"routes over {FIG4F_LONG_CUT:,.0f} {DIST_UNIT}: "
              f"{int((success_lines['route_mi'] > FIG4F_LONG_CUT).sum()):,} of "
              f"{len(success_lines):,}")
        print(f"      truck trips: {_ft.sum():,.0f} total, "
              f"{100 * _ft[_fk < FIG4F_LONG_CUT].sum() / _ft.sum():.1f}% under "
              f"{FIG4F_LONG_CUT:,.0f} {DIST_UNIT}, "
              f"{_ft[_fk < FIG4F_BINWIDTH].sum():,.0f} in the "
              f"0-{FIG4F_BINWIDTH:,.0f} {DIST_UNIT} bin")
        for _cut in FIG4F_CUTS:
            print(f"      trips under {_cut:>3d} {DIST_UNIT}: "
                  f"{_ft[_fk < _cut].sum():>10,.0f} "
                  f"({100 * _ft[_fk < _cut].sum() / _ft.sum():5.1f}%)   "
                  f"routes: {int((success_lines['route_mi'] < _cut).sum()):>3d} "
                  f"({100 * (success_lines['route_mi'] < _cut).mean():5.1f}%)")
        add_panel_label(ax4d, 'd)', fontsize=32, x=-0.10, y=1.08)
    else:
        ax4d.text(0.5, 0.5, "route_km not found in OD_LINES_FILE",
                  ha='center', va='center', fontsize=14)
else:
    ax4d.text(0.5, 0.5, f"{OD_LINES_FILE} not found",
              ha='center', va='center', fontsize=14)

_x = np.arange(len(q_tab))

# --- 4e: exposure by income quintile, on both exposure measures -------
# Whiskers are q_tab's bootstrap percentile intervals. Each y axis extends to
# FIG5C_HEADROOM times its highest upper bound, so no whisker is clipped.
_bar_w = 0.38
CI_KW = dict(fmt="none", capsize=5, elinewidth=1.8, capthick=1.8, zorder=5)
FIG5C_HEADROOM = 1.30

def _ci_arms(lo, hi, mid):
    """Asymmetric yerr rows for errorbar, from an interval and its centre."""
    return np.vstack([np.asarray(mid, float) - np.asarray(lo, float),
                      np.asarray(hi, float) - np.asarray(mid, float)])

ax4e.bar(_x - _bar_w / 2, q_tab["mean_exposure"], _bar_w,
         color="saddlebrown", edgecolor="black", alpha=0.88,
         label=EXPOSURE_LABEL)
ax4e.errorbar(_x - _bar_w / 2, q_tab["mean_exposure"],
              yerr=_ci_arms(q_tab["ci_lo"], q_tab["ci_hi"],
                            q_tab["mean_exposure"]),
              ecolor="0.12", **CI_KW)
ax4e.set_ylabel(EXPOSURE_LABEL, fontsize=21, color="saddlebrown", labelpad=14)
ax4e.tick_params(axis="y", labelsize=17, colors="saddlebrown")
ax4e.set_ylim(0, float(q_tab["ci_hi"].max()) * FIG5C_HEADROOM)
ax4d_r = ax4e.twinx()
if ALT_EXPOSURE_COL in corr4.columns and corr4[ALT_EXPOSURE_COL].notna().any():
    ax4d_r.bar(_x + _bar_w / 2, q_tab["alt_mean"], _bar_w,
               color="black", edgecolor="black", alpha=0.88)
    ax4d_r.errorbar(_x + _bar_w / 2, q_tab["alt_mean"],
                    yerr=_ci_arms(q_tab["alt_ci_lo"], q_tab["alt_ci_hi"],
                                  q_tab["alt_mean"]),
                    ecolor="0.62", **CI_KW)
    ax4d_r.set_ylim(0, float(q_tab["alt_ci_hi"].max()) * FIG5C_HEADROOM)
ax4d_r.set_ylabel(ALT_EXPOSURE_LABEL, fontsize=21, color="black", labelpad=14)
ax4d_r.tick_params(axis="y", labelsize=17, colors="black")
ax4e.set_xticks(_x)
ax4e.set_xticklabels(q_tab["quintile"], fontsize=18)
ax4e.set_xlabel("Median household income quintile", fontsize=23, labelpad=12)
ax4e.legend(handles=[
    Patch(facecolor="saddlebrown", edgecolor="black", alpha=0.88,
          label=EXPOSURE_LABEL),
    Patch(facecolor="black", edgecolor="black", alpha=0.88,
          label=ALT_EXPOSURE_LABEL)],
    loc="upper right", fontsize=15)
add_panel_label(ax4e, "e)", fontsize=30, x=-0.10, y=1.08)

# --- 4f: exposure rate against community size, one point per place -----
# Truck-mi per resident against population for every place with truck traffic,
# coloured by CDP versus city or town.
CDP_COLOR, CITY_COLOR = "#e6550d", "#3182bd"
_pl_pts = place_exp[(place_exp["VMT_j"] > 0)
                    & (place_exp["place_pop"] > 0)].copy()
assert np.allclose(_pl_pts["vmt_per_capita"],
                   _pl_pts["VMT_j"] / _pl_pts["place_pop"], rtol=1e-9), \
    "the plotted rate is not truck-mi over population"
assert len(_pl_pts) == int((place_exp["VMT_j"] > 0).sum()), \
    "an exposed place has no population and would be dropped from the panel"
_is_cdp = _pl_pts["place_type"] == "CDP"
for _mask, _colour, _label in (
        (_is_cdp, CDP_COLOR, f"CDP ({int(_is_cdp.sum())})"),
        (~_is_cdp, CITY_COLOR, f"City or town ({int((~_is_cdp).sum())})")):
    ax4f.scatter(_pl_pts.loc[_mask, "place_pop"],
                 _pl_pts.loc[_mask, "vmt_per_capita"],
                 s=52, color=_colour, edgecolor="black", linewidth=0.5,
                 alpha=0.8, label=_label, zorder=3)
ax4f.set_xscale("log")
ax4f.set_yscale("log")

# Least-squares slope in log-log space (printed, not drawn). A slope of -1
# means total truck-mi does not vary with population.
_lx = np.log10(_pl_pts["place_pop"].to_numpy(float))
_ly = np.log10(_pl_pts["vmt_per_capita"].to_numpy(float))
_slope, _icept = np.polyfit(_lx, _ly, 1)
assert np.isfinite(_slope), "log-log fit did not converge"
# Spearman rank correlation of the same two variables, with a bootstrap CI.
_rho_5d, _p_5d = spearmanr(_pl_pts["place_pop"], _pl_pts["vmt_per_capita"])
_rho_5d_lo, _rho_5d_hi = bootstrap_spearman(
    _pl_pts["place_pop"].to_numpy(float),
    _pl_pts["vmt_per_capita"].to_numpy(float),
    n_boot=CORR_BOOTSTRAP_DRAWS)
# Spearman statistics in the lower-right corner; no fitted line is drawn.
fig4.canvas.draw()
ax4f.text(0.975, 0.045,
          f"Spearman $\\rho$ = {_rho_5d:+.3f}\n"
          f"{'p < 0.001' if _p_5d < 0.001 else f'p = {_p_5d:.3f}'}\n"
          f"n = {len(_pl_pts):,} places",
          transform=ax4f.transAxes, ha="right", va="bottom", fontsize=17,
          color="0.15", zorder=7,
          bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="0.75", lw=1.0))

# Label the top FIG5D_LABEL_N CDPs and cities on truck-mi per resident and per
# mi2; a place leading on both is labelled once.
FIG5D_LABEL_N = 2
# Places left unlabelled because their leader line would cross the scatter.
FIG5D_LABEL_SKIP = {"Escalon city"}
_lab_names = [n for n in pd.unique(pd.concat(
    [_pl_pts[_is_cdp if _cdp else ~_is_cdp].nlargest(FIG5D_LABEL_N, _m)["NAMELSAD"]
     for _m in ("vmt_per_capita", "vmt_per_mi2") for _cdp in (True, False)]))
    if n not in FIG5D_LABEL_SKIP]
_lab_rows = _pl_pts[_pl_pts["NAMELSAD"].isin(_lab_names)]
assert len(_lab_rows) == len(_lab_names), \
    "a place selected for labelling is not among the plotted points"

# Labels sit right of their point, or left beyond FIG5D_FLIP_AT of the log
# x range. They are placed from the most exposed down; one overlapping a
# placed label is stepped down until it clears. A leader line joins each label
# to its marker.
FIG5D_FLIP_AT, FIG5D_STEP_PT, FIG5D_MAX_STEPS = 0.65, 17, 12
FIG5D_LABEL_FS = 17
# Default label offsets from the marker, in points.
FIG5D_LABEL_DY = 13
FIG5D_LABEL_DX = 22
# Per-place offsets, in points, where the default would overlap the frame, a
# marker or another label.
FIG5D_LABEL_DY_BY_NAME = {"Dos Palos Y CDP": 2, "Lakeside CDP": 20,
                          "Tipton CDP": 10, "Shafter city": 20,
                          "Tulare city": 20}
FIG5D_LABEL_DX_BY_NAME = {"Tipton CDP": 48}
# Places labelled to the right regardless of x position.
FIG5D_LABEL_FORCE_RIGHT = {"Tulare city"}
_lx0, _lx1 = np.log10(_pl_pts["place_pop"].min()), np.log10(_pl_pts["place_pop"].max())
_placed = []
for _r in _lab_rows.sort_values("vmt_per_capita", ascending=False).itertuples():
    _right = ((np.log10(_r.place_pop) - _lx0) / (_lx1 - _lx0) < FIG5D_FLIP_AT
              or _r.NAMELSAD in FIG5D_LABEL_FORCE_RIGHT)
    _t = ax4f.annotate(
        _r.NAMELSAD.replace(" city", "").replace(" CDP", ""),
        (_r.place_pop, _r.vmt_per_capita), textcoords="offset points",
        xytext=(FIG5D_LABEL_DX_BY_NAME.get(_r.NAMELSAD, FIG5D_LABEL_DX)
                * (1 if _right else -1),
                FIG5D_LABEL_DY_BY_NAME.get(_r.NAMELSAD, FIG5D_LABEL_DY)),
        ha="left" if _right else "right",
        va="center", fontsize=FIG5D_LABEL_FS, color="0.15", zorder=6,
        arrowprops=dict(arrowstyle="-", color="0.35", lw=1.1,
                        shrinkA=2, shrinkB=5,
                        connectionstyle="arc3,rad=0"),
        path_effects=[path_effects.withStroke(
            linewidth=3.2, foreground="white")])
    for _ in range(FIG5D_MAX_STEPS):
        _bb = _t.get_window_extent(fig4.canvas.get_renderer())
        if not any(_bb.overlaps(_q) for _q in _placed):
            break
        _dx, _dy = _t.get_position()
        _t.set_position((_dx, _dy - FIG5D_STEP_PT))
    _placed.append(_t.get_window_extent(fig4.canvas.get_renderer()))

ax4f.set_xlabel("City / Census Place Population", fontsize=23, labelpad=12)
ax4f.set_ylabel("Truck-mi per Resident", fontsize=21, labelpad=14)
ax4f.tick_params(axis="both", labelsize=17)
ax4f.grid(alpha=0.25, which="both")
ax4f.legend(loc="lower left", fontsize=15)
_y0, _y1 = ax4f.get_ylim()
assert _y0 <= _pl_pts["vmt_per_capita"].min() \
    and _y1 >= _pl_pts["vmt_per_capita"].max(), \
    "the y-axis clips a place; the low-exposure tail must stay visible"
_pooled = float(_pl_pts["VMT_j"].sum() / _pl_pts["place_pop"].sum())
print(f"  Figure 4f: {len(_pl_pts):,} places, log-log slope {_slope:+.3f}, "
      f"pooled rate {_pooled:.3f} truck-mi per resident "
      f"({int((_pl_pts['vmt_per_capita'] > _pooled).sum())} places above it)")
print(f"    Spearman rho(population, truck-mi per resident) = {_rho_5d:+.3f}  "
      f"p={_p_5d:.3g}  95% CI [{_rho_5d_lo:+.3f}, {_rho_5d_hi:+.3f}]  "
      f"n={len(_pl_pts):,} places")
print(f"    labelled ({len(_lab_names)}): " + ", ".join(sorted(_lab_names)))
add_panel_label(ax4f, "f)", fontsize=30, x=-0.10, y=1.08)

# Rescale each panel's text to this figure, then set the a)-f) panel letters
# to 26 pt bold.
scale_axes_fonts(ax4a, 1.95)
scale_axes_fonts(ax4d, 0.74)
scale_axes_fonts(ax4e, 0.70, extra=[ax4d_r] if "ax4d_r" in dir() else ())
if "ax4d_r" in dir():
    ax4d_r.yaxis.labelpad = 5
scale_axes_fonts(ax4f, 0.70)
for _ax4 in (ax4a, ax4b, ax4d, ax4e, ax4f):
    for _t4 in _ax4.texts:
        if _t4.get_text().strip() in ("a)", "b)", "c)", "d)", "e)", "f)"):
            _t4.set_fontsize(26)
            _t4.set_fontweight("bold")

# Geopandas fixes an equal aspect, so each map is shorter than its cell; the
# colourbar slots are matched to the drawn map height.
fig4.canvas.draw()
for _mapax, _capax in ((ax4b, cax4b), (ax4c, cax4c)):
    _mp, _cp = _mapax.get_position(), _capax.get_position()
    _capax.set_position([_cp.x0, _mp.y0, _cp.width, _mp.height])

save_figure(fig4, "Figure_4_Network_And_Exposure.png", dpi=200,
            bbox_inches="tight", rasterize=(ax4a, ax4b, ax4c))
plt.close(fig4)
print("Saved Figure 4.")

print(f"  wrote {out_path('Figure_S1_Reporting_Formats.png')}")
print(f"  wrote {out_path('Figure_S4_Manure_Trucked_Per_Month.png')}")
print(f"  wrote {out_path('Table_S6_Top_Receivers.csv')}")
print(f"\n  Table S6, top 5 receivers")
print(s6.head(5)[["Rank", "Destination (normalised)", "Tons", "Manifests",
                   "Origin dairies", "Address form", "In report set"]].to_string(index=False))


In [ ]:
# =========================================================================
# MANIFEST EXTRACTION: FIGURE S10, TABLE S1 AND SI METHODS COUNTS
#   Rebuilds the manifest-extraction results from the files that
#   review/manifests/extraction/manifest_preparation.ipynb writes to
#   outputs/review/manifests/. Nothing is re-extracted and nothing in that
#   folder is written.
#
#   Figure S10 compares each automatically extracted field with its validated
#   value, record by record. Fields carried by both manifest types are scored
#   on manure records; Is Trucked and wastewater gallons on wastewater
#   records. The validated side is the hand-checked file without its
#   duplicate rows, with the columns the extraction notebook post-processes
#   (origin address backfill, tonnage and solids backfill, destination type,
#   haul dates) taken from the processed manifest files. The error count per
#   field is read from extraction_errors_by_field.csv, since matching two
#   values there uses libpostal address expansion; the counts of non-blank and
#   compared values are recomputed here.
#
#   Outputs:
#     Figure_S10_Manifest_Extraction_Accuracy.png (and .eps)
#     table_s1, s10_stats and the MANIFEST_COUNTS values, read by the next cell
# =========================================================================
print("\n" + "=" * 60)
print("MANIFEST EXTRACTION: FIGURE S10, TABLE S1, SI METHODS COUNTS")
print("=" * 60)

MANIFEST_DIR = ROOT_PATH + "outputs/review/manifests/"
MANIFEST_HELPER_DIR = ROOT_PATH + "review/manifests/helpers/"
MANIFEST_KEYS = ["Source PDF", "Manifest Number"]

_par = pd.read_csv(MANIFEST_HELPER_DIR + "parameters.csv")
MANIFEST_COL_TYPES = dict(zip(_par["parameter_name"], _par["data_type"]))
MANIFEST_TYPE_COLS = {t: set(_par.loc[_par["manifest_type"] == t, "parameter_name"])
                      for t in ("manure", "wastewater")}
DEST_TYPE_KEYWORDS = {"Broker": ["broker"],
                      "Composting Facility": ["compost", "kellogg", "hyponex", "fertilizer", "supply"],
                      "Farmer": ["farm"]}
WW_TRUCK_WORDS = ("load", "haul", "tank")


def std_dest_type(value):
    """Destination type as one or more of Broker, Composting Facility and Farmer, else Other or Blank."""
    if pd.isna(value) or not str(value).strip():
        return "Blank"
    text = str(value).lower()
    matched = [name for name, kws in DEST_TYPE_KEYWORDS.items() if any(k in text for k in kws)]
    return ", ".join(matched) if matched else "Other"


def wastewater_is_trucked(df):
    """True when the hauler name or method mentions loads, hauls or tanks (and not "apply"),
    False for pipelines, otherwise "UNSURE" (Table S3)."""
    text = (df["Hauler Name"].fillna("").astype(str) + " "
            + df["Method Used to Determine Volume of Wastewater"].fillna("").astype(str)).str.lower()
    trucked = text.apply(lambda t: any(w in t for w in WW_TRUCK_WORDS)
                         and not any(w in t for w in ("apply", "applied")))
    pipeline = df["Is Pipeline"].astype(str).str.upper().eq("TRUE")
    return pd.Series("UNSURE", index=df.index, dtype=object).mask(trucked, True).mask(pipeline, False)


def split_manifest_types(df):
    """Manure and wastewater records, each without the other type's fields, with the standardized
    destination type and Is Trucked (always True for manure; for wastewater the recorded value,
    else the keyword rule)."""
    df = df.copy()
    df["Destination Type (Standardized)"] = df["Destination Type"].apply(std_dest_type)
    man = df.loc[df["Manifest Type"].isin(["manure", "both"]),
                 [c for c in df.columns if c not in MANIFEST_TYPE_COLS["wastewater"]]].copy()
    man["Is Trucked"] = True
    ww = df.loc[df["Manifest Type"].isin(["wastewater", "both"]),
                [c for c in df.columns if c not in MANIFEST_TYPE_COLS["manure"]]].copy()
    given = ww.get("Is Trucked", pd.Series(None, index=ww.index, dtype=object)).map(
        {"TRUE": True, "FALSE": False, "UNSURE": "UNSURE", "True": True, "False": False,
         True: True, False: False})
    ww["Is Trucked"] = given.where(given.notna(), wastewater_is_trucked(ww))
    return man, ww


def coerce_manifest_types(df):
    """Numeric, date and boolean fields of parameters.csv converted to those types; unreadable
    values become blank."""
    for c in df.columns:
        t = MANIFEST_COL_TYPES.get(c)
        if t == "numeric":
            df[c] = pd.to_numeric(df[c], errors="coerce")
        elif t == "date":
            df[c] = pd.to_datetime(df[c], format="mixed", errors="coerce")
        elif t == "boolean":
            df[c] = (df[c].astype(str).str.strip().str.upper()
                     .map({"TRUE": True, "FALSE": False, "NAN": None}))
    return df


# --- Figure S10: extraction accuracy by field ---
_auto_all = pd.read_csv(MANIFEST_DIR + "all_manifests_as_written_automatic.csv", low_memory=False)
_valid_all = pd.read_csv(MANIFEST_DIR + "all_manifests_as_written_validated.csv", low_memory=False)
auto_man, auto_ww = split_manifest_types(_auto_all)
_valid = coerce_manifest_types(_valid_all.copy())
_valid = _valid[_valid["Is Duplicate"] != "x"].reset_index(drop=True)
valid_man, valid_ww = split_manifest_types(_valid)
for _v, _f in ((valid_man, "processed_manure_manifests.csv"),
               (valid_ww, "processed_wastewater_manifests.csv")):
    _p = pd.read_csv(MANIFEST_DIR + _f, low_memory=False)
    _p = _p.set_index(pd.MultiIndex.from_frame(_p[MANIFEST_KEYS].astype(str)))
    _k = pd.MultiIndex.from_frame(_v[MANIFEST_KEYS].astype(str))
    assert _k.is_unique and set(_k) == set(_p.index), f"{_f} does not match the validated records"
    for _c in _p.columns:
        if _c not in MANIFEST_KEYS and _c not in ("Start Page", "End Page"):
            _v[_c] = _p[_c].reindex(_k).to_numpy()

s10_errors = pd.read_csv(MANIFEST_DIR + "extraction_errors_by_field.csv")
_summary_rows = s10_errors["Field"].str.startswith("Manifests with")
S10_FIELDS = list(s10_errors.loc[~_summary_rows, "Field"])
_err = s10_errors.set_index("Field")["Errors Corrected"]
_rows, _compared_keys = [], []
for _vdf, _adf in ((valid_man.drop(columns="Is Trucked"), auto_man), (valid_ww, auto_ww)):
    _m = _vdf.merge(_adf, on=MANIFEST_KEYS, suffixes=("_validated", "_auto"))
    _compared_keys.append(_m[MANIFEST_KEYS])
    for _f in S10_FIELDS:
        if _f in {r["field"] for r in _rows} or _f not in _vdf.columns:
            continue
        _n = int((_m[_f + "_validated"].notna() | _m[_f + "_auto"].notna()).sum())
        _rows.append({"field": _f, "manual_count": int(_vdf[_f].notna().sum()),
                      "auto_count": int(_adf[_f].notna().sum()), "compared": _n,
                      "errors": int(_err[_f])})
s10_stats = pd.DataFrame(_rows).set_index("field").loc[S10_FIELDS].reset_index()
s10_stats["accuracy"] = 100 * (1 - s10_stats["errors"] / s10_stats["compared"])
assert s10_stats["accuracy"].is_monotonic_decreasing, "fields are not in accuracy order"
S10_N_COMPARED = len(pd.concat(_compared_keys).drop_duplicates())
S10_N_ANY_ERROR = int(_err["Manifests with at least one error"])
S10_N_IMPACT_ERROR = int(_err["Manifests with a result-impacting correction"])
# Records for which the automatic extraction found no amount are typed "unknown"
# there, so split_manifest_types leaves them out and Figure S10 does not score them.
_auto_type = _auto_all.drop_duplicates(MANIFEST_KEYS)[MANIFEST_KEYS + ["Manifest Type"]]
_man_auto_type = valid_man[MANIFEST_KEYS].merge(_auto_type, on=MANIFEST_KEYS, how="left")["Manifest Type"]
_ww_auto_type = valid_ww[MANIFEST_KEYS].merge(_auto_type, on=MANIFEST_KEYS, how="left")["Manifest Type"]
S10_MANURE_WITH_AUTO = int(_man_auto_type.notna().sum())
S10_MANURE_AMOUNT_FOUND = int(_man_auto_type.isin(["manure", "both"]).sum())
S10_MANURE_NOT_SCORED = int(_man_auto_type.eq("unknown").sum())
S10_WW_NOT_SCORED = int(_ww_auto_type.eq("unknown").sum())
_s10_amount = s10_stats.set_index("field").loc["Total Manure Amount (tons)"]
S10_MANURE_AMOUNT_ACCURACY = float(_s10_amount["accuracy"])
assert S10_MANURE_WITH_AUTO == S10_MANURE_AMOUNT_FOUND + S10_MANURE_NOT_SCORED, \
    "a validated manure record has an automatic record of another type"
assert int(_s10_amount["compared"]) == S10_MANURE_AMOUNT_FOUND, \
    "Total Manure Amount is not compared on exactly the manure records with an extracted amount"

S10_COLORS = {"manual": "#01665e", "extracted": "#bf812d", "accuracy": "#1f78b4"}
# Drawn at Figure S10's 14 pt type.
with plt.rc_context({"font.size": 14}):
    _x = np.arange(len(s10_stats))
    _bw = 0.35
    figS10, axS10 = plt.subplots(figsize=(12, 7))
    axS10b = axS10.twinx()
    _mb = axS10.bar(_x - _bw / 2, s10_stats["manual_count"], width=_bw, alpha=0.7,
                    color=S10_COLORS["manual"])
    _ab = axS10.bar(_x + _bw / 2, s10_stats["auto_count"], width=_bw, alpha=0.7,
                    color=S10_COLORS["extracted"])
    _dots = axS10b.plot(_x, s10_stats["accuracy"], "o", color=S10_COLORS["accuracy"],
                        markersize=8, zorder=5)[0]
    axS10.set_ylabel("Count of Non-Null Values")
    axS10.set_xticks(_x)
    axS10.set_xticklabels(s10_stats["field"].str.replace("(Gallons)", "(gallons)", regex=False),
                          rotation=45, ha="right")
    axS10.legend([_mb, _ab, _dots], ["Manual Count", "Automatic Count", "Accuracy (%)"],
                 bbox_to_anchor=(0.7, 1.25), ncol=3, frameon=False)
    axS10b.set_ylabel("Accuracy (%)")
    axS10b.set_ylim(0, 100)
    figS10.tight_layout()
    save_figure(figS10, "Figure_S10_Manifest_Extraction_Accuracy.png", dpi=200)
    plt.show()

print(f"\n  Figure S10: {S10_N_COMPARED:,} automatically extracted records compared")
print(f"    with at least one correction:          {S10_N_ANY_ERROR:,} "
      f"({100 * S10_N_ANY_ERROR / S10_N_COMPARED:.1f}%)")
print(f"    with a result-relevant correction:     {S10_N_IMPACT_ERROR:,} "
      f"({100 * S10_N_IMPACT_ERROR / S10_N_COMPARED:.1f}%)")
print(s10_stats.to_string(index=False, float_format="%.2f"))
print(f"    not scored (no amount extracted): {S10_MANURE_NOT_SCORED + S10_WW_NOT_SCORED} "
      f"({S10_MANURE_NOT_SCORED} manure, {S10_WW_NOT_SCORED} wastewater); manure amount extracted for "
      f"{S10_MANURE_AMOUNT_FOUND} of {S10_MANURE_WITH_AUTO} manure records, correct on "
      f"{S10_MANURE_AMOUNT_ACCURACY:.1f}% of them")

# --- Table S1: manifests by form template ---
# Shares are of the forms submitted: the analysed manifests plus the CAG018001
# copies filed for the same hauls. The descriptive columns are kept by hand in
# Manifest_Formats.csv, in the table's row order.
_tc = pd.read_csv(MANIFEST_DIR + "2024_manifest_template_counts.csv")
_tc_total = _tc[_tc["template_id"] == "Total"].iloc[0]
_tc = _tc[_tc["template_id"] != "Total"].set_index("template_id")
_mfmt = pd.read_csv(MANIFEST_DIR + "Manifest_Formats.csv", dtype=str, keep_default_na=False)
assert set(_mfmt["Template ID"]) == set(_tc.index), "Manifest_Formats.csv and the template counts list different templates"
table_s1 = pd.DataFrame({
    "Template ID": _mfmt["Template ID"],
    "Attachment Letter": _mfmt["Attachment Letter"],
    "General Order Referenced": _mfmt["General Order Referenced"],
    "Page Count": _mfmt["Page Count"],
    "Percent of Analyzed Manifests": _mfmt["Template ID"].map(_tc["analysis_pct"]),
    "File example": _mfmt["File example"],
    "forms_submitted": _mfmt["Template ID"].map(_tc["forms_submitted"]),
})
table_s1 = table_s1[table_s1["forms_submitted"] > 0].reset_index(drop=True)

# --- SI Methods 0.1.1 and Section 2.3 counts ---
# Manifest documents are identified by report and start page. Documents found
# by the keyword search carry Page Source "automatic"; those added by hand
# after screening carry "manual".
_docs = (_valid_all.drop_duplicates(["Source PDF", "Start Page"])
         .merge(_auto_all[["Source PDF", "Start Page", "Form Template"]]
                .drop_duplicates(["Source PDF", "Start Page"]),
                on=["Source PDF", "Start Page"], how="left"))
_dup = _docs["Is Duplicate"] == "x"
_proc_man = pd.read_csv(MANIFEST_DIR + "processed_manure_manifests.csv")
_rate = _proc_man["Manure Mass per Haul (tons)"]
_amt = _proc_man["Total Manure Amount (tons)"]
# Haul-size classes, as analysis/Routing.ipynb and the extraction notebook count the hauls:
# [5, 15) tons per load is a 10-ton truck, 15 tons or more a 20-ton truck, and a manifest
# under 10 tons in total is one 10-ton haul.
HAUL_BIN_LOW, HAUL_BIN_MID = 5.0, 15.0
SINGLE_HAUL_BELOW_TONS = 10.0
_in10 = _rate.between(HAUL_BIN_LOW, HAUL_BIN_MID, inclusive="left")
_in20 = _rate >= HAUL_BIN_MID
# Templates whose pages differ only in their header share a layout
# (templates.csv "layout"); the layouts counted are those of the templates
# with 2024 forms.
_tmpl = pd.read_csv(MANIFEST_HELPER_DIR + "templates.csv", dtype=str,
                    keep_default_na=False).set_index("template_key")
_used = _tc.index[_tc["forms_submitted"] > 0]
assert set(_used) <= set(_tmpl.index) and _tmpl.loc[_used, "layout"].ne("").all(), \
    "a template with 2024 forms has no layout in templates.csv"
# Transport records: the non-duplicate rows of the validated table. A row's
# Manifest Type is manure, wastewater, both, or unknown (no amount of either).
_records = _valid_all[_valid_all["Is Duplicate"] != "x"]
MANIFEST_COUNTS = {
    "reports_received": len(pd.read_csv(MANIFEST_DIR + "2024_files_by_template.csv")),
    "docs_automatic": int((_docs["Page Source"] == "automatic").sum()),
    "docs_manual": int((_docs["Page Source"] == "manual").sum()),
    "docs_manual_reports": int(_docs.loc[_docs["Page Source"] == "manual", "Source PDF"].nunique()),
    "docs_screened": len(_docs),
    "reports_with_manifests": int(_docs.loc[~_dup, "Source PDF"].nunique()),
    "records": len(_records),
    "records_both": int(_records["Manifest Type"].eq("both").sum()),
    "records_no_type": int(_records["Manifest Type"].eq("unknown").sum()),
    "docs_duplicate": int(_dup.sum()),
    "docs_duplicate_cag": int((_dup & _docs["Form Template"].eq("CAG018001")).sum()),
    "docs_duplicate_exact": int((_dup & ~_docs["Form Template"].eq("CAG018001")).sum()),
    "docs_analysed": int(_tc_total["analysis_manifests"]),
    "forms_submitted": int(_tc_total["forms_submitted"]),
    "templates_used": int((_tc["forms_submitted"] > 0).sum()),
    "layouts_used": int(_tmpl.loc[_used, "layout"].nunique()),
    "pct_r5_2013_0122": float(_tc.loc["R5-2013-0122 General Order", "analysis_pct"]),
    "share_10_ton_bin": float(_amt[_in10].sum() / (_amt[_in10].sum() + _amt[_in20].sum())),
    "hauls_10_ton_manifest": int(_proc_man["Number of 10-ton Hauls for Analysis"].sum()),
    "hauls_20_ton_manifest": int(_proc_man["Number of 20-ton Hauls for Analysis"].sum()),
}
assert MANIFEST_COUNTS["docs_analysed"] == len(_docs) - MANIFEST_COUNTS["docs_duplicate"]
assert MANIFEST_COUNTS["docs_screened"] == MANIFEST_COUNTS["docs_automatic"] + MANIFEST_COUNTS["docs_manual"]
assert MANIFEST_COUNTS["forms_submitted"] - MANIFEST_COUNTS["docs_analysed"] == MANIFEST_COUNTS["docs_duplicate_cag"]

print(f"\n  Table S1: {len(table_s1)} templates in use on {MANIFEST_COUNTS['layouts_used']} page layouts; "
      f"{MANIFEST_COUNTS['docs_analysed']:,} manifests "
      f"analysed, {MANIFEST_COUNTS['forms_submitted']:,} forms submitted")
print(f"  reports received: {MANIFEST_COUNTS['reports_received']:,}")
print(f"  manifest documents: {MANIFEST_COUNTS['docs_automatic']:,} found by the keyword search, "
      f"{MANIFEST_COUNTS['docs_manual']} added by hand from {MANIFEST_COUNTS['docs_manual_reports']} reports")
print(f"  manifests after screening: {MANIFEST_COUNTS['docs_screened']:,}; analysed manifests come from "
      f"{MANIFEST_COUNTS['reports_with_manifests']} reports and hold {MANIFEST_COUNTS['records']:,} transport records "
      f"({MANIFEST_COUNTS['records_both']} with both manure and wastewater, "
      f"{MANIFEST_COUNTS['records_no_type']} with neither)")
print(f"  duplicates removed: {MANIFEST_COUNTS['docs_duplicate']} "
      f"({MANIFEST_COUNTS['docs_duplicate_exact']} exact copies, "
      f"{MANIFEST_COUNTS['docs_duplicate_cag']} CAG018001 forms)")
print(f"  10-ton bin share of tonnage on manifests reporting a haul size: "
      f"{100 * MANIFEST_COUNTS['share_10_ton_bin']:.1f}% ({int(_in10.sum())} manifests in [5, 15) t, "
      f"{int(_in20.sum())} at 15 t or more)")
print(f"  haul estimates in the processed manure file: {MANIFEST_COUNTS['hauls_10_ton_manifest']:,} "
      f"10-ton, {MANIFEST_COUNTS['hauls_20_ton_manifest']:,} 20-ton")

In [ ]:
# =========================================================================
# MANUSCRIPT PARAMETERS AND SUPPLEMENTARY TABLES (LaTeX)
#   Every number the manuscript quotes from these results, and every analysis
#   setting it quotes, becomes one LaTeX macro, formatted as the manuscript
#   prints it. Units and percent signs stay in the manuscript text, a minus
#   sign is \ensuremath{-}, and a p-value macro carries its relation
#   ("\ensuremath{{}\leq 0.001}" or "\ensuremath{{}=0.007}"). Values that no
#   file records are entered here as literals and marked "literal".
#
#   The SI tables are written in the manuscript's longtable layout. Tables S2
#   to S4 describe the manifest fields and keyword rules, so their text is set
#   below (SI_TABLES_S2_S4) and written out as its own file as well.
#
#   Outputs:
#     outputs/results/Manuscript_Parameters.tex   (\input in the preamble)
#     outputs/results/Manuscript_Parameters.csv   (name, value, source)
#     outputs/results/tables/SI_Tables.md
#     outputs/results/tables/SI_Tables_S2_S4.tex
# =========================================================================
print("\n" + "=" * 60)
print("MANUSCRIPT PARAMETERS AND SUPPLEMENTARY TABLES")
print("=" * 60)

RESULTS_DIR = ROOT_PATH + "outputs/results"
PARAMS = {}
_section = ""


def fmt_num(x, dp=0):
    """Thousands separators and dp decimals; a negative value gets \\ensuremath{-}."""
    v = round(float(x), dp)
    s = f"{abs(v):,.{dp}f}"
    return "\\ensuremath{-}" + s if v < 0 else s


def fmt_round(x, step):
    """x to the nearest multiple of step, with thousands separators."""
    return f"{round(float(x) / step) * step:,.0f}"


def fmt_p(p):
    """p-value with its relation: <= 0.001 below 0.001, else = to three decimals."""
    return ("\\ensuremath{{}\\leq 0.001}" if p < 0.001
            else "\\ensuremath{{}=" + f"{p:.3f}" + "}")


def param(name, text, raw, desc, source, kind="result"):
    """Register one macro. kind is result, setting or literal."""
    assert re.fullmatch(r"[A-Za-z]+", name), f"macro name {name!r} must be letters only"
    assert name not in PARAMS, f"macro {name} registered twice"
    PARAMS[name] = {"value": str(text), "raw": raw, "description": desc,
                    "source": source, "kind": kind, "section": _section}


def place_name(p):
    """Census place name without its " city", " town" or " CDP" suffix."""
    return re.sub(r" (city|town|CDP)$", "", str(p))


_t8 = pd.read_csv(out_path("Table_S8_Place_VMT_Total.csv"))
_t9 = pd.read_csv(out_path("Table_S9_Place_VMT_Per_Capita.csv"))
_t10 = pd.read_csv(out_path("Table_S10_Place_VMT_Per_Area.csv"))
_proc_ww = pd.read_csv(MANIFEST_DIR + "processed_wastewater_manifests.csv")
_bal = balance_cont.set_index(["variable", "group"])
_mi_total = float(t10 + t20)
_tons_validated = float(fail.loc[fail["_reason"] == "(validated)", "_tons"].sum())
_wf = dict(WATERFALL)
_n_reports = len(rep_all)
_scraped = rep_all["Scraped August 2026"].astype(bool)
_ww_dest = _wd / _wd.sum() * 100
_s9_lb, _s9_frac = _s9_stats[0], _s9_stats[1]
_q1, _q5 = q_tab.iloc[0], q_tab.iloc[-1]
_corr = corr_tab.set_index(["exposure_col", "income_measure", "sample"])
_c_res = _corr.loc[(EXPOSURE_COL, "median household", "all BGs")]
_c_area = _corr.loc[(ALT_EXPOSURE_COL, "median household", "all BGs")]
_i05 = min(int(np.searchsorted(_cum_pop, 0.05)), len(_cum_vmt) - 1)
_nb_tot = {k: sum(s[1] for s in v) for k, v in nbasis.items()}
_emfac_ver = re.search(r"v(\d+(?:\.\d+)+)", EMFAC_META["emfac_version"]).group(1)

# --- Abstract and Introduction ---
_section = "Abstract and Introduction"
param("NReportsAnalyzed", f"{_n_reports:,}", _n_reports, "dairy annual reports in the analysis sample",
      "cell 3, len(rep_all)")
param("UnaccNMillionLb", fmt_num(-_wf["Unaccounted\n-for"] / 1e6, 1), -_wf["Unaccounted\n-for"],
      "unaccounted-for N over the Figure 3a dairies, as in Section 3.2 (million lb)",
      "cell 3, WATERFALL")
param("TruckMilesMillion", fmt_num(_mi_total / 1e6, 2), _mi_total,
      "validated truck-miles including return trips (million)", "cell 0, t10 + t20")
param("DieselGalRounded", fmt_round(_gal_tot, 1000), float(_gal_tot),
      "diesel burned, nearest thousand gallons", "cell 0, _gal_tot")
param("NTransportRecords", f"{len(manure_df) + len(ww_df):,}", len(manure_df) + len(ww_df),
      "manure plus wastewater transport records", "cell 0, len(manure_df) + len(ww_df)")

# --- Methods ---
_section = "Methods"
param("NReportFormats", f"{len(_rf)}", len(_rf), "annual report form types (Figure S1a)", "cell 5, len(_rf)")
param("NManifestTemplates", f"{MANIFEST_COUNTS['templates_used']}", MANIFEST_COUNTS["templates_used"],
      "manifest templates with 2024 forms (Table S1)", "cell 6, template counts")
param("NManifestLayouts", f"{MANIFEST_COUNTS['layouts_used']}", MANIFEST_COUNTS["layouts_used"],
      "distinct page layouts of the manifest templates with 2024 forms (Table S1)",
      "cell 6, templates.csv layout")
param("NReportsReceived", f"{MANIFEST_COUNTS['reports_received']:,}", MANIFEST_COUNTS["reports_received"],
      "annual reports received", "cell 6, rows of 2024_files_by_template.csv")
_n_rep_transport = pd.concat([_proc_man["Source PDF"], _proc_ww["Source PDF"]]).nunique()
param("NReportsWithTransport", f"{_n_rep_transport:,}", _n_rep_transport,
      "reports with at least one transport record", "cell 7, Source PDFs of the processed manifests")
param("NManureRecords", f"{len(manure_df):,}", len(manure_df), "manure transport records", "cell 0, len(manure_df)")
param("NWastewaterRecords", f"{len(ww_df):,}", len(ww_df), "wastewater transport records", "cell 0, len(ww_df)")
param("NReportsWithManifests", f"{MANIFEST_COUNTS['reports_with_manifests']:,}", MANIFEST_COUNTS["reports_with_manifests"],
      "reports with at least one analysed manifest", "cell 6, reports_with_manifests")
param("NRecordsBothTypes", f"{MANIFEST_COUNTS['records_both']}", MANIFEST_COUNTS["records_both"],
      "transport records counted as both manure and wastewater", "cell 6, records_both")
param("NRecordsNoType", f"{MANIFEST_COUNTS['records_no_type']}", MANIFEST_COUNTS["records_no_type"],
      "transport records with neither a manure nor a wastewater amount", "cell 6, records_no_type")
# The manure and wastewater records together are the transport records, with the
# records of both types counted twice and those of neither type not at all.
assert (MANIFEST_COUNTS["records"] == len(manure_df) + len(ww_df)
        - MANIFEST_COUNTS["records_both"] + MANIFEST_COUNTS["records_no_type"]), \
    "the manure and wastewater records do not reconcile with the validated table"
param("NReportsRobertson", f"{int((~_scraped).sum()):,}", int((~_scraped).sum()),
      "reports parsed by Robertson et al.", "cell 3, rep_all 'Scraped August 2026' False")
param("NReportsScraped", f"{int(_scraped.sum()):,}", int(_scraped.sum()),
      "reports parsed in this study", "cell 3, rep_all 'Scraped August 2026' True")
_n_liv = int(rep_all["Source Form Type"].eq(LIVINGSTON_FORM).sum())
param("NLivingstonReports", f"{_n_liv}", _n_liv, "reports on the Livingston Dairy Consulting form",
      "cell 3, rep_all 'Source Form Type'")
_n_liv_drop = int(next(r[2] for r in sample_log if r[0].startswith("Dropped: Livingston")))
param("NLivingstonExcluded", f"{_n_liv_drop}", _n_liv_drop,
      "Livingston reports dropped from the Figure 5 sample (the others have no herd)", "cell 3, sample_log")
param("NDairiesNitrogenTotals", f"{len(_kept_n):,}", len(_kept_n),
      "complete reports in the unaccounted-for total (Livingston forms left out)", "cell 3, len(_kept_n)")
param("NDairiesWaterfall", f"{WATERFALL_N_DAIRIES:,}", WATERFALL_N_DAIRIES,
      "dairies with all seven nitrogen fields (Figure 3a)", "cell 3, WATERFALL_N_DAIRIES")
param("NNitrogenCategories", f"{len(_wf_cols)}", len(_wf_cols), "nitrogen accounting fields required (Figure 3a)",
      "cell 3, _wf_cols", kind="setting")
param("HaulsTenTonManifest", f"{MANIFEST_COUNTS['hauls_10_ton_manifest']:,}", MANIFEST_COUNTS["hauls_10_ton_manifest"],
      "10-ton hauls in the processed manure manifests", "cell 6, processed_manure_manifests.csv")
param("HaulsTwentyTonManifest", f"{MANIFEST_COUNTS['hauls_20_ton_manifest']:,}", MANIFEST_COUNTS["hauls_20_ton_manifest"],
      "20-ton hauls in the processed manure manifests", "cell 6, processed_manure_manifests.csv")
_h10r = float(pd.to_numeric(manure_df["Number of 10-ton Hauls for Analysis"], errors="coerce").sum())
_h20r = float(pd.to_numeric(manure_df["Number of 20-ton Hauls for Analysis"], errors="coerce").sum())
param("HaulsTenTonRouting", fmt_num(_h10r), _h10r, "10-ton hauls as recomputed by Routing.ipynb (behind the truck-miles)",
      "cell 0, manure_df")
param("HaulsTwentyTonRouting", fmt_num(_h20r), _h20r, "20-ton hauls as recomputed by Routing.ipynb (behind the truck-miles)",
      "cell 0, manure_df")
param("NetworkBufferKm", "25", 25, "road network buffer around the Water Board (km)",
      "literal; Routing.ipynb buffer_km", kind="literal")
param("RoadBufferM", f"{BUFFER_DISTANCE_M}", BUFFER_DISTANCE_M, "near-road exposure buffer (m)",
      "cell 0, BUFFER_DISTANCE_M", kind="setting")
param("NBlockGroups", f"{len(corr4):,}", len(corr4), "block groups with income statistics", "cell 2, len(corr4)")

# --- Figure 2 ---
_section = "Figure 2 and Section 3.1"
for _nm, _key in (("Farmer", "Farmer"), ("Compost", "Composting Facility"), ("Broker", "Broker"),
                  ("Other", "Other"), ("Blank", "Blank")):
    param(f"PctDest{_nm}", fmt_num(by_dest.loc[_key, "pct_tons"], 1), float(by_dest.loc[_key, "pct_tons"]),
          f"manure tonnage share, destination type {_key}", "cell 5, by_dest['pct_tons']")
param("PctAddrStreet", fmt_num(by_trace.loc["Street address", "pct_tons"], 1),
      float(by_trace.loc["Street address", "pct_tons"]), "manure tonnage share to a street address",
      "cell 5, by_trace['pct_tons']")
param("PctAddrIncomplete", fmt_num(100 * _unplaceable / T6_TONS, 1), float(100 * _unplaceable / T6_TONS),
      "manure tonnage share with no or an incomplete address", "cell 5, _unplaceable / T6_TONS")
param("IncompleteAddrTons", fmt_num(_unplaceable), float(_unplaceable),
      "manure tons with no or an incomplete address", "cell 5, _unplaceable")
param("PctWwDestFarmer", fmt_num(_ww_dest["Farmer"], 1), float(_ww_dest["Farmer"]),
      "wastewater gallons share to farmers", "cell 5, _wd")
param("PctWwDestBlank", fmt_num(_ww_dest["Blank"], 1), float(_ww_dest["Blank"]),
      "wastewater gallons share with no destination type", "cell 5, _wd")
param("PctWwModeUnsure", fmt_num(_unsure_pct, 1), float(_unsure_pct), "wastewater gallons, transport mode unknown",
      "cell 5, _unsure_pct")
param("PctWwModePipeline", fmt_num(_pipe_pct, 1), float(_pipe_pct), "wastewater gallons by pipeline", "cell 5, _pipe_pct")
param("PctWwModeTruck", fmt_num(_truck_pct, 1), float(_truck_pct), "wastewater gallons by truck", "cell 5, _truck_pct")
param("NDairiesComplete", f"{len(_kept_all):,}", len(_kept_all),
      "annual reports in the herd, manure and wastewater totals", "cell 3, len(_kept_all)")
param("HerdTotal", fmt_num(_kept_all["Total Herd Size"].astype(float).sum()),
      float(_kept_all["Total Herd Size"].astype(float).sum()),
      "total herd size of those reports (head)", "cell 3, _kept_all['Total Herd Size'].sum()")
_n_ww_stated = int(_num(_kept_all["Total Process Wastewater Generated (gals)"]).notna().sum())
param("NDairiesWwStated", f"{_n_ww_stated:,}", _n_ww_stated,
      "of those reports, the ones stating process wastewater generated", "cell 3, _kept_all")
_man_gen = float(_num(_kept_all["Total Manure Excreted (tons)"]).sum())
_ww_gen = float(_num(_kept_all["Total Process Wastewater Generated (gals)"]).sum())
param("ManureGenMillionTons", fmt_num(_man_gen / 1e6, 1), _man_gen, "fresh manure generated (million tons)",
      "cell 3, _kept_all['Total Manure Excreted (tons)'].sum()")
param("WwGenBillionGal", fmt_num(_ww_gen / 1e9, 1), _ww_gen, "process wastewater generated (billion gallons)",
      "cell 3, _kept_all['Total Process Wastewater Generated (gals)'].sum()")
param("ManureExportTons", fmt_num(T6_TONS), float(T6_TONS), "manifested manure exports (tons)", "cell 5, T6_TONS")
param("WwExportMillionGal", fmt_num(WW_TOTAL / 1e6), float(WW_TOTAL), "manifested wastewater exports (million gallons)",
      "cell 5, WW_TOTAL")
_n_std = int(_rf.get("Standard form", 0))
param("NReportsStandard", f"{_n_std:,}", _n_std, "reports on the standard Water Board form", "cell 5, _rf")
param("PctReportsStandard", fmt_num(100 * _n_std / _rf.sum()), 100 * _n_std / float(_rf.sum()),
      "share of reports on the standard form", "cell 5, _rf")
param("NReportsConsultant", f"{int(_rf.sum()) - _n_std:,}", int(_rf.sum()) - _n_std,
      "reports on a consultant form (including the 2 hybrids)", "cell 5, _rf")
_n_ias_liv = int(_rf.get("Innovative Ag Services form", 0) + _rf.get("Livingston Dairy Consulting form", 0))
param("NReportsIasLivingston", f"{_n_ias_liv:,}", _n_ias_liv,
      "reports wholly on the Innovative Ag Services or Livingston form", "cell 5, _rf")
param("PctManifestsActiveOrder", fmt_num(MANIFEST_COUNTS["pct_r5_2013_0122"], 1), MANIFEST_COUNTS["pct_r5_2013_0122"],
      "analysed manifests on the R5-2013-0122 form", "cell 6, template counts analysis_pct")
_s6_po = s6.iloc[1]
param("PoBoxReceiverTons", fmt_num(_s6_po["Tons"]), float(_s6_po["Tons"]),
      f"tons to the second-largest receiver ({_s6_po['Destination (normalised)']})", "cell 5, s6 rank 2")
param("PoBoxReceiverRecords", f"{int(_s6_po['Manifests'])}", int(_s6_po["Manifests"]),
      "records to the second-largest receiver", "cell 5, s6 rank 2")
param("PctPoBoxReceiver", fmt_num(_s6_po["% of tonnage"], 2), float(_s6_po["% of tonnage"]),
      "second-largest receiver's share of exports with an address", "cell 5, s6 rank 2")

# --- Section 3.2 ---
_section = "Figure 3 and Section 3.2"
param("NitrogenGeneratedTons", fmt_round(_wf["Manure"] / 2000, 1000), _wf["Manure"] / 2000,
      "dry manure N generated (tons, nearest thousand; Figure 3a)", "cell 3, WATERFALL")
param("NitrogenVolatilizedTons", fmt_round(-_wf["Ammonia\nloss"] / 2000, 1000), -_wf["Ammonia\nloss"] / 2000,
      "N lost as ammonia (tons, nearest thousand)", "cell 3, WATERFALL")
_applied_t = -(_wf["Applied\ndry manure"] + _wf["Applied\nwastewater"]) / 2000
param("NitrogenAppliedTons", fmt_round(_applied_t, 1000), _applied_t,
      "N applied to dairy fields, manure plus wastewater (tons, nearest thousand)", "cell 3, WATERFALL")
param("NitrogenExportedTons", fmt_round(-_wf["Exported"] / 2000, 1000), -_wf["Exported"] / 2000,
      "N exported (tons, nearest thousand)", "cell 3, WATERFALL")
param("NitrogenUnaccountedTons", fmt_round(-_wf["Unaccounted\n-for"] / 2000, 1000), -_wf["Unaccounted\n-for"] / 2000,
      "N unaccounted-for (tons, nearest thousand)", "cell 3, WATERFALL")
param("NDairiesOverApplied", f"{WATERFALL_NEG_N:,}", WATERFALL_NEG_N,
      "dairies applying and exporting more N than available", "cell 3, WATERFALL_NEG_N")
param("RatioWindow", f"{RATIO_WINDOW}", RATIO_WINDOW, "moving-median window (dairies)", "cell 3, RATIO_WINDOW",
      kind="setting")
param("BootDraws", f"{RATIO_BOOT_DRAWS:,}", RATIO_BOOT_DRAWS, "bootstrap draws for the rho intervals",
      "cell 3, RATIO_BOOT_DRAWS", kind="setting")
param("MedianUnaccPct", fmt_num(100 * _s9["f_unacc"].median(), 1), float(_s9["f_unacc"].median()),
      "median dairy's unaccounted-for share of available N", "cell 5, _s9['f_unacc'].median()")
param("RhoSnineFracHerd", fmt_num(_s9_frac[4], 3), float(_s9_frac[4]),
      "Spearman rho, unaccounted-for N fraction vs animal units (Figure S9d)", "cell 5, _s9_stats")
param("pSnineFracHerd", fmt_p(_s9_frac[5]), float(_s9_frac[5]), "p for RhoSnineFracHerd", "cell 5, _s9_stats")
param("RhoSnineLbHerd", fmt_num(_s9_lb[4], 3), float(_s9_lb[4]),
      "Spearman rho, unaccounted-for N (lb) vs animal units (Figure S9c)", "cell 5, _s9_stats")
param("pSnineLbHerd", fmt_p(_s9_lb[5]), float(_s9_lb[5]), "p for RhoSnineLbHerd", "cell 5, _s9_stats")
param("NDairiesPressure", f"{len(_rp3):,}", len(_rp3), "dairies with a ratio and crop removal above zero",
      "cell 5, len(_rp3)")
param("MedianPressure", fmt_num(_rp3["pressure"].median(), 1), float(_rp3["pressure"].median()),
      "median nitrogen pressure", "cell 5, _rp3['pressure'].median()")
for _nm, _key, _d in (("RhoPressureExported", ("pressure", "f_exported"),
                       "nitrogen pressure vs exported N fraction (Figure 3d)"),
                      ("RhoPressureUnacc", ("pressure", "f_unacc"),
                       "nitrogen pressure vs unaccounted-for N fraction (Figure 3d)"),
                      ("RhoRatioUnacc", ("ratio", "f_unacc"),
                       "applied-to-removed ratio vs unaccounted-for N fraction (Figure 3c)")):
    _rr = RHO_FIG3[_key]
    param(_nm, fmt_num(_rr[0], 3), float(_rr[0]), "Spearman rho, " + _d, "cell 5, RHO_FIG3")
    param("p" + _nm[3:], fmt_p(_rr[4]), float(_rr[4]), f"p for {_nm}", "cell 5, RHO_FIG3")
param("NDairiesRatioAboveThr", f"{int((_rt['ratio'] > RATIO_THR).sum())}", int((_rt["ratio"] > RATIO_THR).sum()),
      "dairies reporting a ratio above 1.4", "cell 5, _rt")
param("RatioBandLow", fmt_num(RATIO_BAND[0], 1), RATIO_BAND[0], "lower edge of the ratio band", "cell 5, RATIO_BAND",
      kind="setting")
param("RatioBandHigh", fmt_num(RATIO_BAND[1], 1), RATIO_BAND[1], "upper edge of the ratio band", "cell 5, RATIO_BAND",
      kind="setting")
param("MedianUnaccFracBand", fmt_num(_ra.median(), 3), float(_ra.median()),
      "median unaccounted-for N fraction, dairies in the ratio band", "cell 5, _ra")
param("MedianUnaccFracRest", fmt_num(_rb.median(), 3), float(_rb.median()),
      "median unaccounted-for N fraction, other dairies", "cell 5, _rb")
param("pRatioBand", fmt_p(_rp), float(_rp), "Mann-Whitney p, band vs rest", "cell 5, _rp")
param("RankBiserialBand", fmt_num(_rbc, 3), float(_rbc), "rank-biserial correlation, band vs rest", "cell 5, _rbc")
param("NCounterfactualAbove", f"{int((_wi_ratio > RATIO_THR).sum())}", int((_wi_ratio > RATIO_THR).sum()),
      "dairies above 1.4 if unaccounted-for N were applied", "cell 5, _wi_ratio")
param("NCounterfactualDairies", f"{len(_wi):,}", len(_wi), "dairies with a reported crop removal (Figure S5)",
      "cell 5, len(_wi)")
param("PctCounterfactualAbove", fmt_num(100 * (_wi_ratio > RATIO_THR).mean()), float((_wi_ratio > RATIO_THR).mean()),
      "share of those dairies above 1.4", "cell 5, _wi_ratio")
param("MedianCounterfactualRatio", fmt_num(_wi_ratio.median(), 2), float(_wi_ratio.median()),
      "median counterfactual applied-to-removed ratio", "cell 5, _wi_ratio.median()")

# --- Section 3.3 ---
_section = "Section 3.3 and Figure 4"
param("NRoutedValidated", f"{m_final_successfully_routed:,}", m_final_successfully_routed,
      "manure records routed and validated", "cell 0, m_final_successfully_routed")
_pct_routed = 100 * m_final_successfully_routed / len(manure_df)
param("PctRoutedValidated", fmt_num(_pct_routed, 1), _pct_routed, "share of manure records routed and validated",
      "cell 0")
_pct_tons_routed = 100 * _tons_validated / float(tons_total)
param("PctTonsRoutedValidated", fmt_num(_pct_tons_routed, 1), _pct_tons_routed,
      "share of manifest tonnage on validated routes", "cell 0, fail")
param("PctRoutesExcluded", fmt_num(100 - round(_pct_routed, 1), 1), 100 - _pct_routed,
      "share of manure records excluded from routing", "cell 0")
param("PctTonsExcluded", fmt_num(100 - round(_pct_tons_routed, 1), 1), 100 - _pct_tons_routed,
      "share of manifest tonnage excluded from routing", "cell 0, fail")
param("TruckMilesTotal", fmt_num(_mi_total), _mi_total, "validated truck-miles including return trips",
      "cell 0, t10 + t20")
param("TruckMilesTenTon", fmt_num(t10), float(t10), "truck-miles by 10-ton trucks", "cell 0, t10")
param("TruckMilesTwentyTon", fmt_num(t20), float(t20), "truck-miles by 20-ton trucks", "cell 0, t20")
param("LoadedMilesOneWay", fmt_num(loaded_mi), float(loaded_mi), "validated one-way loaded miles", "cell 0, loaded_mi")
param("PMKg", fmt_num(tot_pm / 1000, 1), float(tot_pm), "truck PM2.5 emissions (kg)", "cell 0, tot_pm")
param("NOxKg", fmt_num(tot_nox / 1000, 1), float(tot_nox), "truck NOx emissions (kg)", "cell 0, tot_nox")
param("DieselGal", fmt_num(_gal_tot), float(_gal_tot), "diesel burned (gallons)", "cell 0, _gal_tot")
param("COtwoTons", fmt_num(_co2_tot), float(_co2_tot), "CO2 emitted (metric tons)", "cell 0, _co2_tot")
_trip_cut = 25
param("TripCutMi", f"{_trip_cut}", _trip_cut, "trip-length cut quoted in Section 3.3 (miles)", "cell 5, FIG4F_CUTS",
      kind="setting")
_pct_short = 100 * _ft[_fk < _trip_cut].sum() / _ft.sum()
param("PctTripsUnderCut", fmt_num(_pct_short, 1), float(_pct_short), "share of trips shorter than the cut",
      "cell 5, melted")
param("LongestRouteMi", fmt_num(max_mi, 1), float(max_mi), "longest validated one-way route (miles)", "cell 5, max_mi")
param("MeanRouteMi", fmt_num(success_lines["route_mi"].mean(), 1), float(success_lines["route_mi"].mean()),
      "mean one-way length of the validated routes (miles)", "cell 5, success_lines['route_mi'].mean()")
param("PctTruckMiTopRoutes", fmt_num(100 * TOP_ROUTES_TRUCK_MI_SHARE, 1), float(TOP_ROUTES_TRUCK_MI_SHARE),
      "ten longest routes' share of truck-miles", "cell 0, TOP_ROUTES_TRUCK_MI_SHARE")
param("PctTonsTopRoutes", fmt_num(100 * TOP_ROUTES_TON_SHARE, 1), float(TOP_ROUTES_TON_SHARE),
      "ten longest routes' share of routed tons", "cell 0, TOP_ROUTES_TON_SHARE")
for _nm, _var, _grp in (("DroppedTons", "Total manure (tons)", "Dropped: ALL"),
                        ("FailedGeocodeTons", "Total manure (tons)", "Dropped: Failed geocode"),
                        ("BadDestTons", "Total manure (tons)", "Dropped: Bad Destination Geocode"),
                        ("FailedGeocodeHerd", "Herd size (animal units)", "Dropped: Failed geocode"),
                        ("BadDestHerd", "Herd size (animal units)", "Dropped: Bad Destination Geocode"),
                        ("DroppedHerd", "Herd size (animal units)", "Dropped: ALL")):
    _row = _bal.loc[(_var, _grp)]
    param("p" + _nm, fmt_p(_row["p_value"]), float(_row["p_value"]), f"Mann-Whitney p, {_grp} vs routed, {_var}",
          "cell 1, balance_cont")
    param("r" + _nm, fmt_num(_row["rank_biserial"], 2), float(_row["rank_biserial"]),
          f"rank-biserial r, {_grp} vs routed, {_var}", "cell 1, balance_cont")
param("RhoIncomePerRes", fmt_num(_c_res["rho"], 3), float(_c_res["rho"]),
      "Spearman rho, truck-miles per resident vs median household income", "cell 2, corr_tab")
param("pIncomePerRes", fmt_p(_c_res["p_value"]), float(_c_res["p_value"]), "p for RhoIncomePerRes", "cell 2, corr_tab")
param("RhoIncomePerArea", fmt_num(_c_area["rho"], 3), float(_c_area["rho"]),
      "Spearman rho, truck-miles per mi2 vs median household income", "cell 2, corr_tab")
param("pIncomePerArea", fmt_p(_c_area["p_value"]), float(_c_area["p_value"]), "p for RhoIncomePerArea",
      "cell 2, corr_tab")
param("TruckMiPerPersonQOne", fmt_num(_q1["mean_exposure"], 2), float(_q1["mean_exposure"]),
      "truck-miles per person, lowest income quintile", "cell 2, q_tab")
param("TruckMiPerPersonQFive", fmt_num(_q5["mean_exposure"], 2), float(_q5["mean_exposure"]),
      "truck-miles per person, highest income quintile", "cell 2, q_tab")
for _rk, _wd_rank in ((0, "First"), (1, "Second"), (2, "Third")):
    _r8 = _t8.iloc[_rk]
    param(f"PlaceTotal{_wd_rank}Name", place_name(_r8["Place"]), _r8["Place"],
          f"place ranked {_rk + 1} by total truck-miles (Table S8)", "Table S8")
    param(f"PlaceTotal{_wd_rank}Mi", fmt_num(_r8["Truck-mi"]), float(_r8["Truck-mi"]),
          f"truck-miles of that place", "Table S8")
param("PlaceTopN", f"{PLACE_TOP_N}", PLACE_TOP_N, "communities listed in Tables S8-S10", "cell 0, PLACE_TOP_N",
      kind="setting")
param("NTopPerResCDP", f"{int((_t9['Type'] == 'CDP').sum())}", int((_t9["Type"] == "CDP").sum()),
      "CDPs among the top communities per resident", "Table S9")
param("PlacePerResFirstName", place_name(_t9.iloc[0]["Place"]), _t9.iloc[0]["Place"],
      "place ranked 1 per resident (Table S9)", "Table S9")
param("PlacePerResFirstMi", fmt_num(_t9.iloc[0]["Truck-mi per capita"], 2), float(_t9.iloc[0]["Truck-mi per capita"]),
      "truck-miles per resident of that place", "Table S9")
param("NTopPerAreaCDP", f"{int((_t10['Type'] == 'CDP').sum())}", int((_t10["Type"] == "CDP").sum()),
      "CDPs among the top communities per mi2", "Table S10")
param("PlacePerAreaFirstName", place_name(_t10.iloc[0]["Place"]), _t10.iloc[0]["Place"],
      "place ranked 1 per mi2 (Table S10)", "Table S10")
param("PlacePerAreaFirstMi", fmt_num(_t10.iloc[0]["Truck-mi per mi2"]), float(_t10.iloc[0]["Truck-mi per mi2"]),
      "truck-miles per mi2 of that place", "Table S10")
param("TopResidentsPct", "5", 5, "most-exposed share of residents (percent)", "cell 2, quantiles printed",
      kind="setting")
param("PctVmtTopResidents", fmt_num(100 * _cum_vmt[_i05], 1), float(_cum_vmt[_i05]),
      "in-place truck-miles carried by the most-exposed residents", "cell 2, _cum_vmt")
param("NPlacesTopResidents", f"{_i05 + 1}", _i05 + 1,
      "most-exposed places, by truck-miles per resident, that first hold 5% of residents", "cell 2, _cum_pop")
param("PctResidentsTopPlaces", fmt_num(100 * _cum_pop[_i05], 1), float(_cum_pop[_i05]),
      "share of exposed-place residents living in those places", "cell 2, _cum_pop")
param("NDairiesMapped", f"{len(_sized):,}", len(_sized), "dairies drawn in Figure 4a", "cell 5, len(_sized)")
param("RhoPlacePopExposure", fmt_num(_rho_5d, 3), float(_rho_5d),
      "Spearman rho, place population vs truck-miles per resident (Figure 4f)", "cell 5, _rho_5d")
param("pPlacePopExposure", fmt_p(_p_5d), float(_p_5d), "p for RhoPlacePopExposure", "cell 5, _p_5d")
param("NPlacesExposed", f"{len(_pl_pts):,}", len(_pl_pts), "places with truck-miles (Figure 4f)", "cell 5, len(_pl_pts)")

# --- Section 3.4 and Figure 5 ---
_section = "Section 3.4 and Figure 5"
param("PctLandApplication", fmt_num(_nb_tot["Land Application"], 1), _nb_tot["Land Application"],
      "land application share of available N (Figure 5b)", "cell 3, nbasis")
param("PctLiquid", fmt_num(_nb_tot["Liquid"], 1), _nb_tot["Liquid"], "liquid/slurry share of available N",
      "cell 3, nbasis")
param("PctDryStorage", fmt_num(_nb_tot["Other Dry Storage"], 1), _nb_tot["Other Dry Storage"],
      "dry storage share of available N", "cell 3, nbasis")
param("ExportGapTons", fmt_num(_exp_gap_lb / 2000), _exp_gap_lb / 2000,
      "reported export N not documented in manifests (tons)", "cell 3, _exp_gap_lb")
param("PctExportsManifestedLow", fmt_num(100 * _exp_cov_lmh[0]), _exp_cov_lmh[0],
      "manifested share of reported export N, low N contents", "cell 3, _exp_cov_lmh")
param("PctExportsManifestedHigh", fmt_num(100 * _exp_cov_lmh[2]), _exp_cov_lmh[2],
      "manifested share of reported export N, high N contents", "cell 3, _exp_cov_lmh")
param("ExportGapTonsLowContent", fmt_num(_exp_gap_lb_lmh[0] / 2000), _exp_gap_lb_lmh[0] / 2000,
      "reported export N not documented in manifests, low N contents (tons)", "cell 3, _exp_gap_lb_lmh")
param("ExportGapTonsHighContent", fmt_num(_exp_gap_lb_lmh[2] / 2000), _exp_gap_lb_lmh[2] / 2000,
      "reported export N not documented in manifests, high N contents (tons)", "cell 3, _exp_gap_lb_lmh")
param("NDairiesFigFive", f"{N_DAIRIES:,}", N_DAIRIES, "dairies in the Figure 5 sample", "cell 3, N_DAIRIES")
param("NitrogenAvailableMillionLb", fmt_num(TOTAL_N / 1e6), float(TOTAL_N),
      "available N in the Figure 5 sample (million lb)", "cell 3, TOTAL_N")
param("PctUnaccountedFigFive", fmt_num(remainder_shown, 1), float(remainder_shown),
      "unaccounted-for share in Figure 5b", "cell 3, remainder_shown")
param("PctUnknownLowContent", fmt_num(UNKNOWN_PCT_LMH[0], 1), float(UNKNOWN_PCT_LMH[0]),
      "Unknown column of Figure 5b at the low N contents (percent)", "cell 3, UNKNOWN_PCT_LMH")
param("PctUnknownHighContent", fmt_num(UNKNOWN_PCT_LMH[2], 1), float(UNKNOWN_PCT_LMH[2]),
      "Unknown column of Figure 5b at the high N contents (percent)", "cell 3, UNKNOWN_PCT_LMH")

# --- SI Methods 0.1.1 ---
_section = "SI Methods 0.1.1"
param("NReportsDropped", f"{MANIFEST_COUNTS['reports_received'] - _n_reports:,}",
      MANIFEST_COUNTS["reports_received"] - _n_reports, "reports received but not analysed", "cells 3 and 6")
_n_geo = int(_geo["lat"].notna().sum())
param("NDairiesGeocoded", f"{_n_geo:,}", _n_geo, "dairies with a geocoded address", "cell 5, _geo['lat']")
param("NManifestsAutoFound", f"{MANIFEST_COUNTS['docs_automatic']:,}", MANIFEST_COUNTS["docs_automatic"],
      "manifests found by the keyword search", "cell 6")
_lit = MANIFEST_COUNTS["docs_automatic"]
for _nm, _v, _d in (("NFalsePositives", 2, "keyword-search false positives"),
                    ("NPartialManifests", 1, "partial manifests without a signature or record"),
                    ("NMissedManifests", 58, "manifests missed by the keyword search")):
    param(_nm, f"{_v}", _v, _d + " (from the manual screening)", "literal; no file records it", kind="literal")
    param("Pct" + _nm[1:], fmt_num(100 * _v / _lit, 1), 100 * _v / _lit, "share of the manifests found", "cell 7")
# The screening counts must give the screened total: the manifests the keyword
# search proposed, less its false positives, plus the manifests it missed.
_n_fp, _n_missed = PARAMS["NFalsePositives"]["raw"], PARAMS["NMissedManifests"]["raw"]
assert MANIFEST_COUNTS["docs_automatic"] - _n_fp + _n_missed == MANIFEST_COUNTS["docs_screened"], \
    "the manual-screening counts do not give the number of manifests after screening"
assert _n_missed >= MANIFEST_COUNTS["docs_manual"], "fewer missed manifests than manifests added by hand"
param("NManifestsScreened", f"{MANIFEST_COUNTS['docs_screened']:,}", MANIFEST_COUNTS["docs_screened"],
      "manifests after the manual screening, duplicates included", "cell 6, docs_screened")
param("NDuplicates", f"{MANIFEST_COUNTS['docs_duplicate']}", MANIFEST_COUNTS["docs_duplicate"],
      "duplicate manifests removed", "cell 6")
param("NDuplicatesExact", f"{MANIFEST_COUNTS['docs_duplicate_exact']}", MANIFEST_COUNTS["docs_duplicate_exact"],
      "exact-copy duplicates", "cell 6")
param("NDuplicatesCag", f"{MANIFEST_COUNTS['docs_duplicate_cag']}", MANIFEST_COUNTS["docs_duplicate_cag"],
      "CAG018001 duplicates", "cell 6")
param("NManifestsAnalyzed", f"{MANIFEST_COUNTS['docs_analysed']:,}", MANIFEST_COUNTS["docs_analysed"],
      "manifest documents analysed", "cell 6")
_pc = pd.to_numeric(table_s1["Page Count"], errors="coerce")
param("PageCountMin", f"{int(_pc.min())}", int(_pc.min()), "fewest pages of a manifest template", "cell 6, table_s1")
param("PageCountMax", f"{int(_pc.max())}", int(_pc.max()), "most pages of a manifest template", "cell 6, table_s1")
param("NManifestsManual", f"{MANIFEST_COUNTS['docs_manual']}", MANIFEST_COUNTS["docs_manual"],
      "manifests whose pages were specified by hand", "cell 6")
param("NReportsManualManifests", f"{MANIFEST_COUNTS['docs_manual_reports']}", MANIFEST_COUNTS["docs_manual_reports"],
      "reports holding those manifests", "cell 6")
param("NMissedManifestsOther", f"{_n_missed - MANIFEST_COUNTS['docs_manual']}", _n_missed - MANIFEST_COUNTS["docs_manual"],
      "missed manifests corrected during validation (upside-down or out-of-order pages)",
      "literal NMissedManifests less cell 6 docs_manual", kind="literal")
param("NRecordsCompared", f"{S10_N_COMPARED:,}", S10_N_COMPARED, "records compared in Figure S10", "cell 6")
param("NRecordsNotScored", f"{S10_MANURE_NOT_SCORED + S10_WW_NOT_SCORED}", S10_MANURE_NOT_SCORED + S10_WW_NOT_SCORED,
      "records with no automatically extracted amount, not scored in Figure S10", "cell 6")
param("NManureNotScored", f"{S10_MANURE_NOT_SCORED}", S10_MANURE_NOT_SCORED, "of those, manure records", "cell 6")
param("NWwNotScored", f"{S10_WW_NOT_SCORED}", S10_WW_NOT_SCORED, "of those, wastewater records", "cell 6")
param("NManureRecordsWithAuto", f"{S10_MANURE_WITH_AUTO:,}", S10_MANURE_WITH_AUTO,
      "manure records with an automatic extraction record", "cell 6")
param("NManureAmountFound", f"{S10_MANURE_AMOUNT_FOUND:,}", S10_MANURE_AMOUNT_FOUND,
      "of those, records with an automatically extracted manure amount", "cell 6")
param("PctManureAmountFound", fmt_num(100 * S10_MANURE_AMOUNT_FOUND / S10_MANURE_WITH_AUTO, 1),
      S10_MANURE_AMOUNT_FOUND / S10_MANURE_WITH_AUTO, "share of manure records with an extracted amount", "cell 6")
param("PctManureAmountCorrect", fmt_num(S10_MANURE_AMOUNT_ACCURACY, 1), S10_MANURE_AMOUNT_ACCURACY,
      "extracted manure amounts matching the validated value (Figure S10)", "cell 6, s10_stats")
param("NRecordsAnyError", f"{S10_N_ANY_ERROR:,}", S10_N_ANY_ERROR, "records with at least one correction", "cell 6")
param("PctRecordsAnyError", fmt_num(100 * S10_N_ANY_ERROR / S10_N_COMPARED, 1), S10_N_ANY_ERROR / S10_N_COMPARED,
      "share of compared records with a correction", "cell 6")
param("NRecordsImpactError", f"{S10_N_IMPACT_ERROR:,}", S10_N_IMPACT_ERROR,
      "records with a correction to a result-relevant field", "cell 6")
param("PctRecordsImpactError", fmt_num(100 * S10_N_IMPACT_ERROR / S10_N_COMPARED, 1),
      S10_N_IMPACT_ERROR / S10_N_COMPARED, "share of compared records with such a correction", "cell 6")
_acc = s10_stats.set_index("field")["accuracy"]
for _nm, _f in (("AccDestAddress", "Destination Address"), ("AccDestContact", "Destination Contact Address"),
                ("AccDestParcel", "Destination Assessor Parcel Number")):
    param(_nm, fmt_num(_acc[_f], 1), float(_acc[_f]), f"extraction accuracy, {_f}", "cell 6, s10_stats")
param("HaulBinLow", fmt_num(HAUL_BIN_LOW), HAUL_BIN_LOW, "lower edge of the 10-ton bin (tons per haul)",
      "cell 6, HAUL_BIN_LOW", kind="setting")
param("HaulBinMid", fmt_num(HAUL_BIN_MID), HAUL_BIN_MID, "edge between the 10- and 20-ton bins (tons per haul)",
      "cell 6, HAUL_BIN_MID", kind="setting")
param("SingleHaulBelowTons", fmt_num(SINGLE_HAUL_BELOW_TONS), SINGLE_HAUL_BELOW_TONS,
      "a manifest under this many tons is one 10-ton haul", "cell 6, SINGLE_HAUL_BELOW_TONS", kind="setting")
param("PctTenTonBin", fmt_num(100 * MANIFEST_COUNTS["share_10_ton_bin"]), MANIFEST_COUNTS["share_10_ton_bin"],
      "10-ton bin's share of tonnage on manifests reporting a haul size", "cell 6")
param("NRecordsHaulDate", f"{int(_dated.sum()):,}", int(_dated.sum()), "manure records with a haul date",
      "cell 5, _dated")
param("NRecordsLastDateOnly", f"{int(_last_only.sum()):,}", int(_last_only.sum()),
      "manure records with only a last haul date", "cell 5, _last_only")
for _nm, _k in (("AuMilkCow", "Milk Cows"), ("AuDryCow", "Dry Cows"), ("AuLargeHeifer", "Bred Heifers"),
                ("AuSmallHeifer", "Heifers"), ("AuCalf", "Calves (0-3 mo.)")):
    param(_nm, fmt_num(AU_FACTORS[_k], 2), AU_FACTORS[_k], f"animal units per head, {_k}", "cell 0, AU_FACTORS",
          kind="setting")

# --- SI Methods 0.1.2 and 0.1.3 ---
_section = "SI Methods 0.1.2 and 0.1.3"
param("WastewaterSourceTons", fmt_round(WATERFALL_WW / 2000, 1000), WATERFALL_WW / 2000,
      "added unaccounted-for N if wastewater N were a separate source (tons, nearest thousand)",
      "cell 3, WATERFALL_WW")
param("OsmnxVersion", "2.1.0", "2.1.0", "OSMnx version used by Routing.ipynb",
      "literal; environment/environment-routing.yml", kind="literal")
param("NetworkxVersion", "3.6.1", "3.6.1", "NetworkX version", "literal; environment/environment.yml", kind="literal")
for _nm, _v, _d in (("SpeedHighwayKph", 100, "default highway speed"),
                    ("SpeedResidentialKph", 30, "default residential speed"),
                    ("SpeedFallbackKph", 40, "speed for segments with no speed or category")):
    param(_nm, f"{_v}", _v, _d + " (kph)", "literal; Routing.ipynb speed defaults", kind="literal")
param("EmfacVersion", _emfac_ver, EMFAC_META["emfac_version"], "EMFAC model version", "cell 0, EMFAC_META")
for _nm, _key, _dp in (("NOx", "NOx", 2), ("COtwo", "CO2", 2), ("PMtwofive", "PM2.5", 4)):
    param(f"EmfacTenTon{_nm}", fmt_num(ef_c7_g_per_mi[_key], _dp), float(ef_c7_g_per_mi[_key]),
          f"EMFAC {_key} rate, 10-ton class (g/mile)", "cell 0, ef_c7_g_per_mi")
    param(f"EmfacTwentyTon{_nm}", fmt_num(ef_c8_g_per_mi[_key], _dp), float(ef_c8_g_per_mi[_key]),
          f"EMFAC {_key} rate, 20-ton class (g/mile)", "cell 0, ef_c8_g_per_mi")
param("MpgTenTon", fmt_num(_fuel["10-ton"]["mpg"], 2), float(_fuel["10-ton"]["mpg"]), "miles per gallon, 10-ton class",
      "cell 0, _fuel")
param("MpgTwentyTon", fmt_num(_fuel["20-ton"]["mpg"], 2), float(_fuel["20-ton"]["mpg"]),
      "miles per gallon, 20-ton class", "cell 0, _fuel")

# --- SI Methods 0.1.4 and SI figure captions ---
_section = "SI Methods 0.1.4 and SI figure captions"
param("MinHerd", f"{MIN_HERD}", MIN_HERD, "herd size at or below which a report is dropped", "cell 3, MIN_HERD",
      kind="setting")
_n_nondairy = int(next(r[2] for r in sample_log if r[0].startswith("Dropped: not a dairy-cow")))
param("NNonDairy", f"{_n_nondairy}", _n_nondairy, "non-dairy operations dropped", "cell 3, sample_log")
param("NManureRecordsLinked", f"{int(man_n['_rep'].notna().sum()):,}", int(man_n["_rep"].notna().sum()),
      "manure records linked to a sample dairy", "cell 3, man_n['_rep']")
param("NWwRecordsLinked", f"{int(ww_n['_rep'].notna().sum()):,}", int(ww_n["_rep"].notna().sum()),
      "wastewater records linked to a sample dairy", "cell 3, ww_n['_rep']")
param("SolidsDefaultPct", fmt_num(100 * SOLIDS_DEFAULT, 1), SOLIDS_DEFAULT,
      "tonnage-weighted mean solids, used where none is reported", "cell 3, SOLIDS_DEFAULT")
param("NDairiesSnine", f"{len(_s9):,}", len(_s9), "dairies in Figure S9 (the Figure 3b sample)", "cell 5, len(_s9)")
param("MedianUnaccLb", fmt_num(_s9["unacc"].median()), float(_s9["unacc"].median()),
      "median unaccounted-for N per dairy (lb)", "cell 5, _s9['unacc'].median()")
param("NDairiesSnineAu", f"{int(_s9['au'].notna().sum()):,}", int(_s9["au"].notna().sum()),
      "Figure S9 dairies with all six herd classes", "cell 5, _s9['au']")

# --- Manuscript_Parameters.tex and .csv ---
_tex = ["% Manuscript parameters, written by analysis/Analysis.ipynb. Put",
        "% \\input{Manuscript_Parameters} in the preamble. Each macro prints one",
        "% number as the manuscript shows it; units and percent signs are in the",
        "% text. Negative values use \\ensuremath{-}; p-value macros include their",
        "% relation. Macros marked [literal] are counts no file records.", ""]
for _sec in dict.fromkeys(p["section"] for p in PARAMS.values()):
    _tex.append(f"% --- {_sec} ---")
    for _nm, _p in PARAMS.items():
        if _p["section"] == _sec:
            _tag = " [literal]" if _p["kind"] == "literal" else ""
            _tex.append(f"\\newcommand{{\\{_nm}}}{{{_p['value']}}} % {_p['description']}{_tag}")
    _tex.append("")
with open(os.path.join(RESULTS_DIR, "Manuscript_Parameters.tex"), "w") as _fh:
    _fh.write("\n".join(_tex))
pd.DataFrame([{"name": _nm, "value": _p["value"], "raw": _p["raw"], "kind": _p["kind"],
               "section": _p["section"], "description": _p["description"], "source": _p["source"]}
              for _nm, _p in PARAMS.items()]).to_csv(
    os.path.join(RESULTS_DIR, "Manuscript_Parameters.csv"), index=False)
print(f"\n  {len(PARAMS)} macros written to Manuscript_Parameters.tex "
      f"({sum(p['kind'] == 'literal' for p in PARAMS.values())} literals, "
      f"{sum(p['kind'] == 'setting' for p in PARAMS.values())} settings)")

# --- SI tables ---
TEX_ESCAPES = {"\\": r"\textbackslash{}", "&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#",
               "_": r"\_", "{": r"\{", "}": r"\}", "~": r"\textasciitilde{}", "^": r"\textasciicircum{}",
               "\u2013": "--", "\u2019": "'"}


def tex_escape(s):
    """Text with LaTeX special characters escaped."""
    return "".join(TEX_ESCAPES.get(ch, ch) for ch in str(s))


def tex_colspec(spec):
    """Column spec in the manuscript's layout from (alignment, width in cm) pairs."""
    align = {"l": r"\raggedright", "r": r"\raggedleft", "c": r"\centering"}
    return "| " + " | ".join(f">{{{align[a]}\\arraybackslash}}p{{{w}cm}}" for a, w in spec) + " |"


def tex_longtable(spec, caption, header, rows):
    """A longtable with a rule after every row. A row given as a string is a
    section heading spanning the table."""
    out = [f"\\begin{{longtable}}{{{tex_colspec(spec)}}}", f"\\caption{{{caption}}} \\\\", "\\hline",
           " & ".join(f"\\textbf{{{h}}}" for h in header) + " \\\\", "\\hline"]
    for r in rows:
        if isinstance(r, str):
            out.append(f"\\multicolumn{{{len(header)}}}{{|l|}}{{\\textit{{{r}}}}} \\\\")
        else:
            assert len(r) == len(header), f"row has {len(r)} cells for {len(header)} columns"
            out.append(" & ".join(r) + " \\\\")
        out.append("\\hline")
    out.append("\\end{longtable}")
    return "\n".join(out)


def tex_signed(x, dp):
    """Number with thousands separators; a negative value gets $-$."""
    v = round(float(x), dp)
    return ("$-$" if v < 0 else "") + f"{abs(v):,.{dp}f}"


SI_TABLES = {}

SI_TABLES["S1"] = tex_longtable(
    [("l", 3.2), ("c", 1.6), ("l", 2.2), ("r", 1.2), ("r", 1.9), ("l", 3.4)],
    "Breakdown of 2024 Water Board manure and wastewater transport manifests by form template used. "
    "In total, there were \\NManifestTemplates{} unique templates on \\NManifestLayouts{} unique page "
    "formats. The percentages are of the \\NManifestsAnalyzed{} manifests analyzed in the main "
    "manuscript and exclude \\NDuplicatesCag{} duplicates where a facility submitted two different form "
    "templates for the same hauling event(s).",
    ["Template ID", "Attachment Letter", "General Order Referenced", "Page Count",
     "Percent of Analyzed Manifests", "File example"],
    [[tex_escape(r["Template ID"]), tex_escape(r["Attachment Letter"]),
      tex_escape(r["General Order Referenced"]), tex_escape(r["Page Count"]),
      f"{r['Percent of Analyzed Manifests']:.1f}\\%", tex_escape(r["File example"])]
     for _, r in table_s1.iterrows()])

SI_TABLES_S2_S4 = r"""% Tables S2-S4 of the Supplementary Information, written by
% analysis/Analysis.ipynb (cell 7). These tables describe the manifest
% fields and keyword rules, so their text is set in that cell.

% === Table S2
\begin{longtable}{| >{\raggedright\arraybackslash}p{4.4cm} | >{\raggedright\arraybackslash}p{1.6cm} | >{\raggedright\arraybackslash}p{1.9cm} | >{\raggedright\arraybackslash}p{5.2cm} |}
\caption{All manifest parameter fields and post-processing applied.} \\
\hline
\textbf{Parameter Name} & \textbf{Data Type} & \textbf{Manifest Type} & \textbf{Post-Processing Applied} \\
\hline
Origin Dairy Name & text & both &  \\
\hline
Origin Dairy Address & text & both & Filled from annual report address if missing. Geocoding \\
\hline
Hauler Name & text & both &  \\
\hline
Hauler Address & text & both & Geocoding \\
\hline
Destination Name & text & both &  \\
\hline
Destination Contact Address & text & both & Geocoding \\
\hline
Destination Address & text & both & Geocoding, querying the address together with its cross street and county first, if given \\
\hline
Destination Nearest Cross Street & text & both & Geocoding, only where two distinct street names are given and the geocoder returns a street intersection \\
\hline
Destination Type & text & both & Standardization of categories \\
\hline
Destination Assessor Parcel Number & text & both & Geocoding against the statewide parcel locator, within counties covered by ZIP codes in the manifest \\
\hline
Destination Latitude & numeric & both &  \\
\hline
Destination Longitude & numeric & both &  \\
\hline
Haul Date & text & both & Parsed into First Haul Date (if applicable) and Last Haul Date \\
\hline
Is Pipeline & boolean & both & Set true if ``pipeline'' or ``ditch'' is included in the manifest text \\
\hline
Is Trucked & boolean & both & Always true for manure. For wastewater, ``True'' if number of hauls, trucks, or tanks are mentioned, ``False'' if Is Pipeline, otherwise ``Unsure'' \\
\hline
Total Process Wastewater Exports (Gallons) & numeric & wastewater &  \\
\hline
Total Manure Amount (tons) & numeric & manure &  \\
\hline
Total Manure Amount (cubic yards) & numeric & manure & Converted to tons by dividing by Manure Density (cubic yards per ton) \\
\hline
Manure Density & numeric & manure &  \\
\hline
Manure Solids (\%) & numeric & manure &  \\
\hline
Manure Moisture (\%) & numeric & manure & Converted to Solids \% \\
\hline
Method Used to Determine Volume of Manure & text & manure & Broken out into Manure Mass per Haul (tons), Manure Volume per Haul (cubic yards), and Manure Number of Hauls as relevant \\
\hline
Method Used to Determine Volume of Wastewater & text & wastewater & Broken out into Wastewater Volume per Haul (Gallons), Wastewater Number of Hauls, Wastewater Hours Pumped (Hours), and Wastewater Pumping Rate (GPM), as relevant \\
\hline
\end{longtable}

% === Table S3
\begin{longtable}{| >{\raggedright\arraybackslash}p{2.6cm} | >{\raggedright\arraybackslash}p{3.8cm} | >{\raggedright\arraybackslash}p{6.6cm} |}
\caption{Determining trucked versus pipeline wastewater transport.} \\
\hline
\textbf{Transport Type} & \textbf{Detection Keywords} & \textbf{Text Examples} \\
\hline
Pipeline & ``pipeline'' or ``ditch'' & (via concrete ditch)\newline Pipeline\newline LUMAR PIPELINE \\
\hline
Trucked & ``load'', ``haul'', or ``tank'' (without ``apply'' or ``applied'') & 247 Loads @ 9500 Gallons Per Load.\newline Haulers Calc\newline Hauler's Calculations\newline 100 Loads With Slurry Tankers-8400 Gallons/Load\newline Number Of Loads Multiplied By Tank Volume \\
\hline
\end{longtable}

% === Table S4
\begin{longtable}{| >{\raggedright\arraybackslash}p{2.6cm} | >{\raggedright\arraybackslash}p{3.8cm} | >{\raggedright\arraybackslash}p{6.6cm} |}
\caption{Methods used to calculate wastewater volume for transport manifest exports.} \\
\hline
\textbf{Calculation Method} & \textbf{Data Extracted} & \textbf{Text Examples} \\
\hline
Flowrate-based & Hours pumped, pumping rate (GPM) & Gpm X Runtime\newline Run-Times Multiplied By The Gpm Of The Lagoon Pump.\newline 6 Hours @ 1000 Gpm\newline 20 Hours At 1500 Gallons Per Minute\newline Hours Pumped X 1200 Gpm\newline 8 Hours Of Liquid Manure @ 1000 Gallons Per Minute \\
\hline
Pond level-based & Pond level drop & Pond Drop Calculation Based On 5 Feet Drop In First Quarter\newline Pond Drop Calculations\newline PD \\
\hline
Haul-based & Number of hauls, volume per haul & 247 Loads @ 9500 Gallons Per Load.\newline 370 Loads X 10000 Gallons\newline Number Of Loads Multiplied By Tank Volume\newline 100 Loads With Slurry Tankers-8400 Gallons/Load\newline Slurry Tankers - 250 loads at 8,400 gallons per load.\newline Haulers Load Count-17 Loads At 9500 Gallons Per Load.\newline 150 Loads At 9500 Gals/Load\newline 60 Loads Of Surry @ 9500 Gallons Per Load.\newline 10000 Gallons Per Load X 282 Loads\newline @ 10000 Gallons Per Load \\
\hline
Other/Unspecified & Varies & Tank Capacity Multiplied By The Number Of Tanks Applied\newline Precipitation Records\newline Estimated Annual Total To Independent Farming Field.\newline PO \\
\hline
\end{longtable}
"""
SI_STATIC_TABLES_FILE = os.path.join(TABLES_DIR, "SI_Tables_S2_S4.tex")
with open(SI_STATIC_TABLES_FILE, "w") as _fh:
    _fh.write(SI_TABLES_S2_S4)
for _m in re.finditer(r"^% === Table (S\d+)\n(.*?)(?=^% === Table |\Z)", SI_TABLES_S2_S4, flags=re.M | re.S):
    SI_TABLES[_m.group(1)] = _m.group(2).strip()
assert {"S2", "S3", "S4"} <= set(SI_TABLES), "SI_TABLES_S2_S4 lacks a table"

S5_GROUPS = {"Routed": "Routed", "Dropped: No destination address": "No destination address",
             "Dropped: Outside water board": "Outside Water Board", "Dropped: Failed geocode": "Failed geocode",
             "Dropped: Cross-county APN": "Cross-county APN", "Dropped: Routing Failed": "Routing failed",
             "Dropped: Bad Origin Geocode": "Bad origin geocode",
             "Dropped: Bad Destination Geocode": "Bad destination geocode",
             "Dropped: Routing Incorrect": "Routing incorrect", "Dropped: ALL": "All dropped"}
S5_PANELS = {"Total manure (tons)": "Manure per manifest record (tons)",
             "Herd size (animal units)": "Herd size of the origin dairy (animal units)"}
_s5_rows = []
for _var, _panel in S5_PANELS.items():
    _s5_rows.append(_panel)
    for _, r in balance_cont[balance_cont["variable"] == _var].iterrows():
        _tested = pd.notna(r["p_value"])
        _s5_rows.append([
            S5_GROUPS[r["group"]], f"{int(r['n']):,}", tex_signed(r["median"], 1),
            tex_signed(r["median_diff_vs_routed"], 1) if _tested else "",
            tex_signed(r["rank_biserial"], 2) if _tested else "",
            ("$<$0.001" if r["p_value"] < 0.001 else f"{r['p_value']:.3f}") if _tested else ""])
SI_TABLES["S5"] = tex_longtable(
    [("l", 4.2), ("r", 0.8), ("r", 1.5), ("r", 2.0), ("r", 2.4), ("r", 1.2)],
    "Routed manure manifests versus dropped manure manifests. Rank-biserial correlations and p-values "
    "are from two-sided Mann-Whitney tests of dropped groups against the routed group (Fig. S7).",
    ["Group", "$n$", "Median", "Median difference vs routed", "Rank-biserial correlation", "$p$"], _s5_rows)

SI_TABLES["S6"] = tex_longtable(
    [("r", 0.64), ("l", 2.85), ("r", 1.6), ("r", 1.3), ("r", 1.3), ("r", 1.3), ("r", 1.5), ("l", 1.5)],
    "Largest receivers of dairy-exported manure across \\NManureRecords{} manure manifest records. The "
    "tonnage percentages are shares of total manifested manure exports from manifests with an address. "
    "Address types are as in Fig. 2b.",
    ["Rank", "Destination", "Tonnage", "Records", "Origin dairies", "\\% of tonnage", "Cum. \\%",
     "Address type"],
    [[f"{int(r['Rank'])}", tex_escape(re.sub(r"^PO BOX\b", "P.O. Box", r["Destination (normalised)"])),
      f"{r['Tons']:,.0f}", f"{int(r['Manifests'])}", f"{int(r['Origin dairies'])}",
      f"{r['% of tonnage']:.2f}", f"{r['Cumulative %']:.2f}",
      tex_escape("P.O. Box" if r["Address form"] == "PO box" else r["Address form"])]
     for _, r in s6.iterrows()])

# Two destination spellings in the manifests, corrected for display.
S7_DISPLAY_FIXES = {"Macaurthur": "Macarthur", "Waslo": "Wasco"}


def s7_text(s):
    s = str(s)
    for _a, _b in S7_DISPLAY_FIXES.items():
        s = s.replace(_a, _b)
    return tex_escape(s)


SI_TABLES["S7"] = tex_longtable(
    [("r", 0.6), ("r", 0.8), ("l", 1.4), ("l", 1.8), ("l", 2.0), ("l", 1.3), ("r", 1.0), ("r", 0.9), ("r", 1.2)],
    "The ten longest validated manure trucking records sorted by one-way routed travel distance. "
    "Identical origin-destination pairs can have multiple trucking records, denoted here by the Route ID.",
    ["Rank", "Route ID", "Origin dairy", "Origin address", "Destination address", "Dest. type",
     "One-way miles", "Total trips", "Manure (tons)"],
    [[f"{int(r['Rank'])}", f"{int(r['RouteID'])}", s7_text(r["Origin dairy"]), s7_text(r["Origin address"]),
      s7_text(r["Destination address"]), s7_text(r["Destination type"]),
      f"{r['Route distance one way (mi)']:.1f}", f"{int(r['Total truck trips']):,}",
      f"{r['Manure shipped, reported (tons)']:,.1f}"]
     for _, r in table_s7.iterrows()])

for _key, _tab, _col, _hdr, _dp, _cap in (
        ("S8", _t8, "Truck-mi", "Truck-miles", 0,
         "Communities with the largest total manure trucking miles, based on borders provided by the "
         "Census Place dataset \\cite{Census_Place_Shp_2024}. Truck-miles are allotted based on the fraction of "
         "\\RoadBufferM{} m road buffers falling inside the borders of each Census Place."),
        ("S9", _t9, "Truck-mi per capita", "Truck-miles per resident", 2,
         "Communities with the most manure trucking miles per resident. Population is derived from "
         "2020-2024 ACS five-year estimates \\cite{Census_Places}."),
        ("S10", _t10, "Truck-mi per mi2", "Truck-miles per mi$^2$", 0,
         "Communities with the most manure trucking miles per square mile. Area of communities is derived "
         "from Census Place outlines \\cite{Census_Place_Shp_2024}.")):
    SI_TABLES[_key] = tex_longtable(
        [("r", 1.2), ("l", 6.0), ("r", 3.0)], _cap, ["Rank", "Place", _hdr],
        [[f"{int(r['Rank'])}", tex_escape(r["Place"]), f"{r[_col]:,.{_dp}f}"] for _, r in _tab.iterrows()])

# Tables S1 and S2 are centred in the text block instead of starting at the
# journal class's longtable indent; the group keeps the setting to these two.
for _key in ("S1", "S2"):
    SI_TABLES[_key] = ("{\\setlength{\\LTleft}{\\fill}\\setlength{\\LTright}{\\fill}\n"
                       + SI_TABLES[_key].rstrip("\n") + "\n}")

_md = ["# Supplementary Information tables (LaTeX)", "",
       "Written by `analysis/Analysis.ipynb`, which also writes Tables S2 to S4 on their own to `SI_Tables_S2_S4.tex`.",
       "Each block is a `longtable` that needs `\\usepackage{array,longtable}`. The captions use the",
       "macros in `outputs/results/Manuscript_Parameters.tex`, so that file must be `\\input` first.",
       "The citation keys are the manuscript's (`Census_Place_Shp_2024`, `Census_Places`).", ""]
for _key in sorted(SI_TABLES, key=lambda k: int(k[1:])):
    _md += [f"## Table {_key}", "", "```latex", SI_TABLES[_key], "```", ""]
SI_TABLES_FILE = os.path.join(TABLES_DIR, "SI_Tables.md")
with open(SI_TABLES_FILE, "w") as _fh:
    _fh.write("\n".join(_md))

# Every macro the captions use is defined.
_used = set(re.findall(r"\\([A-Za-z]+)\{\}", "\n".join(SI_TABLES.values())))
assert _used <= set(PARAMS), f"captions use undefined macros: {sorted(_used - set(PARAMS))}"
print(f"  {len(SI_TABLES)} SI tables written to {os.path.basename(SI_TABLES_FILE)}")